# Fig 6a upstream: TIL-zone DE, pathway filters and NicheNet per zone (not re-run)

**Produces** the frozen inputs of the TIL-zone figures, on EFS `results/visium/til_zones/`:
- `deg_pseudobulk_edger/`: edgeR TIL-high vs TIL-low DE per TIL zone on Visium pseudobulk, and the
  UP / DOWN gene panels;
- `nichenet_filtering/`: ULM pathway scores per zone, snRNA-seq inputs, ligand / receptor DE per cell type;
- `nichenet_analysis/<zone>/`: `all_axes.csv`, `top_axes.csv`, `axes_catalog.csv`, `axes_shortlist.csv`
  and the hand-picked `axes_selected.csv` drawn in Fig 6a and Supp Fig 6a–b
  (`fig6a_suppfig6_NicheNet_circos.ipynb`).

**Origin**: Spencer Watson's NicheNet_v3.ipynb (`analysis/survival/`), with the outputs of the original
run. The circos cells are in `fig6a_suppfig6_NicheNet_circos.ipynb`, the cell-type correlation modules
in `fig2de_celltype_modules.ipynb`, and the TIL-zone section maps in `fig5a_TIL_zones_map.ipynb`.

**Compared with the published figures**: the three circos plots are byte-identical when drawn from the
frozen `axes_selected.csv`. Points to know about this run:
- `PATHWAY_FILTER` differs between zones: TIL+ 1,934 genes (Hallmark sets + a brain ECM list), BZ1 1,537
  genes in the run behind the BZ1 axes against 1,963 printed as saved (Hallmark + KEGG Medicus +
  BioCarta + custom lists), BZ2 1,934. The Methods describe one filter.
- The TIL+ block ran a different enrichment (sign-aligned ULM on the receiver-filtered target programs)
  where BZ1 and BZ2 ran the zone-DE ULM.
- The frozen TIL+ `axes_selected.csv` predates the current TIL+ axis catalog: 6 of its 54 axis IDs
  (H03C0E5, H1E8A4B, H238F85, H5281DC, HF15A68, L517C1F) are not in `TILplus/axes_catalog.csv`, so the
  TIL+ axis-selection cell fails on the current state; the frozen file is the input of Supp Fig 6a. The
  BZ1 and BZ2 selections match their catalogs (46 and 44 axes).


# NicheNet Analysis of TIL Regions v3

In [ ]:
# Autoreload extension for IPython/Jupyter
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except (ImportError, NameError):
    pass

import pandas as pd
import numpy as np
import anndata as ad
import s3fs

import scanpy as sc
import matplotlib.pyplot as plt
import openslide
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter, AllDataCenters

sc.set_figure_params(dpi=200, color_map="viridis_r")
sc.settings.verbosity = 0

from cloudpathlib import S3Path
from scipy.sparse import issparse
from joblib import Parallel, delayed
from tqdm import tqdm

from gbmcollaboration.definitions import (
    DATA_PATH_CHUV_FILES,
    DATA_PATH_UKER_FILES,
    DATA_PATH_CR_FILES,
)
from gbmcollaboration import AllDataCenters

%load_ext rpy2.ipython

### Load clinical data

In [ ]:
import pandas as pd
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter

# Load processed clinical tables for all centers and combine into one cohort

def load_processed_clinical(dc, center_name):
    d = dc.load_tabular(exclude_idh1_mut=False)
    clin = d["clinical"]
    key = next((k for k in clin if "processed" in str(k).lower() and "clinical" in str(k).lower()), None)
    df = clin[key].copy() if key else list(clin.values())[0].copy()
    df["center"] = center_name
    return df

ch = load_processed_clinical(CHUVDataCenter, "CHUV")
uk = load_processed_clinical(UKERDataCenter, "UKER")
cr = load_processed_clinical(CRDataCenter, "Charite")

clin_all = pd.concat([ch, uk, cr], ignore_index=True, sort=False)
clin_out = clin_all.copy()

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

### Load single cell data

In [ ]:
data_center = AllDataCenters # or "CHUV", "UKER", "CR"
adata_sc = data_center.load_singlecell()

### Create new log_norm layer for merged single cell object

import scanpy as sc
import scipy.sparse as sp

COUNT_LAYER = "unnormalized_counts"
OUT_LAYER   = "LogNormalize"

TARGET_SUM     = 1e4
BATCH_KEY      = "center"   # set to None if you do not want batch-aware HVGs
HVG_N_TOP      = 3000
N_PCS          = 50
N_NEIGHBORS    = 15
UMAP_MIN_DIST  = 0.3
UMAP_SPREAD    = 1.0
RANDOM_STATE   = 13

if COUNT_LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{COUNT_LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

if BATCH_KEY is not None and BATCH_KEY not in adata_sc.obs.columns:
    raise KeyError(f"adata_sc.obs['{BATCH_KEY}'] not found. Set BATCH_KEY=None or choose an existing key.")

adata_sc.layers[OUT_LAYER] = adata_sc.layers[COUNT_LAYER].copy()
sc.pp.normalize_total(adata_sc, layer=OUT_LAYER, target_sum=TARGET_SUM)
sc.pp.log1p(adata_sc, layer=OUT_LAYER)

sc.pp.highly_variable_genes(
    adata_sc,
    n_top_genes=HVG_N_TOP,
    flavor="seurat_v3",
    layer=COUNT_LAYER,
    batch_key=BATCH_KEY,
    subset=False,
)

adata_sc.obsm.pop("X_pca", None)
adata_sc.obsm.pop("X_umap", None)
adata_sc.uns.pop("neighbors", None)
adata_sc.obsp.pop("connectivities", None)
adata_sc.obsp.pop("distances", None)

sc.tl.pca(
    adata_sc,
    n_comps=N_PCS,
    svd_solver="arpack",
    use_highly_variable=True,
)

sc.pp.neighbors(adata_sc, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
sc.tl.umap(adata_sc, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD, random_state=RANDOM_STATE)

print("done")
print("LogNormalize layer:", OUT_LAYER, adata_sc.layers[OUT_LAYER].shape, type(adata_sc.layers[OUT_LAYER]))
print("HVGs:", int(adata_sc.var["highly_variable"].sum()))
# print("X_pca:", adata_sc.obsm["X_pca"].shape, "X_umap:", adata_sc.obsm["X_umap"].shape)

### Save lognorm and UMAP for merged object

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp

SAVE_DIR = "/home/sagemaker-user/gbm-collaboration_2/analysis_sc_merged"
LAYER    = "LogNormalize"
UMAP_KEY = "X_umap"

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")
if UMAP_KEY not in adata_sc.obsm:
    raise KeyError(f"adata_sc.obsm['{UMAP_KEY}'] not found. Available: {list(adata_sc.obsm.keys())}")

os.makedirs(SAVE_DIR, exist_ok=True)

M = adata_sc.layers[LAYER]
M = M.tocsr() if sp.issparse(M) else sp.csr_matrix(M)
sp.save_npz(f"{SAVE_DIR}/{LAYER}.npz", M)

pd.Series(adata_sc.obs_names, name="obs").to_csv(f"{SAVE_DIR}/obs_names.csv", index=False)
pd.Series(adata_sc.var_names, name="var").to_csv(f"{SAVE_DIR}/var_names.csv", index=False)

U = adata_sc.obsm[UMAP_KEY]
pd.DataFrame(U, index=adata_sc.obs_names, columns=["UMAP1", "UMAP2"]).to_csv(f"{SAVE_DIR}/umap_coords.csv")

print("saved:", SAVE_DIR)

### Load lognorm and UMAP from file

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp

LOAD_DIR = "/home/sagemaker-user/gbm-collaboration_2/analysis_sc_merged"
LAYER    = "LogNormalize"
UMAP_KEY = "X_umap"

# file checks
npz_path = f"{LOAD_DIR}/{LAYER}.npz"
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"Missing: {npz_path}")

for fn in ("obs_names.csv", "var_names.csv", "umap_coords.csv"):
    p = f"{LOAD_DIR}/{fn}"
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")

# load saved artifacts
obs_saved = pd.read_csv(f"{LOAD_DIR}/obs_names.csv")["obs"].astype(str)
var_saved = pd.read_csv(f"{LOAD_DIR}/var_names.csv")["var"].astype(str)
M_saved   = sp.load_npz(npz_path)

U_saved = pd.read_csv(f"{LOAD_DIR}/umap_coords.csv", index_col=0)
U_saved.index = U_saved.index.astype(str)

# hard shape checks
if M_saved.shape != (len(obs_saved), len(var_saved)):
    raise ValueError(
        f"M_saved shape {M_saved.shape} does not match "
        f"(len(obs_saved), len(var_saved)) = {(len(obs_saved), len(var_saved))}"
    )

if obs_saved.duplicated().any():
    raise ValueError("Duplicate entries found in saved obs_names.csv")
if var_saved.duplicated().any():
    raise ValueError("Duplicate entries found in saved var_names.csv")

if U_saved.shape[1] < 2:
    raise ValueError(f"umap_coords.csv must have at least 2 columns, got {U_saved.shape[1]}")

# current object indices
obs_cur = pd.Index(adata_sc.obs_names.astype(str))
var_cur = pd.Index(adata_sc.var_names.astype(str))

# overlap
obs_common = obs_cur.intersection(pd.Index(obs_saved))
var_common = var_cur.intersection(pd.Index(var_saved))

if len(obs_common) == 0:
    raise ValueError("No overlapping obs_names between current adata_sc and saved obs_names.")
if len(var_common) == 0:
    raise ValueError("No overlapping var_names between current adata_sc and saved var_names.")

# indexers
cur_r = obs_cur.get_indexer(obs_common)
sav_r = pd.Index(obs_saved).get_indexer(obs_common)
cur_c = var_cur.get_indexer(var_common)
sav_c = pd.Index(var_saved).get_indexer(var_common)

# rebuild layer into current shape (only overlapping entries get filled)
sub = M_saved[sav_r, :][:, sav_c].tocoo()
rows = cur_r[sub.row]
cols = cur_c[sub.col]
M_cur = sp.csr_matrix((sub.data, (rows, cols)), shape=(len(obs_cur), len(var_cur)))
adata_sc.layers[LAYER] = M_cur

# align UMAP
U_cur = np.full((len(obs_cur), 2), np.nan, dtype=float)

U_common = U_saved.loc[obs_common]
U_vals = U_common.iloc[:, :2].to_numpy(dtype=float, copy=False)

U_cur[cur_r, :] = U_vals
adata_sc.obsm[UMAP_KEY] = U_cur

print("loaded into adata_sc:")
print("  layer:", LAYER, adata_sc.layers[LAYER].shape, "nnz:", int(adata_sc.layers[LAYER].nnz))
print("  umap:", UMAP_KEY, adata_sc.obsm[UMAP_KEY].shape, "nan_rows:", int(np.isnan(adata_sc.obsm[UMAP_KEY]).any(axis=1).sum()))
print("  obs_overlap:", len(obs_common), "/", len(obs_cur))
print("  var_overlap:", len(var_common), "/", len(var_cur))

### Load curated cell annotations from file

In [ ]:
import os
import pandas as pd

# user vars
ADATA_OBJ = adata_sc  # set to adata_sc (single cell) or adata (visium), explicitly
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"

LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

if not os.path.exists(KEY_PATH):
    raise FileNotFoundError(f"Missing KEY_PATH: {KEY_PATH}")

key = pd.read_csv(KEY_PATH)

# checks
if LEVEL4_KEY not in ADATA_OBJ.obs.columns:
    raise KeyError(f"Missing {LEVEL4_KEY} in ADATA_OBJ.obs")
missing_cols = [c for c in [LEVEL4_KEY] + NEW_COLS if c not in key.columns]
if missing_cols:
    raise KeyError(f"Missing column(s) in key file: {missing_cols}")

# normalize + deduplicate key
key = key.copy()
key[LEVEL4_KEY] = key[LEVEL4_KEY].astype(str).str.strip()
for c in NEW_COLS:
    key[c] = key[c].astype(str).where(key[c].notna(), None)

key = key.drop_duplicates(subset=[LEVEL4_KEY], keep="first").copy()

# mapping dicts from level4 -> curated levels
key_map = key.set_index(LEVEL4_KEY)[NEW_COLS].to_dict(orient="series")

# apply mappings to create new obs columns
src = ADATA_OBJ.obs[LEVEL4_KEY].astype(str).str.strip()

for col in NEW_COLS:
    mapped = src.map(key_map[col])
    out = mapped.fillna("Unmapped").astype(str)

    cats = pd.Index(key[col].dropna().astype(str).unique())
    if "Unmapped" in out.values and "Unmapped" not in cats:
        cats = cats.append(pd.Index(["Unmapped"]))

    ADATA_OBJ.obs[col] = pd.Categorical(out, categories=cats)

# quick report
lvl4_all = pd.Index(src.unique())
lvl4_in_key = pd.Index(key[LEVEL4_KEY].astype(str).unique())
missing = lvl4_all.difference(lvl4_in_key)

print(f"Target object: {'adata_sc' if ADATA_OBJ is adata_sc else 'adata'}")
print(f"Added columns: {NEW_COLS}")
print(f"Level4 labels in target: {len(lvl4_all)}")
print(f"Level4 labels covered by key: {len(lvl4_all.intersection(lvl4_in_key))}")
print(f"Level4 labels missing from key: {len(missing)}")
if len(missing):
    print("Unmapped labels:", ", ".join(list(missing)[:20]) + (" ..." if len(missing) > 20 else ""))

### Load Visium data

In [ ]:
# Load raw_count for DEG analysis, log_norm_count for ligand-receptor analysis

adata = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    exclude_patient_ids=True,
    exclude_relapse=True,
    list_sample_id=None,
    normalization="raw_count",
    radius=1,
    deconvolution_slot_name="deconv_cohort_level_4",
)

print(adata)

### Load Cell2Location absolute abundance data

In [ ]:
# Load ALL C2L abundance data (CHUV, UKER, Charite) - COMBINED
import gc
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor

def load_sample(center_name, sample_id, h5ad_file):
    """Load and process a single sample using Cell2LocationLoader approach."""
    try:
        if isinstance(h5ad_file, S3Path) or str(h5ad_file).startswith('s3://'):
            s3 = s3fs.S3FileSystem(anon=False)
            h5ad_path_str = str(h5ad_file).replace('s3://', '')
            with s3.open(h5ad_path_str, 'rb') as f:
                adata = ad.read_h5ad(f)
        else:
            adata = ad.read_h5ad(h5ad_file)
        
        if 'means_cell_abundance_w_sf' not in adata.obsm:
            return None
        
        celltype_names = None
        if 'mod' in adata.uns and 'factor_names' in adata.uns['mod']:
            celltype_names = adata.uns['mod']['factor_names']
        elif 'factor_names' in adata.uns:
            celltype_names = adata.uns['factor_names']
        elif 'cell_type_names' in adata.uns:
            celltype_names = adata.uns['cell_type_names']
        
        if celltype_names is None:
            n_celltypes = adata.obsm['means_cell_abundance_w_sf'].shape[1]
            celltype_names = [f'celltype_{i}' for i in range(n_celltypes)]
        
        celltype_names = [f'meanscell_abundance_w_sf_{name}' for name in celltype_names]
        
        cell_abundance = adata.obsm['means_cell_abundance_w_sf']
        if issparse(cell_abundance):
            cell_abundance = cell_abundance.toarray()
        
        df = pd.DataFrame(cell_abundance, index=adata.obs_names, columns=celltype_names)
        df = df.fillna(0)
        df.index = pd.MultiIndex.from_tuples(
            [(sample_id, idx) for idx in df.index],
            names=['sample', 'barcode']
        )
        df['total_cells'] = df.sum(axis=1)
        
        return df
    except Exception as e:
        return None


def load_center(center_name, cell2loc_path):
    """Load one center with original fast parallel, then kill workers."""
    print(f"\nProcessing {center_name}...")
    base_path = S3Path(str(cell2loc_path))
    sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
    print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")
    
    samples_to_process = [
        (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
        for sample_id in sample_dirs
    ]
    
    # Original fast parallel call
    center_results = Parallel(n_jobs=-1)(
        delayed(load_sample)(center_name, sample_id, h5ad_file)
        for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
    )
    
    center_results = [df for df in center_results if df is not None]
    
    for df in center_results:
        new_index = pd.MultiIndex.from_tuples(
            [(center_name, *idx) for idx in df.index],
            names=["center", "sample", "barcode"],
        )
        df.index = new_index
        df["center"] = center_name
    
    df_center = pd.concat(center_results, axis=0)
    
    print(f"  Loaded {df_center.index.get_level_values('sample').nunique()} samples from {center_name}")
    print(f"  Total spots: {df_center.shape[0]}")
    
    # FORCE KILL parallel workers to prevent hang on next call
    get_reusable_executor().shutdown(wait=True)
    gc.collect()
    
    return df_center


# Load each center (workers killed between each)
df_chuv = load_center("CHUV", DATA_PATH_CHUV_FILES["Cell2location"])
df_uker = load_center("UKER", DATA_PATH_UKER_FILES["Cell2location"])
df_cr = load_center("CR", DATA_PATH_CR_FILES["Cell2location"])


# Merge all centers
df_all = pd.concat([df_chuv, df_uker, df_cr], axis=0)

print("\n" + "=" * 60)
print("Combined df_all summary:")
print("  Total spots:", df_all.shape[0])
print("  Total samples:", df_all.index.get_level_values("sample").nunique())
print("  Index levels:", df_all.index.names)
print("  Example columns:", df_all.columns[:5].tolist())
print("=" * 60)

### Merge C2L abundance data with Visium data

In [ ]:
# Drop the 'center' level from df_all's index for alignment
if isinstance(df_all.index, pd.MultiIndex) and "center" in df_all.index.names:
    df_all_no_center = df_all.copy()
    df_all_no_center.index = df_all_no_center.index.droplevel("center")
else:
    df_all_no_center = df_all

print("df_all_no_center index names:", df_all_no_center.index.names)
print("df_all_no_center shape:", df_all_no_center.shape)

# Build (sample, barcode) keys for each obs row, in obs order
barcodes = pd.Index(adata.obs_names).str.split("_").str[-1]
keys = pd.MultiIndex.from_arrays(
    [adata.obs["sample"].values, barcodes.values],
    names=["sample", "barcode"],
)

print("First 3 keys:", list(keys[:3]))
print("Number of keys:", len(keys))

# Align df_all_no_center to these keys
df_all_aligned = df_all_no_center.reindex(keys)
print("df_all_aligned shape:", df_all_aligned.shape)

# Drop columns from df_all_aligned that already exist in adata.obs
cols_to_add = [c for c in df_all_aligned.columns if c not in adata.obs.columns]
df_to_add = df_all_aligned[cols_to_add].copy()
df_to_add.index = adata.obs.index

# Join instead of concat to avoid fragmentation
adata.obs = adata.obs.join(df_to_add)

print(f"\nNumber of added columns: {len(cols_to_add)}")
print(f"Skipped (already exist): {len(df_all_aligned.columns) - len(cols_to_add)}")
print(f"adata.obs shape: {adata.obs.shape}")

### Load custom cell annotations from file for Visium

In [ ]:
import pandas as pd

# user vars
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

# 1) Load key
key_raw = pd.read_csv(KEY_PATH)

# 2) Basic column checks
required_cols = [LEVEL4_KEY] + NEW_COLS
missing_cols = [c for c in required_cols if c not in key_raw.columns]
if missing_cols:
    raise ValueError(f"Missing columns in key file: {missing_cols}")

# 3) Deduplicate on the level4 key and standardize to string
key_unique = key_raw.drop_duplicates(subset=[LEVEL4_KEY]).copy()
key_unique[LEVEL4_KEY] = key_unique[LEVEL4_KEY].astype(str)

# 4) Build a tidy DataFrame indexed by level4
celltype_key = key_unique.set_index(LEVEL4_KEY)[NEW_COLS]
celltype_key = celltype_key.apply(lambda s: s.astype(str))

# 5) Attach as the canonical celltype hierarchy in adata.uns
adata.uns["celltype_annotation_key"] = celltype_key

# 6) Quick coverage checks vs Visium abundance columns
lvl4_in_key = pd.Index(celltype_key.index)

obs_cols = pd.Index(adata.obs.columns)

# deconv_cohort_level_4 columns should use the raw level4 names
deconv_level4_cols = obs_cols.intersection(lvl4_in_key)

# C2L totals use "meanscell_abundance_w_sf_<level4>"
c2l_cols = [c for c in obs_cols if c.startswith("meanscell_abundance_w_sf_")]
c2l_lvl4 = pd.Index([c.replace("meanscell_abundance_w_sf_", "") for c in c2l_cols])
c2l_in_key = c2l_lvl4.intersection(lvl4_in_key)

missing_for_deconv = lvl4_in_key.difference(deconv_level4_cols)
missing_for_c2l = lvl4_in_key.difference(c2l_in_key)

print("Stored celltype_annotation_key in adata.uns['celltype_annotation_key']")
print(f"Total level4 types in key: {len(lvl4_in_key)}")
print(f"Matching deconv level4 columns in adata.obs: {len(deconv_level4_cols)}")
print(f"Matching C2L level4 types (from meanscell_abundance_w_sf_*): {len(c2l_in_key)}")

if len(missing_for_deconv):
    print(f"Level4 types in key not seen as deconv columns: {len(missing_for_deconv)}")
if len(missing_for_c2l):
    print(f"Level4 types in key not seen in C2L columns: {len(missing_for_c2l)}")

### Load TIL spot metadata from file

In [ ]:
import pandas as pd
from pathlib import Path

# paths
EFS_ROOT = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd")
IN_CSV   = EFS_ROOT / "results" / "visium" / "til_zones" / "til_zone_annotations.csv"

# options
overwrite = True

# load
usecols = ["spot_id", "sample", "TIL_positive", "TIL_zone"]
df = pd.read_csv(
    IN_CSV,
    usecols=usecols,
    dtype={"spot_id": "string", "sample": "string", "TIL_zone": "string"},
)
df["spot_id"] = df["spot_id"].astype(str)
df = df.set_index("spot_id", drop=True)

# align
obs_names = pd.Index(adata.obs_names.astype(str))
df_aligned = df.reindex(obs_names)

# validate
if df_aligned["TIL_zone"].isna().any():
    n_missing = int(df_aligned["TIL_zone"].isna().sum())
    raise ValueError(f"{n_missing} adata.obs_names not found in {IN_CSV.name}")

if "sample" in adata.obs.columns:
    adata_sample = adata.obs["sample"].astype(str)
    csv_sample = df_aligned["sample"].astype(str)
    mism = (adata_sample != csv_sample) & csv_sample.notna()
    print("sample_mismatch_n:", int(mism.sum()))

# add
new_cols = ["TIL_positive", "TIL_zone"]
for c in new_cols:
    if (c in adata.obs.columns) and (not overwrite):
        raise KeyError(f"adata.obs['{c}'] already exists (set overwrite=True to replace)")

adata.obs["TIL_positive"] = df_aligned["TIL_positive"].astype("boolean")
til_order = ["TIL-", "BZ2", "BZ1", "TIL+"]
til_levels = [x for x in til_order if x in set(df_aligned["TIL_zone"].dropna().unique())]
if len(til_levels) >= 2:
    adata.obs["TIL_zone"] = pd.Categorical(df_aligned["TIL_zone"], categories=til_levels, ordered=True)
else:
    adata.obs["TIL_zone"] = df_aligned["TIL_zone"].astype("string")

# report
print("added_cols:", new_cols)
print("TIL_positive_counts:\n", adata.obs["TIL_positive"].value_counts(dropna=False))
print("TIL_zone_counts:\n", adata.obs["TIL_zone"].value_counts(dropna=False))

### Define TIL-low and TIL-high patients

In [ ]:
import pandas as pd
import numpy as np

VISIUM_SAMPLE_COL = "sample"
SC_SAMPLE_COL     = "orig.ident"
TIL_FLAG_COL      = "TIL_positive"

QUANTILE_CUTOFF = 0.75
MIN_TIL_SPOTS_FOR_HIGH = 0
MIN_TOTAL_SPOTS = 0
MIN_TIL_FRACTION_FOR_HIGH = None

KEY_LEN = 9

if "adata" not in globals():
    raise ValueError("adata (Visium) not found in the session.")
if "adata_sc" not in globals():
    raise ValueError("adata_sc (single-cell) not found in the session.")

for col in [VISIUM_SAMPLE_COL, TIL_FLAG_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing '{col}' in adata.obs")

if SC_SAMPLE_COL not in adata_sc.obs.columns:
    raise KeyError(f"Missing '{SC_SAMPLE_COL}' in adata_sc.obs")

visium_id_full = adata.obs[VISIUM_SAMPLE_COL].astype(str).str.replace(r"_vis$", "", regex=True)
visium_key = visium_id_full.str[:KEY_LEN]

df = (
    pd.DataFrame(
        {
            "join_key": visium_key.values,
            "visium_id_example": visium_id_full.values,
            "til_flag": adata.obs[TIL_FLAG_COL].astype(int).values,
        },
        index=adata.obs_names,
    )
    .groupby("join_key", as_index=False)
    .agg(
        TIL_spots=("til_flag", "sum"),
        Total_Spots=("til_flag", "size"),
        visium_id_example=("visium_id_example", lambda s: s.iloc[0]),
    )
)

df["TIL_fraction"] = df["TIL_spots"] / df["Total_Spots"]
df["TIL_percent"] = 100.0 * df["TIL_fraction"]

if MIN_TOTAL_SPOTS > 0:
    df = df[df["Total_Spots"] >= int(MIN_TOTAL_SPOTS)].copy()

if df.empty:
    raise ValueError("No groups remain after applying MIN_TOTAL_SPOTS filter.")

cutoff = float(df["TIL_fraction"].quantile(float(QUANTILE_CUTOFF)))
if MIN_TIL_FRACTION_FOR_HIGH is not None:
    cutoff = max(cutoff, float(MIN_TIL_FRACTION_FOR_HIGH))

df["TIL_group"] = "TIL_low"
high_mask = df["TIL_fraction"] >= cutoff
if MIN_TIL_SPOTS_FOR_HIGH > 0:
    high_mask &= df["TIL_spots"] >= int(MIN_TIL_SPOTS_FOR_HIGH)
df.loc[high_mask, "TIL_group"] = "TIL_high"

key_to_label = df.set_index("join_key")["TIL_group"].to_dict()

adata.obs["TIL_group"] = visium_key.map(key_to_label)
adata.obs["TIL_group"] = pd.Categorical(
    adata.obs["TIL_group"],
    categories=["TIL_high", "TIL_low"],
    ordered=True,
)

sc_id_full = adata_sc.obs[SC_SAMPLE_COL].astype(str)
sc_key = sc_id_full.str[:KEY_LEN]

adata_sc.obs["sample_id"] = sc_key.astype(str)

sc_labels = sc_key.map(key_to_label).fillna("Unmapped")
adata_sc.obs["TIL_group"] = pd.Categorical(
    sc_labels,
    categories=["TIL_high", "TIL_low", "Unmapped"],
    ordered=False,
)

adata.uns["TIL_group_stats"] = df.set_index("join_key")

print(f"Join key: first {KEY_LEN} chars of Visium sample (after stripping '_vis') and of single-cell orig.ident")
print(f"TIL_fraction cutoff for TIL_high (Q={QUANTILE_CUTOFF:.2f}): {cutoff:.6f}")

print("\nVisium group counts (unique join keys):")
print(df["TIL_group"].value_counts().to_string())

print("\nSingle-cell TIL_group value counts:")
print(adata_sc.obs["TIL_group"].value_counts(dropna=False).to_string())

n_sc_mapped = int((adata_sc.obs["TIL_group"].astype(str) != "Unmapped").sum())
print(f"\nSingle-cell mapped (not Unmapped): {n_sc_mapped} / {adata_sc.n_obs}")

vis_keys = set(pd.unique(visium_key))
sc_keys = set(pd.unique(sc_key))
only_in_sc = sorted(sc_keys - vis_keys)
only_in_vis = sorted(vis_keys - sc_keys)

print(f"\nKey coverage (unique join keys):")
print(f"  keys in Visium: {len(vis_keys)}")
print(f"  keys in scRNA : {len(sc_keys)}")
print(f"  keys only in scRNA (unmappable from Visium): {len(only_in_sc)}")
print(f"  keys only in Visium (no scRNA): {len(only_in_vis)}")
print(f"  examples only_in_sc (up to 20): {only_in_sc[:20]}")
print(f"  examples only_in_vis (up to 20): {only_in_vis[:20]}")

In [ ]:
print(df.to_string(index=False))

### Clean Visium genes

In [ ]:
BAD_FEATURES = ["imagecol", "imagerow"]

present_bad = [g for g in BAD_FEATURES if g in adata.var_names]

if len(present_bad) == 0:
    adata_deg = adata
    print("No bad features in var_names, using adata as-is.")
else:
    keep = ~adata.var_names.isin(present_bad)
    adata_deg = adata[:, keep].copy()
    print("Dropped from var_names:", present_bad)
    print("Genes before:", adata.n_vars, "after:", adata_deg.n_vars)

### Pseudobulk TIL zones per patient

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

ZONE_KEY = "TIL_zone"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample"
RAW_COUNT_LAYER = "raw_count"

ZONES = ["TIL+", "BZ1", "BZ2", "TIL-"]
GROUPS = ["TIL_low", "TIL_high"]  # low is reference

MIN_SPOTS_PER_SAMPLE_IN_ZONE = 1
MIN_SAMPLES_PER_GROUP = 3

for k in [ZONE_KEY, GROUP_KEY, SAMPLE_KEY]:
    if k not in adata_deg.obs.columns:
        raise KeyError(f"adata_deg.obs['{k}'] not found")
if RAW_COUNT_LAYER not in adata_deg.layers:
    raise KeyError(f"adata_deg.layers['{RAW_COUNT_LAYER}'] not found")

def pseudobulk_counts_for_zone(adata_in, zone: str):
    obs = adata_in.obs
    mask = (obs[ZONE_KEY].astype(str) == str(zone)) & (obs[GROUP_KEY].astype(str).isin(GROUPS))
    idx = np.flatnonzero(mask.to_numpy())
    if idx.size == 0:
        return None

    sub_obs = obs.iloc[idx][[SAMPLE_KEY, GROUP_KEY]].copy()
    sub_obs[SAMPLE_KEY] = sub_obs[SAMPLE_KEY].astype(str)
    sub_obs[GROUP_KEY] = sub_obs[GROUP_KEY].astype(str)

    spot_counts = sub_obs[SAMPLE_KEY].value_counts()
    keep_samples = spot_counts[spot_counts >= int(MIN_SPOTS_PER_SAMPLE_IN_ZONE)].index
    keep_mask = sub_obs[SAMPLE_KEY].isin(keep_samples).to_numpy()

    idx = idx[keep_mask]
    sub_obs = sub_obs.iloc[keep_mask].copy()
    if idx.size == 0:
        return None

    nuniq = sub_obs.groupby(SAMPLE_KEY)[GROUP_KEY].nunique()
    bad = nuniq[nuniq > 1]
    if len(bad):
        raise ValueError(f"{len(bad)} samples map to >1 group in zone={zone}. Examples: {bad.index[:5].tolist()}")

    sample_group = sub_obs.groupby(SAMPLE_KEY)[GROUP_KEY].first()

    X = adata_in.layers[RAW_COUNT_LAYER]
    samples = sub_obs[SAMPLE_KEY].to_numpy()
    uniq_samples, inv = np.unique(samples, return_inverse=True)

    if sp.issparse(X):
        Xz = X[idx, :].tocsr()
        G = sp.csr_matrix(
            (np.ones(inv.size, dtype=np.float32), (inv, np.arange(inv.size))),
            shape=(len(uniq_samples), inv.size),
        )
        Xpb = (G @ Xz).toarray()
    else:
        Xz = np.asarray(X)[idx, :]
        Xpb = np.zeros((len(uniq_samples), adata_in.n_vars), dtype=np.float64)
        np.add.at(Xpb, inv, Xz)

    Xpb = np.rint(Xpb).astype(np.int64)
    if (Xpb < 0).any():
        raise ValueError(f"Negative counts after pseudobulk in zone={zone}")

    gene_mask = Xpb.sum(axis=0) > 0
    genes = pd.Index(adata_in.var_names.astype(str))[gene_mask]
    Xpb = Xpb[:, gene_mask]

    meta = pd.DataFrame(index=pd.Index(uniq_samples, name="sample"))
    meta["TIL_group"] = sample_group.reindex(uniq_samples).astype(str).values
    meta["n_spots_in_zone"] = spot_counts.reindex(uniq_samples).fillna(0).astype(int).values
    meta["zone"] = str(zone)

    counts_df = pd.DataFrame(Xpb.T, index=genes, columns=uniq_samples)
    return counts_df, meta

counts_by_zone = {}
meta_by_zone = {}
summary_rows = []

for zone in ZONES:
    out = pseudobulk_counts_for_zone(adata_deg, zone)
    if out is None:
        summary_rows.append({"zone": zone, "status": "skip", "reason": "no spots after filtering"})
        continue

    counts_df, meta = out
    meta["TIL_group"] = pd.Categorical(meta["TIL_group"].astype(str), categories=GROUPS, ordered=True)

    n_by_group = meta["TIL_group"].value_counts().to_dict()
    ok = all(int(n_by_group.get(g, 0)) >= int(MIN_SAMPLES_PER_GROUP) for g in GROUPS)

    summary_rows.append({
        "zone": zone,
        "status": "ok" if ok else "skip",
        "reason": "" if ok else f"too few samples per group: {n_by_group}",
        "n_samples_total": int(meta.shape[0]),
        "n_TIL_low": int(n_by_group.get("TIL_low", 0)),
        "n_TIL_high": int(n_by_group.get("TIL_high", 0)),
        "n_spots_total_in_zone": int(meta["n_spots_in_zone"].sum()),
        "median_spots_per_sample": float(np.median(meta["n_spots_in_zone"].to_numpy())),
        "n_genes_nonzero": int(counts_df.shape[0]),
    })

    if ok:
        counts_by_zone[zone] = counts_df
        meta_by_zone[zone] = meta

df_pb_summary = pd.DataFrame(summary_rows)
display(df_pb_summary)
print("Zones kept for DE:", list(counts_by_zone.keys()))

### DE analysis of each TIL zone

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

GROUPS = ["TIL_low", "TIL_high"]  # low is reference

OUT_DIR = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/visium/til_zones/deg_pseudobulk_edger")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ro.r("suppressPackageStartupMessages(library(edgeR))")

r_run_edger = ro.r(r"""
function(counts_mat, group_vec) {
  group_vec <- factor(group_vec, levels=c("TIL_low","TIL_high"))

  counts <- as.matrix(counts_mat)
  storage.mode(counts) <- "integer"

  y <- DGEList(counts=counts)
  design <- model.matrix(~group_vec)

  keep <- filterByExpr(y, design=design)
  y <- y[keep, , keep.lib.sizes=FALSE]
  y <- calcNormFactors(y)

  y <- estimateDisp(y, design, robust=TRUE)
  fit <- glmQLFit(y, design, robust=TRUE)
  qlf <- glmQLFTest(fit, coef=2)

  tt <- topTags(qlf, n=Inf)$table
  tt$gene <- rownames(tt)
  tt <- tt[, c("gene","logFC","logCPM","F","PValue","FDR")]
  tt
}
""")

de_tables = {}
summary = []

for zone in counts_by_zone.keys():
    counts_df = counts_by_zone[zone]
    meta = meta_by_zone[zone]

    group_vec = meta.loc[counts_df.columns, "TIL_group"].astype(str).tolist()

    with localconverter(ro.default_converter + pandas2ri.converter):
        r_counts = ro.conversion.py2rpy(counts_df)

    res_r = r_run_edger(r_counts, ro.StrVector(group_vec))

    with localconverter(ro.default_converter + pandas2ri.converter):
        df_de = ro.conversion.rpy2py(res_r)

    n_by_group = meta["TIL_group"].astype(str).value_counts().to_dict()
    df_de["zone"] = zone
    df_de["contrast"] = "TIL_high_vs_TIL_low"
    df_de["n_samples_total"] = int(counts_df.shape[1])
    df_de["n_TIL_low"] = int(n_by_group.get("TIL_low", 0))
    df_de["n_TIL_high"] = int(n_by_group.get("TIL_high", 0))

    out_de = OUT_DIR / f"DE_edger_zone_{zone}_TIL_high_vs_TIL_low.csv"
    out_meta = OUT_DIR / f"DE_edger_zone_{zone}_sample_metadata.csv"

    df_de.to_csv(out_de, index=False)
    meta.reset_index().to_csv(out_meta, index=False)

    de_tables[zone] = df_de

    summary.append({
        "zone": zone,
        "n_genes_tested": int(df_de.shape[0]),
        "n_samples_total": int(counts_df.shape[1]),
        "n_TIL_low": int(n_by_group.get("TIL_low", 0)),
        "n_TIL_high": int(n_by_group.get("TIL_high", 0)),
        "de_csv": str(out_de),
        "meta_csv": str(out_meta),
    })

df_de_summary = pd.DataFrame(summary)
summary_csv = OUT_DIR / "DE_edger_summary.csv"
df_de_summary.to_csv(summary_csv, index=False)

print("Wrote:", summary_csv)
display(df_de_summary)

### Create UP and DOWN gene panels

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

FDR_MAX = 0.05
LOGFC_MIN = 0.25          # set to 0.0 if you want "all significant" regardless of effect size
MIN_LOGCPM = None         # e.g. 0.0 or 1.0, or keep as None to skip

if "df_de_summary" not in globals():
    raise NameError("df_de_summary is not defined. Run the edgeR summary cell that creates it.")

out_dir = Path(df_de_summary["de_csv"].iloc[0]).parent
out_dir.mkdir(parents=True, exist_ok=True)

de_tables = {}
panel_rows = []
panel_counts = []

for _, row in df_de_summary.iterrows():
    zone = str(row["zone"])
    de_path = Path(row["de_csv"])
    if not de_path.exists():
        raise FileNotFoundError(f"Missing DE file for zone={zone}: {de_path}")

    df = pd.read_csv(de_path)

    required = ["gene", "logFC", "logCPM", "PValue", "FDR"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"DE table for zone={zone} is missing columns: {missing}")

    df = df.dropna(subset=["gene", "logFC", "FDR"]).copy()
    df["gene"] = df["gene"].astype(str)

    if MIN_LOGCPM is not None:
        df = df[df["logCPM"] >= float(MIN_LOGCPM)].copy()

    sig = df[df["FDR"] <= float(FDR_MAX)].copy()

    up = sig[sig["logFC"] >= float(LOGFC_MIN)].sort_values(["FDR", "PValue"], ascending=[True, True]).copy()
    down = sig[sig["logFC"] <= -float(LOGFC_MIN)].sort_values(["FDR", "PValue"], ascending=[True, True]).copy()

    up_genes = up["gene"].tolist()
    down_genes = down["gene"].tolist()

    (out_dir / "gene_panels").mkdir(parents=True, exist_ok=True)
    pd.Series(up_genes, name="gene").to_csv(out_dir / "gene_panels" / f"{zone}_UP_FDR{FDR_MAX}_logFC{LOGFC_MIN}.txt",
                                            index=False, header=False)
    pd.Series(down_genes, name="gene").to_csv(out_dir / "gene_panels" / f"{zone}_DOWN_FDR{FDR_MAX}_logFC{LOGFC_MIN}.txt",
                                              index=False, header=False)

    up_min = up[["gene", "logFC", "logCPM", "PValue", "FDR"]].copy()
    up_min["zone"] = zone
    up_min["direction"] = "UP"

    down_min = down[["gene", "logFC", "logCPM", "PValue", "FDR"]].copy()
    down_min["zone"] = zone
    down_min["direction"] = "DOWN"

    panel_rows.append(up_min)
    panel_rows.append(down_min)

    panel_counts.append({
        "zone": zone,
        "n_sig_total": int(sig.shape[0]),
        "n_up": int(len(up_genes)),
        "n_down": int(len(down_genes)),
        "FDR_MAX": float(FDR_MAX),
        "LOGFC_MIN": float(LOGFC_MIN),
        "MIN_LOGCPM": (None if MIN_LOGCPM is None else float(MIN_LOGCPM)),
        "de_csv": str(de_path),
    })

    de_tables[zone] = df

df_panels = pd.concat(panel_rows, ignore_index=True)
df_panel_counts = pd.DataFrame(panel_counts).sort_values("zone")

panels_csv = out_dir / f"DE_gene_panels_by_zone_FDR{FDR_MAX}_logFC{LOGFC_MIN}.csv"
counts_csv = out_dir / f"DE_gene_panel_counts_FDR{FDR_MAX}_logFC{LOGFC_MIN}.csv"

df_panels.to_csv(panels_csv, index=False)
df_panel_counts.to_csv(counts_csv, index=False)

print("Wrote:", panels_csv)
print("Wrote:", counts_csv)
display(df_panel_counts)

### Cell type association with TIL zones

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

ZONE_KEY = "TIL_zone"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample"

DECONV_PREFIX = "meanscell_abundance_w_sf_"
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
SCANVI_KEY = "celltype_level4_scanvi"
CUR_KEY = "celltype_level4_curated"

ZONES = ["TIL+", "BZ1", "BZ2", "TIL-"]
GROUPS = ["TIL_low", "TIL_high"]

OUT_DIR = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/visium/til_zones/celltype_zone_specificity_curated")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)

    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out

    p_ok = p[ok]
    m = p_ok.size
    order = np.argsort(p_ok)
    ranked = p_ok[order]
    q = ranked * (m / (np.arange(1, m + 1)))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)

    q_full = np.empty_like(p_ok)
    q_full[order] = q
    out[ok] = q_full
    return out

for k in [ZONE_KEY, GROUP_KEY, SAMPLE_KEY]:
    if k not in adata_deg.obs.columns:
        raise KeyError(f"adata_deg.obs['{k}'] not found")

key_path = Path(KEY_PATH)
if not key_path.exists():
    raise FileNotFoundError(f"Missing KEY_PATH: {KEY_PATH}")

key = pd.read_csv(key_path)
for c in [SCANVI_KEY, CUR_KEY]:
    if c not in key.columns:
        raise KeyError(f"Missing column '{c}' in key file")

key = key.copy()
key[SCANVI_KEY] = key[SCANVI_KEY].astype(str).str.strip()
key[CUR_KEY] = key[CUR_KEY].astype(str).str.strip()
scanvi_to_cur = dict(zip(key[SCANVI_KEY].tolist(), key[CUR_KEY].tolist()))

pref_cols = [c for c in adata_deg.obs.columns if c.startswith(DECONV_PREFIX)]
if len(pref_cols) == 0:
    raise ValueError(f"No deconvolution columns found with prefix: {DECONV_PREFIX}")

base_labels = [c[len(DECONV_PREFIX):] for c in pref_cols]
missing_in_key = sorted(set(base_labels).difference(set(scanvi_to_cur.keys())))
if len(missing_in_key) > 0:
    raise ValueError(
        f"{len(missing_in_key)} deconv base labels are missing from the key mapping. "
        f"Examples: {missing_in_key[:20]}"
    )

col_to_cur = {c: scanvi_to_cur[c[len(DECONV_PREFIX):]] for c in pref_cols}

obs = adata_deg.obs
df = obs[[SAMPLE_KEY, GROUP_KEY, ZONE_KEY] + pref_cols].copy()
df[SAMPLE_KEY] = df[SAMPLE_KEY].astype(str)
df[GROUP_KEY] = df[GROUP_KEY].astype(str)
df[ZONE_KEY] = df[ZONE_KEY].astype(str)

df = df[df[GROUP_KEY].isin(GROUPS)].copy()
df = df[df[ZONE_KEY].isin(ZONES)].copy()
if df.empty:
    raise ValueError("No rows remain after filtering to GROUPS and ZONES")

df_means = (
    df.groupby([SAMPLE_KEY, GROUP_KEY, ZONE_KEY], observed=True)[pref_cols]
      .mean()
      .rename(columns=col_to_cur)
      .reset_index()
)

celltypes_cur = [c for c in df_means.columns if c not in [SAMPLE_KEY, GROUP_KEY, ZONE_KEY]]
if len(celltypes_cur) == 0:
    raise ValueError("No curated celltype columns present after renaming")

coverage = (
    df_means[[SAMPLE_KEY, GROUP_KEY, ZONE_KEY]]
    .drop_duplicates()
    .groupby([GROUP_KEY, ZONE_KEY], observed=True)[SAMPLE_KEY]
    .nunique()
    .unstack(ZONE_KEY)
    .reindex(index=GROUPS, columns=ZONES)
)
print("n_samples with >=1 spot in zone (by group):")
display(coverage)

eps = 1e-9
rows = []

for g in GROUPS:
    df_g = df_means[df_means[GROUP_KEY] == g].copy()
    if df_g.empty:
        continue

    for ct in celltypes_cur:
        mat = df_g.pivot_table(index=SAMPLE_KEY, columns=ZONE_KEY, values=ct, aggfunc="mean")

        present_zones = [z for z in ZONES if z in mat.columns]
        if len(present_zones) < 2:
            continue

        for z in present_zones:
            other = [zz for zz in present_zones if zz != z]
            v_zone = mat[z]
            v_rest = mat[other].mean(axis=1, skipna=True)

            m = v_zone.notna() & v_rest.notna()
            n = int(m.sum())
            if n == 0:
                continue

            l2 = np.log2((v_zone[m].to_numpy(dtype=float) + eps) / (v_rest[m].to_numpy(dtype=float) + eps))

            mean_zone = float(np.mean(v_zone[m].to_numpy(dtype=float)))
            mean_rest = float(np.mean(v_rest[m].to_numpy(dtype=float)))
            mean_l2 = float(np.mean(l2))
            med_l2 = float(np.median(l2))

            p = np.nan
            if n >= 5:
                try:
                    p = float(wilcoxon(l2, zero_method="wilcox", alternative="two-sided").pvalue)
                except Exception:
                    p = np.nan

            rows.append({
                "group": g,
                "zone": z,
                "celltype_level4_curated": ct,
                "n_samples_used": n,
                "mean_zone": mean_zone,
                "mean_rest_other_zones": mean_rest,
                "delta_zone_minus_rest": mean_zone - mean_rest,
                "mean_log2FC_zone_vs_rest": mean_l2,
                "median_log2FC_zone_vs_rest": med_l2,
                "pval_wilcoxon_paired": p,
            })

res = pd.DataFrame(rows)
if res.empty:
    raise ValueError("No results generated. This usually means too few zones per sample within group after aggregation.")

res["fdr_bh_within_group_zone"] = np.nan
for (g, z), sub_idx in res.groupby(["group", "zone"]).groups.items():
    res.loc[sub_idx, "fdr_bh_within_group_zone"] = bh_fdr(res.loc[sub_idx, "pval_wilcoxon_paired"].to_numpy())

out_csv = OUT_DIR / "celltype_zone_specificity_level4_curated.csv"
res.to_csv(out_csv, index=False)
print("Wrote:", str(out_csv))

for g in GROUPS:
    for z in ZONES:
        sub = res[(res["group"] == g) & (res["zone"] == z)].copy()
        if sub.empty:
            continue
        sub = sub.sort_values("mean_log2FC_zone_vs_rest", key=lambda s: np.abs(s), ascending=False)

        print(f"\n(group={g}) zone={z}: top 20 by abs(mean_log2FC_zone_vs_rest)")
        print(sub[[
            "celltype_level4_curated",
            "n_samples_used",
            "mean_zone",
            "mean_rest_other_zones",
            "mean_log2FC_zone_vs_rest",
            "pval_wilcoxon_paired",
            "fdr_bh_within_group_zone",
        ]].head(20).to_string(index=False))

### Cell types in single cell data per TIL group

In [ ]:
import pandas as pd

CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"

for k in (CT_KEY, GROUP_KEY):
    if k not in adata_sc.obs.columns:
        raise KeyError(f"Missing adata_sc.obs['{k}'].")

df = adata_sc.obs[[CT_KEY, GROUP_KEY]].copy()
df[CT_KEY] = df[CT_KEY].astype(str)
df[GROUP_KEY] = df[GROUP_KEY].astype(str)

df = df[df[GROUP_KEY].isin(["TIL_high", "TIL_low"])].copy()

print("Unique TIL_group values (after filter):", sorted(df[GROUP_KEY].unique().tolist()))
print("\nTIL_group counts (after filter):")
print(df[GROUP_KEY].value_counts().to_string())

ct_counts = (
    df.groupby([CT_KEY, GROUP_KEY], observed=True)
      .size()
      .unstack(GROUP_KEY, fill_value=0)
      .reindex(columns=["TIL_high", "TIL_low"])
      .sort_values(["TIL_high", "TIL_low"], ascending=False)
      .reset_index()
)

ct_counts.columns.name = None

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
    display(ct_counts)

print("\ncelltypes:", int(ct_counts.shape[0]))
print("total_TIL_high_cells:", int(ct_counts["TIL_high"].sum()))
print("total_TIL_low_cells:", int(ct_counts["TIL_low"].sum()))

### Install MultiNicheNet data sources

%%R
options(timeout = 3600)

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager", repos = "https://cloud.r-project.org")
}

if (!requireNamespace("OmnipathR", quietly = TRUE)) {
  BiocManager::install("OmnipathR", update = FALSE, ask = FALSE)
}

if (!requireNamespace("reticulate", quietly = TRUE)) {
  install.packages("reticulate", repos = "https://cloud.r-project.org")
} else {
  if (packageVersion("reticulate") < "1.25") {
    install.packages("reticulate", repos = "https://cloud.r-project.org")
  }
}

### Load and filter NicheNet priors based on Omnipath allinfo consensus_score (secreted-intracelullar)

In [ ]:
%%R -o high_conf_ligands -o high_conf_receptors -o lr_network -o ligand_target_matrix
options(timeout = 3600)

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
})

# vars
# Consensus cutoffs: you can use absolute thresholds, percentile thresholds, or both
USE_PERCENTILES <- TRUE
CONS_PCTL_LIGAND   <- 0.50
CONS_PCTL_RECEPTOR <- 0.50

# Absolute floors (still enforced even if USE_PERCENTILES=TRUE)
MIN_CONS_LIGAND   <- 3
MIN_CONS_RECEPTOR <- 3

# Penalize intracellular annotations
# net_score = max_role_consensus - max_intracellular_consensus
MIN_NET_LIGAND   <- 2
MIN_NET_RECEPTOR <- 1

# Role and topology constraints
REQUIRE_TRANSMITTER <- TRUE
REQUIRE_RECEIVER    <- TRUE
REQUIRE_LIGAND_TOPOLOGY_SEC_OR_PMP <- TRUE     # secreted or plasma membrane peripheral
REQUIRE_RECEPTOR_PMTM <- TRUE                  # plasma membrane transmembrane

# Cache and load NicheNet priors
cache_dir <- Sys.getenv("NICHENET_CACHE", unset = file.path(Sys.getenv("HOME"), ".cache", "nichenetr"))
dir.create(cache_dir, recursive = TRUE, showWarnings = FALSE)
cat("Cache directory:", cache_dir, "\n")

read_and_cache_rds <- function(name, url) {
  out <- file.path(cache_dir, paste0(name, ".rds"))
  if (file.exists(out)) {
    cat("Loading cached:", name, "\n")
    return(readRDS(out))
  }
  cat("Downloading:", name, "\n")
  obj <- readRDS(url(url))
  saveRDS(obj, out)
  obj
}

base_url <- "https://zenodo.org/records/7074291/files/"

lr_network_raw <- read_and_cache_rds(
  "lr_network",
  paste0(base_url, "lr_network_human_21122021.rds?download=1")
)

ligand_target_matrix <- read_and_cache_rds(
  "ligand_target_matrix",
  paste0(base_url, "ligand_target_matrix_nsga2r_final.rds?download=1")
)

weighted_networks <- read_and_cache_rds(
  "weighted_networks",
  paste0(base_url, "weighted_networks_nsga2r_final.rds?download=1")
)

stopifnot(is.data.frame(lr_network_raw))
stopifnot(all(c("from", "to") %in% colnames(lr_network_raw)))
stopifnot(is.matrix(ligand_target_matrix) || is.array(ligand_target_matrix))

# Orient ligand_target_matrix (rows=ligands, cols=targets)
canon_ligs <- c("TGFB1", "TNF", "IFNG", "CXCL12", "VEGFA", "CCL2", "IL6", "PDGFB")
canon_tgts <- c("COL1A1", "COL3A1", "DCN", "LUM", "CXCL10", "ICAM1", "VCAM1", "STAT1")

score_orientation <- function(mat, ligs, tgts) {
  rn <- rownames(mat)
  cn <- colnames(mat)
  pct_lig_rows <- sum(ligs %in% rn) / max(1L, length(ligs))
  pct_lig_cols <- sum(ligs %in% cn) / max(1L, length(ligs))
  pct_tgt_rows <- sum(tgts %in% rn) / max(1L, length(tgts))
  pct_tgt_cols <- sum(tgts %in% cn) / max(1L, length(tgts))
  (pct_lig_rows + pct_tgt_cols) - (pct_lig_cols + pct_tgt_rows)
}

if (score_orientation(t(ligand_target_matrix), canon_ligs, canon_tgts) >
    score_orientation(ligand_target_matrix, canon_ligs, canon_tgts)) {
  cat("Transposing ligand_target_matrix to orient rows=ligands, cols=targets\n")
  ligand_target_matrix <- t(ligand_target_matrix)
  saveRDS(ligand_target_matrix, file.path(cache_dir, "ligand_target_matrix.rds"))
}

cat("\nNicheNet priors loaded:\n")
cat("  lr_network:", nrow(lr_network_raw), "interactions\n")
cat("  ligand_target_matrix:", nrow(ligand_target_matrix), "ligands x", ncol(ligand_target_matrix), "targets\n")


# Fetch OmniPath intercell annotations from web service
cat("\nFetching OmniPath intercell annotations...\n")

# Ask for the columns we actually use, to avoid relying on defaults
intercell_url <- paste0(
  "https://omnipathdb.org/intercell?format=tsv&entity_types=protein&scope=generic",
  "&fields=genesymbol,category,parent,consensus_score,transmitter,receiver,secreted,plasma_membrane_peripheral,plasma_membrane_transmembrane"
)

intercell_cache <- file.path(cache_dir, "omnipath_intercell_strict_fields.tsv")
if (file.exists(intercell_cache)) {
  cat("Loading cached intercell annotations\n")
  intercell <- read_tsv(intercell_cache, show_col_types = FALSE)
} else {
  cat("Downloading intercell annotations...\n")
  intercell <- read_tsv(intercell_url, show_col_types = FALSE)
  write_tsv(intercell, intercell_cache)
}

cat("Intercell annotations:", nrow(intercell), "rows\n")

need_cols <- c(
  "genesymbol","category","parent","consensus_score",
  "transmitter","receiver",
  "secreted","plasma_membrane_peripheral","plasma_membrane_transmembrane"
)
missing_cols <- setdiff(need_cols, colnames(intercell))
if (length(missing_cols) > 0) {
  stop("Missing required intercell columns: ", paste(missing_cols, collapse = ", "))
}


# Build per-gene max scores, topology flags, and net scores
ligand_categories <- c("ligand","cell_surface_ligand")
receptor_categories <- c("receptor")

intercell2 <- intercell %>%
  mutate(
    genesymbol = as.character(genesymbol),
    category = as.character(category),
    parent = as.character(parent),
    consensus_score = suppressWarnings(as.numeric(consensus_score)),
    transmitter = as.logical(transmitter),
    receiver = as.logical(receiver),
    secreted = as.logical(secreted),
    plasma_membrane_peripheral = as.logical(plasma_membrane_peripheral),
    plasma_membrane_transmembrane = as.logical(plasma_membrane_transmembrane),

    is_intracellular = (parent == "intracellular") | (category == "intracellular"),
    is_liglike = (category %in% ligand_categories) | (parent %in% ligand_categories),
    is_reclike = (category %in% receptor_categories) | (parent %in% receptor_categories)
  )

summ <- intercell2 %>%
  group_by(genesymbol) %>%
  summarise(
    max_lig = suppressWarnings(max(consensus_score[is_liglike], na.rm = TRUE)),
    max_rec = suppressWarnings(max(consensus_score[is_reclike], na.rm = TRUE)),
    max_intra = suppressWarnings(max(consensus_score[is_intracellular], na.rm = TRUE)),

    net_lig = max_lig - max_intra,
    net_rec = max_rec - max_intra,

    transmitter_any = any(transmitter %in% TRUE, na.rm = TRUE),
    receiver_any    = any(receiver %in% TRUE, na.rm = TRUE),

    secreted_any = any(secreted %in% TRUE, na.rm = TRUE),
    pmp_any      = any(plasma_membrane_peripheral %in% TRUE, na.rm = TRUE),
    pmtm_any     = any(plasma_membrane_transmembrane %in% TRUE, na.rm = TRUE),
    .groups = "drop"
  )

# Percentile thresholds (computed on finite max scores)
thr_lig <- if (USE_PERCENTILES) {
  quantile(summ$max_lig[is.finite(summ$max_lig)], probs = CONS_PCTL_LIGAND, na.rm = TRUE, names = FALSE)
} else {
  -Inf
}

thr_rec <- if (USE_PERCENTILES) {
  quantile(summ$max_rec[is.finite(summ$max_rec)], probs = CONS_PCTL_RECEPTOR, na.rm = TRUE, names = FALSE)
} else {
  -Inf
}

thr_lig <- max(thr_lig, MIN_CONS_LIGAND)
thr_rec <- max(thr_rec, MIN_CONS_RECEPTOR)

cat("\nThresholds used:\n")
cat("  ligand max_consensus >=", thr_lig, "\n")
cat("  receptor max_consensus >=", thr_rec, "\n")
cat("  MIN_NET_LIGAND:", MIN_NET_LIGAND, "\n")
cat("  MIN_NET_RECEPTOR:", MIN_NET_RECEPTOR, "\n")

# Ligand filter (no blacklists)
lig_filt <- summ %>%
  filter(is.finite(max_lig), max_lig >= thr_lig, is.finite(net_lig), net_lig >= MIN_NET_LIGAND)

if (REQUIRE_TRANSMITTER) {
  lig_filt <- lig_filt %>% filter(transmitter_any)
}
if (REQUIRE_LIGAND_TOPOLOGY_SEC_OR_PMP) {
  lig_filt <- lig_filt %>% filter(secreted_any | pmp_any)
}

# Receptor filter (no blacklists)
rec_filt <- summ %>%
  filter(is.finite(max_rec), max_rec >= thr_rec, is.finite(net_rec), net_rec >= MIN_NET_RECEPTOR)

if (REQUIRE_RECEIVER) {
  rec_filt <- rec_filt %>% filter(receiver_any)
}
if (REQUIRE_RECEPTOR_PMTM) {
  rec_filt <- rec_filt %>% filter(pmtm_any)
}

high_conf_ligands   <- unique(lig_filt$genesymbol)
high_conf_receptors <- unique(rec_filt$genesymbol)

cat("\nHigh-confidence genes:\n")
cat("  Ligands:", length(high_conf_ligands), "\n")
cat("  Receptors:", length(high_conf_receptors), "\n")


# Cross-reference with NicheNet priors and filter LR network
nn_ligands   <- unique(as.character(lr_network_raw$from))
nn_receptors <- unique(as.character(lr_network_raw$to))
ltm_ligands  <- rownames(ligand_target_matrix)

cat("\nOverlap with NicheNet:\n")
cat("  LR ligands in filtered set:", length(intersect(high_conf_ligands, nn_ligands)), "/", length(nn_ligands), "\n")
cat("  LR receptors in filtered set:", length(intersect(high_conf_receptors, nn_receptors)), "/", length(nn_receptors), "\n")
cat("  LTM ligands in filtered set:", length(intersect(high_conf_ligands, ltm_ligands)), "/", length(ltm_ligands), "\n")

lr_network <- lr_network_raw %>%
  filter(from %in% high_conf_ligands, to %in% high_conf_receptors)

cat("\nFiltered lr_network:", nrow(lr_network), "interactions (was", nrow(lr_network_raw), ")\n")

# Diagnostics (print-only)
cat("\nDiagnostic - selected genes:\n")
check_genes <- c("COPA", "CD74", "B2M", "CALM3", "ARF1", "TGFB1", "TNF", "IFNG", "CXCL12", "VEGFA")
for (g in check_genes) {
  in_lig <- g %in% high_conf_ligands
  in_rec <- g %in% high_conf_receptors
  status <- if (in_lig && in_rec) {
    "PASS (ligand + receptor)"
  } else if (in_lig) {
    "PASS (ligand)"
  } else if (in_rec) {
    "PASS (receptor)"
  } else {
    "FILTERED OUT"
  }
  cat("  ", g, ": ", status, "\n", sep = "")
}

cat("\nExporting to Python: high_conf_ligands, high_conf_receptors, lr_network, ligand_target_matrix\n")

### Pass NicheNet priors to python

In [ ]:
# ### Convert NicheNet priors to Python
# Converts R objects from the Omnipath filtering cell into Python DataFrames and sets.

import numpy as np
import pandas as pd

required_exports = ["high_conf_ligands", "high_conf_receptors", "lr_network", "ligand_target_matrix"]
missing = [x for x in required_exports if x not in globals()]
if missing:
    raise NameError(f"Missing R exports: {missing}. Run the Omnipath filtering cell first.")

# Convert high_conf_ligands and high_conf_receptors to Python sets
if hasattr(high_conf_ligands, '__iter__') and not isinstance(high_conf_ligands, str):
    high_conf_ligands_set = set(str(x) for x in high_conf_ligands)
else:
    high_conf_ligands_set = {str(high_conf_ligands)}

if hasattr(high_conf_receptors, '__iter__') and not isinstance(high_conf_receptors, str):
    high_conf_receptors_set = set(str(x) for x in high_conf_receptors)
else:
    high_conf_receptors_set = {str(high_conf_receptors)}

# Convert lr_network to pandas DataFrame
if not isinstance(lr_network, pd.DataFrame):
    lr_network = pd.DataFrame(lr_network)

if "from" in lr_network.columns and "to" in lr_network.columns:
    lr_network["from"] = lr_network["from"].astype(str)
    lr_network["to"] = lr_network["to"].astype(str)
else:
    raise KeyError(f"lr_network missing 'from'/'to' columns. Found: {list(lr_network.columns)}")

# Convert ligand_target_matrix to pandas DataFrame
if not isinstance(ligand_target_matrix, pd.DataFrame):
    import rpy2.robjects as ro
    rownames_r = ro.r["rownames"]
    colnames_r = ro.r["colnames"]
    
    ltm_array = np.asarray(ligand_target_matrix)
    ltm_rownames = [str(x) for x in list(rownames_r(ro.globalenv["ligand_target_matrix"]))]
    ltm_colnames = [str(x) for x in list(colnames_r(ro.globalenv["ligand_target_matrix"]))]
    
    ligand_target_matrix = pd.DataFrame(
        ltm_array,
        index=pd.Index(ltm_rownames, name="ligand"),
        columns=pd.Index(ltm_colnames, name="target")
    )
else:
    ligand_target_matrix.index = ligand_target_matrix.index.astype(str)
    ligand_target_matrix.columns = ligand_target_matrix.columns.astype(str)

# Validate LTM orientation (rows=ligands, cols=targets)
canon_ligs = {"TGFB1", "TNF", "IFNG", "CXCL12", "VEGFA", "CCL2", "IL6", "PDGFB"}
canon_tgts = {"COL1A1", "COL3A1", "DCN", "LUM", "CXCL10", "ICAM1", "VCAM1", "STAT1"}

ligs_in_rows = len(canon_ligs & set(ligand_target_matrix.index))
ligs_in_cols = len(canon_ligs & set(ligand_target_matrix.columns))
tgts_in_rows = len(canon_tgts & set(ligand_target_matrix.index))
tgts_in_cols = len(canon_tgts & set(ligand_target_matrix.columns))

if (ligs_in_cols + tgts_in_rows) > (ligs_in_rows + tgts_in_cols):
    print("LTM appears transposed, fixing...")
    ligand_target_matrix = ligand_target_matrix.T

# Create derived sets
ltm_ligands = set(ligand_target_matrix.index.astype(str))
ltm_targets = set(ligand_target_matrix.columns.astype(str))
lr_ligands = set(lr_network["from"].unique())
lr_receptors = set(lr_network["to"].unique())

# Store in dict for organized access
nichenet_priors = {
    "lr_network": lr_network,
    "ligand_target_matrix": ligand_target_matrix,
    "high_conf_ligands": high_conf_ligands_set,
    "high_conf_receptors": high_conf_receptors_set,
    "ltm_ligands": ltm_ligands,
    "ltm_targets": ltm_targets,
    "lr_ligands": lr_ligands,
    "lr_receptors": lr_receptors,
}

print("NicheNet priors loaded into Python:")
print(f"  high_conf_ligands: {len(high_conf_ligands_set)}")
print(f"  high_conf_receptors: {len(high_conf_receptors_set)}")
print(f"  lr_network: {len(lr_network)} interactions")
print(f"  ligand_target_matrix: {ligand_target_matrix.shape[0]} x {ligand_target_matrix.shape[1]}")

# TIL+ zone NicheNet

### Define TIL+ senders and receivers 

In [ ]:
def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

TILPLUS_GROUP = "TIL_high"

SENDERS = [
    "Malignant_AC-like_IFN_response",
    "Malignant_MES-hypoxia",
    "Astrocyte_inflammatory",
    "Macrophage_IFN_response",
    "Microglia_IFN_response",
    "cDC1", "cDC2",
    "T_cell_IFN_response", "T_cell_CD8", "T_cell_CD4", "T_reg",
]

RECEIVERS = [
    "T_cell_IFN_response", "T_cell_CD8", "T_cell_CD4", "T_reg",
    "NK_KLRC1_pos", "Plasmablast",
    "Endothelial_veinous", "Endothelial_arterial",
    "Perivascular_fibroblast_type-I",
    "Perivascular_fibroblast_type-II",
    "Perivascular_fibroblast_type-III",
    "Pericyte_fibro-like",
    "Macrophage_IFN_response", "Microglia_IFN_response",
    "cDC1", "cDC2",
]

SENDERS_STD = [std_ct(x) for x in SENDERS]
RECEIVERS_STD = [std_ct(x) for x in RECEIVERS]

tilplus_panel = {
    "group": TILPLUS_GROUP,
    "senders": SENDERS_STD,
    "receivers": RECEIVERS_STD,
}

print("TIL+ panel defined:")
print("  group:", tilplus_panel["group"])
print("  senders:", len(tilplus_panel["senders"]))
print("  receivers:", len(tilplus_panel["receivers"]))

### GSEA of HIGH and LOW TIL+ programs

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import requests
import decoupler as dc

# Paths

cache_dir = Path.home() / ".cache" / "msigdb"
cache_dir.mkdir(parents=True, exist_ok=True)

EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
FILTER_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_filtering"

high_targets_file = FILTER_DIR / "TILplus_targets_HIGH_visiumUP_receiverFiltered_LTMsupported.csv"
low_targets_file  = FILTER_DIR / "TILplus_targets_LOW_visiumDOWN_receiverFiltered_LTMsupported.csv"
visium_de_file    = EFS_ROOT / "results/visium/til_zones/deg_pseudobulk_edger/DE_edger_zone_TIL+_TIL_high_vs_TIL_low.csv"

for p in [high_targets_file, low_targets_file, visium_de_file]:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")

# MSigDB GMT download + parse

urls = {
    "hallmark": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.symbols.gmt",
    "kegg":     "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.kegg_medicus.v2023.2.Hs.symbols.gmt",
    "biocarta": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.biocarta.v2023.2.Hs.symbols.gmt",
}
paths = {k: cache_dir / Path(v).name for k, v in urls.items()}

def download_text(url: str, out_path: Path, timeout: int = 120) -> None:
    if out_path.exists() and out_path.stat().st_size > 0:
        return
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    out_path.write_text(r.text)

def parse_gmt(gmt_path: Path) -> dict:
    gene_sets = {}
    with gmt_path.open("r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                gene_sets[parts[0]] = [g for g in parts[2:] if g]
    return gene_sets

for k, url in urls.items():
    download_text(url, paths[k])

hallmark_sets = parse_gmt(paths["hallmark"])
kegg_sets = parse_gmt(paths["kegg"])
biocarta_sets = parse_gmt(paths["biocarta"])

def gmt_to_net(gene_sets: dict) -> pd.DataFrame:
    rows = [(gs, g, 1.0) for gs, genes in gene_sets.items() for g in genes]
    return pd.DataFrame(rows, columns=["source", "target", "weight"])

net = pd.concat(
    [gmt_to_net(hallmark_sets), gmt_to_net(kegg_sets), gmt_to_net(biocarta_sets)],
    ignore_index=True
)

# Inputs: target gene lists + Visium DE

def load_gene_list(path: Path) -> set:
    df = pd.read_csv(path)
    if "gene" in df.columns:
        return set(df["gene"].astype(str))
    if df.shape[1] == 1:
        return set(df.iloc[:, 0].astype(str))
    raise ValueError(f"Could not find a gene column in {path.name}. Columns: {df.columns.tolist()}")

high_genes = load_gene_list(high_targets_file)
low_genes  = load_gene_list(low_targets_file)

visium_de = pd.read_csv(visium_de_file)
for c in ["gene", "logFC"]:
    if c not in visium_de.columns:
        raise ValueError(f"Missing column '{c}' in {visium_de_file.name}. Columns: {visium_de.columns.tolist()}")

visium_de["gene"] = visium_de["gene"].astype(str)
visium_de["logFC"] = pd.to_numeric(visium_de["logFC"], errors="coerce")

# Quick orientation sanity check:
# For TIL_high_vs_low, HIGH genes should skew positive, LOW genes should skew negative.
m_high = np.nanmedian(visium_de.loc[visium_de["gene"].isin(high_genes), "logFC"].to_numpy())
m_low  = np.nanmedian(visium_de.loc[visium_de["gene"].isin(low_genes),  "logFC"].to_numpy())
print(f"Median logFC among HIGH genes: {m_high:.4f}")
print(f"Median logFC among LOW genes : {m_low:.4f}")
if (np.isfinite(m_high) and m_high < 0) or (np.isfinite(m_low) and m_low > 0):
    print("WARNING: logFC sign looks inconsistent with expected TIL_high_vs_low orientation.")


# ULM scoring (flip LOW sign so 'TIL_low program' is positive)

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    p0 = p[ok]
    n = p0.size
    order = np.argsort(p0)
    ranked = p0[order]
    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty_like(p0)
    out[order] = np.clip(adj, 0, 1)
    q[ok] = out
    return q

def run_ulm(gene_set: set, sign: float, label: str) -> tuple[pd.DataFrame, int]:
    df = visium_de.loc[visium_de["gene"].isin(gene_set), ["gene", "logFC"]].dropna().copy()
    if df.empty:
        return pd.DataFrame(), 0

    df = df.drop_duplicates(subset=["gene"], keep="first")
    w = (sign * df.set_index("gene")["logFC"]).astype(float)

    mat = w.to_frame().T
    mat.index = [label]

    est, pvs = dc.run_ulm(
        mat=mat,
        net=net,
        source="source",
        target="target",
        weight="weight",
        min_n=5,
        verbose=False,
    )

    res = pd.DataFrame({
        "gene_set": est.columns.astype(str),
        "score": est.iloc[0].to_numpy(dtype=float),
        "pval":  pvs.iloc[0].to_numpy(dtype=float),
    })
    res["fdr"] = bh_fdr(res["pval"].to_numpy())
    res["collection"] = np.select(
        [
            res["gene_set"].str.startswith("HALLMARK_"),
            res["gene_set"].str.startswith("KEGG_"),
            res["gene_set"].str.startswith("BIOCARTA_"),
        ],
        ["Hallmark", "KEGG", "BioCarta"],
        default="Other",
    )
    return res, mat.shape[1]

ulm_high, n_high = run_ulm(high_genes, sign=+1.0, label="TIL_high_program")
ulm_low,  n_low  = run_ulm(low_genes,  sign=-1.0, label="TIL_low_program")  # SIGN FLIP HERE

out_high = FILTER_DIR / "TILplus_ULM_high_program_all_sets_signAligned.csv"
out_low  = FILTER_DIR / "TILplus_ULM_low_program_all_sets_signAligned.csv"

ulm_high.sort_values("score", ascending=False).to_csv(out_high, index=False)
ulm_low.sort_values("score", ascending=False).to_csv(out_low, index=False)

print(f"\nGene sets loaded: Hallmark={len(hallmark_sets)}, KEGG={len(kegg_sets)}, BioCarta={len(biocarta_sets)}")
print(f"Network rows: {len(net)} (sets={net['source'].nunique()}, genes={net['target'].nunique()})")
print(f"HIGH genes used: {n_high}")
print(f"LOW genes used:  {n_low}")
print(f"\nWrote:\n  {out_high}\n  {out_low}")

def show_top(res: pd.DataFrame, title: str, n: int = 25) -> None:
    if res.empty:
        print(f"\n{title}: no results")
        return
    top = res.sort_values("score", ascending=False).head(n).copy()
    top["score"] = top["score"].round(3)
    top["pval"] = top["pval"].map(lambda x: f"{x:.2e}" if np.isfinite(x) else "nan")
    top["fdr"] = top["fdr"].map(lambda x: f"{x:.2e}" if np.isfinite(x) else "nan")
    print(f"\n{title}: top {n} (score desc, sign-aligned)")
    print(top[["gene_set", "collection", "score", "pval", "fdr"]].to_string(index=False))

show_top(ulm_high, "HIGH program (TIL-high UP)")
show_top(ulm_low,  "LOW program (TIL-low UP)")

### Define pathway filters

In [ ]:
selected_hallmark = [
    "HALLMARK_ANGIOGENESIS",
    "HALLMARK_COMPLEMENT",
    "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
    "HALLMARK_HYPOXIA",
    "HALLMARK_APOPTOSIS",
    "HALLMARK_UV_RESPONSE_DN",
    "HALLMARK_IL6_JAK_STAT3_SIGNALING",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE",
    "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "HALLMARK_TGF_BETA_SIGNALING",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "HALLMARK_IL2_STAT5_SIGNALING",
    "HALLMARK_GLYCOLYSIS",
    "HALLMARK_MTORC1_SIGNALING",
    "HALLMARK_MYOGENESIS",
    "HALLMARK_APICAL_JUNCTION"
]

ecm_genes_brain = [
    "COL1A1","COL1A2","COL3A1","COL4A1","COL4A2","COL5A1","COL6A1","COL6A2","COL6A3","COL7A1",
    "COL8A1","COL10A1","COL11A1","COL12A1","COL14A1","COL15A1","COL16A1","FN1","FBN1","FBN2",
    "TNC","SPARC","POSTN","LAMA1","LAMA2","LAMA3","LAMB1","LAMB2","LAMC1","THBS1","THBS2","THBS4",
    "VTN","NID1","NID2","VCAN","DCN","BGN","LUM","HSPG2","AGRN","GPC1","GPC3","MMP1","MMP2","MMP9",
    "MMP11","MMP14","ADAMTS1","ADAMTS4","ADAMTS5","TIMP1","TIMP2","LOX","LOXL1","SERPINE1","PLAU",
    "PLAT","CSPG4","BCAN","NCAN","HAPLN1","HAPLN2","PRELP","FMOD","COL18A1","LOXL2","LOXL3","HPSE",
    "SDC1","GPC2","CSPG5","PTPRZ1","COL5A2","COL5A3","LAMA4","LAMA5","ACAN","TNR","HAPLN3","HAPLN4",
    "FBLN1","FBLN2","FBLN5","SDC4","SPP1","IGFBP2","IGFBP5","SRGN","MMP3","MMP7","MMP12","ADAMTS8",
    "ADAM8","TIMP3","LOXL4",
]

missing = [h for h in selected_hallmark if h not in hallmark_sets]
if missing:
    raise RuntimeError("Missing in hallmark_sets:\n" + "\n".join(missing))

pathway_genes = set(ecm_genes_brain)
for h in selected_hallmark:
    pathway_genes.update(hallmark_sets[h])

PATHWAY_FILTER = sorted(pathway_genes)
print(f"PATHWAY_FILTER n={len(PATHWAY_FILTER)}")


### Config

In [ ]:
# Cell 1: Configuration

from pathlib import Path

ZONE = "TIL+"

# Set this to False for unbiased targets
USE_PATHWAY_FILTER = True

# If USE_PATHWAY_FILTER is True, PATHWAY_FILTER must already exist as an iterable of gene symbols
if USE_PATHWAY_FILTER:
    if "PATHWAY_FILTER" not in globals():
        raise NameError("USE_PATHWAY_FILTER=True but PATHWAY_FILTER is not defined in the notebook.")
    PATHWAY_FILTER = set(map(str, PATHWAY_FILTER))
else:
    PATHWAY_FILTER = None

FDR_CUTOFF = 0.05
LOGFC_CUTOFF = 0.25

# Axis construction and selection controls
TOP_LIGANDS_PER_DIRECTION = 50
LTM_TOPK_PER_TARGET = 7
TOP_AXES_PER_TARGET = 7
SENDER_TOPK_PER_LIGAND = 7  # set to None to keep all passing senders

# Ligand diversity controls within a target (used in Cell 5)
TOP_LIGANDS_PER_TARGET = 7
TOP_AXES_PER_LIGAND_PER_TARGET = 7

# Catalog truncation control (used in Cell 6)
CATALOG_MAX_AXES_PER_TARGET = 7  # set to None for no truncation, or increase to browse more

# Expression threshold for target->receiver mapping
FRACTION_CUTOFF = 0.05

# IMPORTANT: keep this consistent with other cells (they use /mnt/...)
EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
if not EFS_ROOT.exists():
    raise FileNotFoundError(f"EFS_ROOT does not exist: {EFS_ROOT}")

GATING_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_gating"
OUT_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_analysis" / ZONE.replace("+", "plus").replace("-", "minus")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Zone: {ZONE}")
print(f"Pathway filter: {'None (unbiased)' if PATHWAY_FILTER is None else f'{len(PATHWAY_FILTER)} genes'}")
print(f"FDR_CUTOFF: {FDR_CUTOFF}")
print(f"LOGFC_CUTOFF: {LOGFC_CUTOFF}")
print(f"FRACTION_CUTOFF: {FRACTION_CUTOFF}")
print(f"Output: {OUT_DIR}")

print(f"TOP_LIGANDS_PER_DIRECTION: {TOP_LIGANDS_PER_DIRECTION}")
print(f"LTM_TOPK_PER_TARGET: {LTM_TOPK_PER_TARGET}")
print(f"TOP_AXES_PER_TARGET: {TOP_AXES_PER_TARGET}")
print(f"SENDER_TOPK_PER_LIGAND: {SENDER_TOPK_PER_LIGAND}")

print(f"TOP_LIGANDS_PER_TARGET: {TOP_LIGANDS_PER_TARGET}")
print(f"TOP_AXES_PER_LIGAND_PER_TARGET: {TOP_AXES_PER_LIGAND_PER_TARGET}")
print(f"CATALOG_MAX_AXES_PER_TARGET: {CATALOG_MAX_AXES_PER_TARGET}")

### Zone background gene universe

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Defines these as sorted Python lists of gene symbols for downstream %%R cells:
#   background_high, background_low
#   ligands_high, ligands_low
#   receptors_high, receptors_low
#
# Also keeps a structured dict for inspection:
#   universes_panel

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
MIN_PCT_EXPRESSED = 0.10

required = ["adata_sc", "nichenet_priors", "SENDERS", "RECEIVERS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required variables: {missing}")

if LAYER not in adata_sc.layers:
    raise KeyError(
        f"adata_sc.layers['{LAYER}'] not found. Available layers: {list(adata_sc.layers.keys())}"
    )
for k in [CT_KEY, GROUP_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(
            f"adata_sc.obs['{k}'] not found. Available obs columns (first 60): {list(adata_sc.obs.columns)[:60]}"
        )

for k in ["ltm_targets", "ltm_ligands", "lr_ligands", "lr_receptors"]:
    if k not in nichenet_priors:
        raise KeyError(f"nichenet_priors['{k}'] not found. Available keys: {list(nichenet_priors.keys())}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = set(std_ct(x) for x in SENDERS)
receivers_std = set(std_ct(x) for x in RECEIVERS)

ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

expected_groups = {"TIL_high", "TIL_low"}
groups_present = set(grp.unique())
if not expected_groups.issubset(groups_present):
    raise ValueError(f"Expected {expected_groups} in {GROUP_KEY}, found: {sorted(groups_present)}")

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

ltm_targets = set(map(str, nichenet_priors["ltm_targets"]))
ltm_ligands = set(map(str, nichenet_priors["ltm_ligands"]))
lr_ligands = set(map(str, nichenet_priors["lr_ligands"]))
lr_receptors = set(map(str, nichenet_priors["lr_receptors"]))

candidate_background = ltm_targets
candidate_ligands = ltm_ligands.intersection(lr_ligands)
candidate_receptors = lr_receptors

def expressed_genes(mask_cells: np.ndarray, candidate_genes: set, min_pct: float) -> set:
    idx = np.flatnonzero(mask_cells)
    if idx.size == 0:
        return set()

    genes_present = sorted(candidate_genes.intersection(var_set))
    if not genes_present:
        return set()

    gene_idx = var_names.get_indexer(genes_present)
    Xsub = X[idx, :][:, gene_idx]

    nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
    pct = nnz / float(idx.size)

    keep = pct >= float(min_pct)
    return set(np.asarray(genes_present, dtype=object)[keep])

universes_panel = {}

for g in ["TIL_high", "TIL_low"]:
    sender_mask = (grp == g).to_numpy() & ct.isin(senders_std).to_numpy()
    receiver_mask = (grp == g).to_numpy() & ct.isin(receivers_std).to_numpy()

    bg = expressed_genes(receiver_mask, candidate_background, MIN_PCT_EXPRESSED)
    lig = expressed_genes(sender_mask, candidate_ligands, MIN_PCT_EXPRESSED)
    rec = expressed_genes(receiver_mask, candidate_receptors, MIN_PCT_EXPRESSED)

    universes_panel[g] = {
        "background": bg,
        "ligands": lig,
        "receptors": rec,
        "n_sender_cells": int(sender_mask.sum()),
        "n_receiver_cells": int(receiver_mask.sum()),
    }

background_high = sorted(map(str, universes_panel["TIL_high"]["background"]))
background_low = sorted(map(str, universes_panel["TIL_low"]["background"]))
ligands_high = sorted(map(str, universes_panel["TIL_high"]["ligands"]))
ligands_low = sorted(map(str, universes_panel["TIL_low"]["ligands"]))
receptors_high = sorted(map(str, universes_panel["TIL_high"]["receptors"]))
receptors_low = sorted(map(str, universes_panel["TIL_low"]["receptors"]))

summary_df = pd.DataFrame(
    [
        {
            "group": g,
            "n_sender_cells": universes_panel[g]["n_sender_cells"],
            "n_receiver_cells": universes_panel[g]["n_receiver_cells"],
            "n_background": len(universes_panel[g]["background"]),
            "n_ligands": len(universes_panel[g]["ligands"]),
            "n_receptors": len(universes_panel[g]["receptors"]),
        }
        for g in ["TIL_high", "TIL_low"]
    ]
)

print(summary_df.to_string(index=False))

print("\nUniverse integrity checks:")
print("  candidate_background (ltm_targets):", len(candidate_background))
print("  candidate_ligands (ltm_ligands ∩ lr_ligands):", len(candidate_ligands))
print("  candidate_receptors (lr_receptors):", len(candidate_receptors))
print("  var genes:", len(var_names))
print("  overlap candidate_ligands with var:", len(candidate_ligands.intersection(var_set)))
print("  overlap candidate_receptors with var:", len(candidate_receptors.intersection(var_set)))

if len(background_high) == 0 or len(background_low) == 0:
    raise ValueError("Background is empty for at least one group. Check receiver panel, group labels, and MIN_PCT_EXPRESSED.")
if len(ligands_high) == 0 or len(ligands_low) == 0:
    raise ValueError("Ligands are empty for at least one group. Check sender panel, group labels, and MIN_PCT_EXPRESSED.")
if len(receptors_high) == 0 or len(receptors_low) == 0:
    raise ValueError("Receptors are empty for at least one group. Check receiver panel, group labels, and MIN_PCT_EXPRESSED.")

print("\nExported lists ready for R:")
print("  background_high:", len(background_high), "background_low:", len(background_low))
print("  ligands_high   :", len(ligands_high), "ligands_low   :", len(ligands_low))
print("  receptors_high :", len(receptors_high), "receptors_low :", len(receptors_low))

### Target gene universe

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Builds canonical objects for this session:
#   targets_high, targets_low
#   universes (dict with targets/ligands/background/receptors per direction)
# Keeps de_df for downstream axis construction (Cell 5 expects de_df).

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
ZONE_KEY = ZONE  # CHANGED: was "TIL+"

MIN_PCT_EXPRESSED = 0.10

required = [
    "adata_sc", "nichenet_priors", "SENDERS", "RECEIVERS",
    "de_tables", "PATHWAY_FILTER", "FDR_CUTOFF", "LOGFC_CUTOFF"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects/params (run upstream cells): {missing}")

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

for k in [CT_KEY, GROUP_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found. Available (first 60): {list(adata_sc.obs.columns)[:60]}")

if ZONE_KEY not in de_tables:
    raise KeyError(f"de_tables['{ZONE_KEY}'] not found. Available keys: {list(de_tables.keys())}")

for k in ["ltm_targets", "ltm_ligands", "lr_ligands", "lr_receptors"]:
    if k not in nichenet_priors:
        raise KeyError(f"nichenet_priors['{k}'] not found. Available keys: {list(nichenet_priors.keys())}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = set(std_ct(x) for x in SENDERS)
receivers_std = set(std_ct(x) for x in RECEIVERS)

ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

expected_groups = {"TIL_high", "TIL_low"}
groups_present = set(grp.unique())
if not expected_groups.issubset(groups_present):
    raise ValueError(f"Expected {expected_groups} in {GROUP_KEY}, found: {sorted(groups_present)}")

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

ltm_targets = set(map(str, nichenet_priors["ltm_targets"]))
ltm_ligands = set(map(str, nichenet_priors["ltm_ligands"]))
lr_ligands = set(map(str, nichenet_priors["lr_ligands"]))
lr_receptors = set(map(str, nichenet_priors["lr_receptors"]))

candidate_background = ltm_targets
candidate_ligands = ltm_ligands.intersection(lr_ligands)
candidate_receptors = lr_receptors

def expressed_genes(mask_cells: np.ndarray, candidate_genes: set, min_pct: float) -> set:
    idx = np.flatnonzero(mask_cells)
    if idx.size == 0:
        return set()

    genes_present = sorted(candidate_genes.intersection(var_set))
    if not genes_present:
        return set()

    gene_idx = var_names.get_indexer(genes_present)
    Xsub = X[idx, :][:, gene_idx]

    nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
    pct = nnz / float(idx.size)

    keep = pct >= float(min_pct)
    return set(np.asarray(genes_present, dtype=object)[keep])

panel = {}
for g in ["TIL_high", "TIL_low"]:
    sender_mask = (grp == g).to_numpy() & ct.isin(senders_std).to_numpy()
    receiver_mask = (grp == g).to_numpy() & ct.isin(receivers_std).to_numpy()

    panel[g] = {
        "n_sender_cells": int(sender_mask.sum()),
        "n_receiver_cells": int(receiver_mask.sum()),
        "background": expressed_genes(receiver_mask, candidate_background, MIN_PCT_EXPRESSED),
        "ligands": expressed_genes(sender_mask, candidate_ligands, MIN_PCT_EXPRESSED),
        "receptors": expressed_genes(receiver_mask, candidate_receptors, MIN_PCT_EXPRESSED),
    }

de_df = de_tables[ZONE_KEY].copy()

req = {"gene", "FDR", "logFC"}
if not req.issubset(set(de_df.columns)):
    raise KeyError(f"de_tables['{ZONE_KEY}'] must contain {sorted(req)}. Found: {list(de_df.columns)}")

de_df["gene"] = de_df["gene"].astype(str)
de_df["FDR"] = pd.to_numeric(de_df["FDR"], errors="coerce")
de_df["logFC"] = pd.to_numeric(de_df["logFC"], errors="coerce")

# DE-defined targets by direction (sign anchored to TIL_high_vs_low)
de_high = set(
    de_df.loc[
        (de_df["FDR"] <= float(FDR_CUTOFF)) & (de_df["logFC"] >= float(LOGFC_CUTOFF)),
        "gene"
    ].dropna()
)
de_low = set(
    de_df.loc[
        (de_df["FDR"] <= float(FDR_CUTOFF)) & (de_df["logFC"] <= -float(LOGFC_CUTOFF)),
        "gene"
    ].dropna()
)

# Gate to receiver-expressed background per group
targets_high = de_high.intersection(panel["TIL_high"]["background"])
targets_low = de_low.intersection(panel["TIL_low"]["background"])

targets_high_pre = set(targets_high)
targets_low_pre = set(targets_low)

# Optional pathway filter (apply after background gating)
if PATHWAY_FILTER is not None:
    pf = set(map(str, PATHWAY_FILTER))
    targets_high = targets_high.intersection(pf)
    targets_low = targets_low.intersection(pf)

universes = {
    "targets_high": sorted(map(str, targets_high)),
    "targets_low": sorted(map(str, targets_low)),
    "ligands_high": sorted(map(str, panel["TIL_high"]["ligands"])),
    "ligands_low": sorted(map(str, panel["TIL_low"]["ligands"])),
    "background_high": sorted(map(str, panel["TIL_high"]["background"])),
    "background_low": sorted(map(str, panel["TIL_low"]["background"])),
    "receptors_high": sorted(map(str, panel["TIL_high"]["receptors"])),
    "receptors_low": sorted(map(str, panel["TIL_low"]["receptors"])),
}

summary_df = pd.DataFrame(
    [
        {
            "group": g,
            "n_sender_cells": panel[g]["n_sender_cells"],
            "n_receiver_cells": panel[g]["n_receiver_cells"],
            "n_background": len(panel[g]["background"]),
            "n_ligands": len(panel[g]["ligands"]),
            "n_receptors": len(panel[g]["receptors"]),
        }
        for g in ["TIL_high", "TIL_low"]
    ]
)

print(summary_df.to_string(index=False))

print("\nTargets:")
print("  targets_high (DE∩background) pre pathway:", len(targets_high_pre), "post:", len(targets_high))
print("  targets_low  (DE∩background) pre pathway:", len(targets_low_pre),  "post:", len(targets_low))
print("  pathway filter enabled:", PATHWAY_FILTER is not None)

if PATHWAY_FILTER is not None:
    pf_set = set(map(str, PATHWAY_FILTER))
    print("  targets_high removed by PATHWAY_FILTER:", len(targets_high_pre - pf_set))
    print("  targets_low  removed by PATHWAY_FILTER:", len(targets_low_pre - pf_set))

if len(universes["targets_high"]) == 0 or len(universes["targets_low"]) == 0:
    raise ValueError(
        "Targets empty after DE/background gating and (optional) pathway filter. "
        "Check thresholds, DE orientation, and PATHWAY_FILTER."
    )

### Map targets to receiver cell types by expression fraction

In [ ]:
# %% [markdown]
# ### Map targets to receiver cell types by expression fraction

# %%
import numpy as np
import pandas as pd
import scipy.sparse as sp

# === CLEAR STALE VARIABLES ===
for _v in ["target_receiver_map_high", "target_receiver_map_low", "target_panel_coverage",
           "global_best_high", "global_best_low", "all_celltypes", "valid_celltypes", 
           "excluded_celltypes", "ct", "grp", "X", "var_names", "var_set"]:
    if _v in globals():
        del globals()[_v]

required = ["adata_sc", "targets_high", "targets_low", "RECEIVERS", "FRACTION_CUTOFF"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects/params: {missing}")

# === VERIFY INPUTS ===
print(f"INPUT CHECK:")
print(f"  targets_high: {len(targets_high)} genes")
print(f"  targets_low: {len(targets_low)} genes")
print(f"  RECEIVERS: {len(RECEIVERS)} cell types")
print(f"  FRACTION_CUTOFF: {FRACTION_CUTOFF}")

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"

MIN_SAMPLES_EXPRESSING = 3
PANEL_COVERAGE_THRESHOLD = 0.50

# Minimum cells required for a cell type to be considered in global best computation
# This prevents rare cell types (e.g., 1 cell) from dominating due to noise
MIN_CELLS_FOR_GLOBAL_BEST = 50

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found.")
for k in [CT_KEY, GROUP_KEY, SAMPLE_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found.")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

def is_garbage_ct(ct_name: str) -> bool:
    """Exclude Doublet, Other, Unknown, Unassigned cell types."""
    s = str(ct_name)
    for bad in ["Doublet", "doublet", "Other", "other", "Unknown", "Unassigned"]:
        if bad in s:
            return True
    return False

receivers_std = sorted(set(std_ct(x) for x in RECEIVERS))

# Build fresh from adata_sc
ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)
sample_ids = adata_sc.obs[SAMPLE_KEY].astype(str)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

# Get valid cell types (excluding garbage)
all_celltypes_raw = sorted(ct.unique())
valid_celltypes = [c for c in all_celltypes_raw if not is_garbage_ct(c)]
excluded_celltypes = [c for c in all_celltypes_raw if is_garbage_ct(c)]

print(f"\nCELL TYPE FILTERING:")
print(f"  Raw cell types: {len(all_celltypes_raw)}")
print(f"  Valid cell types (no Doublet/Other): {len(valid_celltypes)}")
print(f"  Excluded: {len(excluded_celltypes)}")

# Compute cell counts per cell type per group
celltype_counts = {}
for group_label in ["TIL_high", "TIL_low"]:
    celltype_counts[group_label] = {}
    for celltype in valid_celltypes:
        m = (grp == group_label).to_numpy() & (ct == celltype).to_numpy()
        n_cells = m.sum()
        celltype_counts[group_label][celltype] = n_cells

# Filter valid cell types for global best by minimum cell count
valid_celltypes_for_global = {}
for group_label in ["TIL_high", "TIL_low"]:
    valid_celltypes_for_global[group_label] = [
        c for c in valid_celltypes 
        if celltype_counts[group_label][c] >= MIN_CELLS_FOR_GLOBAL_BEST
    ]
    n_excluded_by_count = len(valid_celltypes) - len(valid_celltypes_for_global[group_label])
    print(f"  {group_label}: {len(valid_celltypes_for_global[group_label])} cell types with >= {MIN_CELLS_FOR_GLOBAL_BEST} cells ({n_excluded_by_count} excluded)")

def build_trm_for_group(group_label: str, targets_panel) -> pd.DataFrame:
    """Build target-receiver map for panel receivers only."""
    targets_panel = [str(x) for x in list(targets_panel)]
    targets_present = [t for t in targets_panel if t in var_set]
    if len(targets_present) == 0:
        raise ValueError(f"No targets found in var_names for {group_label}.")

    print(f"  {group_label}: {len(targets_present)} targets in var_names")
    
    gene_idx = var_names.get_indexer(targets_present)
    rows = []

    for rct in receivers_std:
        m = (grp == group_label).to_numpy() & (ct == rct).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue

        sample_sub = sample_ids.iloc[idx].values
        n_samples = len(np.unique(sample_sub))

        Xsub = X[idx, :][:, gene_idx]
        nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
        pct = nnz / float(idx.size)
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        Xsub_dense = Xsub.toarray()

        for i, target in enumerate(targets_present):
            gene_expr = Xsub_dense[:, i]
            n_samples_expressing = len(set(sample_sub[gene_expr > 0]))
            sample_frac = n_samples_expressing / n_samples if n_samples > 0 else 0.0

            if pct[i] >= float(FRACTION_CUTOFF) and n_samples_expressing >= MIN_SAMPLES_EXPRESSING:
                rows.append({
                    "target": target,
                    "receiver_celltype": rct,
                    "target_pct_expr": float(pct[i]),
                    "target_mean_expr": float(mean_expr[i]),
                    "n_samples": n_samples,
                    "n_samples_expressing": n_samples_expressing,
                    "sample_frac": float(sample_frac),
                })

    if not rows:
        return pd.DataFrame(columns=[
            "target", "receiver_celltype", "target_pct_expr", "target_mean_expr",
            "n_samples", "n_samples_expressing", "sample_frac", "specificity", "receiver_rank_score"
        ])

    out = pd.DataFrame(rows)

    # Specificity within panel
    target_mean_all = out.groupby("target")["target_mean_expr"].transform("mean").replace(0, np.nan)
    out["specificity"] = (out["target_mean_expr"] / target_mean_all).fillna(0)

    # Receiver rank score
    out["_mr"] = out.groupby("target")["target_mean_expr"].rank(ascending=False)
    out["_pr"] = out.groupby("target")["target_pct_expr"].rank(ascending=False)
    out["_sfr"] = out.groupby("target")["sample_frac"].rank(ascending=False)
    out["_spr"] = out.groupby("target")["specificity"].rank(ascending=False)
    out["receiver_rank_score"] = 0.35/out["_mr"] + 0.25/out["_pr"] + 0.20/out["_sfr"] + 0.20/out["_spr"]
    out = out.drop(columns=["_mr", "_pr", "_sfr", "_spr"])

    return out

def compute_global_best(group_label: str, targets_list) -> pd.DataFrame:
    """Find global best cell type per target.
    
    Only considers cell types with >= MIN_CELLS_FOR_GLOBAL_BEST cells
    to avoid noise from rare populations.
    """
    targets_list = [str(x) for x in targets_list if str(x) in var_set]
    if not targets_list:
        return pd.DataFrame(columns=["target", "global_best_celltype", "global_best_mean_expr", "global_best_n_cells"])

    gene_idx = var_names.get_indexer(targets_list)
    best = {t: (None, 0.0, 0) for t in targets_list}  # (celltype, mean_expr, n_cells)

    # Only use cell types with sufficient cells
    celltypes_to_check = valid_celltypes_for_global[group_label]
    
    for celltype in celltypes_to_check:
        m = (grp == group_label).to_numpy() & (ct == celltype).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue
        n_cells = idx.size
        Xsub = X[idx, :][:, gene_idx]
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        for i, target in enumerate(targets_list):
            if mean_expr[i] > best[target][1]:
                best[target] = (celltype, float(mean_expr[i]), n_cells)

    results = [{"target": t, "global_best_celltype": v[0], "global_best_mean_expr": v[1], "global_best_n_cells": v[2]} 
               for t, v in best.items() if v[0] is not None]
    return pd.DataFrame(results)

def add_coverage(trm: pd.DataFrame, global_best: pd.DataFrame) -> pd.DataFrame:
    if trm.empty:
        return trm
    trm = trm.merge(global_best, on="target", how="left")
    panel_best = trm.groupby("target")["target_mean_expr"].transform("max")
    trm["panel_coverage_ratio"] = np.where(
        trm["global_best_mean_expr"] > 0,
        panel_best / trm["global_best_mean_expr"],
        1.0
    )
    trm["panel_is_adequate"] = trm["panel_coverage_ratio"] >= PANEL_COVERAGE_THRESHOLD
    trm["coverage_flag"] = np.where(trm["panel_is_adequate"], "OK", "LOW_COVERAGE")
    return trm

# === BUILD TRMs ===
print("\nBUILDING TARGET-RECEIVER MAPS:")
target_receiver_map_high = build_trm_for_group("TIL_high", targets_high)
target_receiver_map_low = build_trm_for_group("TIL_low", targets_low)

# === COMPUTE GLOBAL BEST ===
print(f"\nCOMPUTING GLOBAL BEST (min {MIN_CELLS_FOR_GLOBAL_BEST} cells, excluding Doublets/Other):")
global_best_high = compute_global_best("TIL_high", targets_high)
global_best_low = compute_global_best("TIL_low", targets_low)

# Verify no garbage in global_best
for name, gb in [("high", global_best_high), ("low", global_best_low)]:
    if not gb.empty:
        garbage = [ct for ct in gb["global_best_celltype"].dropna().unique() if is_garbage_ct(ct)]
        if garbage:
            raise RuntimeError(f"BUG: Garbage cell types in global_best_{name}: {garbage}")
        print(f"  global_best_{name}: {len(gb)} targets, no garbage cell types ✓")

# === ADD COVERAGE ===
target_receiver_map_high = add_coverage(target_receiver_map_high, global_best_high)
target_receiver_map_low = add_coverage(target_receiver_map_low, global_best_low)

# === SUMMARY ===
print("\nTARGET-RECEIVER MAP SUMMARY:")
for name, trm in [("TIL_high", target_receiver_map_high), ("TIL_low", target_receiver_map_low)]:
    print(f"\n  {name}:")
    print(f"    rows: {len(trm)}")
    print(f"    targets: {trm['target'].nunique()}")
    print(f"    receivers: {trm['receiver_celltype'].nunique()}")
    if "coverage_flag" in trm.columns:
        n_ok = (trm["coverage_flag"] == "OK").sum()
        n_low = (trm["coverage_flag"] == "LOW_COVERAGE").sum()
        print(f"    coverage: OK={n_ok}, LOW_COVERAGE={n_low}")

# Show LOW_COVERAGE targets
print("\nLOW_COVERAGE TARGETS:")
for name, trm in [("TIL_high", target_receiver_map_high), ("TIL_low", target_receiver_map_low)]:
    low = trm[trm["coverage_flag"] == "LOW_COVERAGE"][["target", "global_best_celltype", "global_best_n_cells", "panel_coverage_ratio"]].drop_duplicates()
    if low.empty:
        print(f"  {name}: None")
    else:
        print(f"  {name}:")
        for _, r in low.iterrows():
            print(f"    {r['target']}: global_best={r['global_best_celltype']} (n={r['global_best_n_cells']}), coverage={r['panel_coverage_ratio']:.1%}")

if target_receiver_map_high.empty or target_receiver_map_low.empty:
    raise ValueError("One or both target_receiver_maps are empty!")

# Store for downstream
target_panel_coverage = pd.concat([
    global_best_high.assign(direction="TIL_high"),
    global_best_low.assign(direction="TIL_low")
], ignore_index=True)

print("\n✓ Created: target_receiver_map_high, target_receiver_map_low, target_panel_coverage")

### Build SCE (python)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmwrite

if "adata_sc" not in globals():
    raise NameError("adata_sc not found")
if "SENDERS" not in globals() or "RECEIVERS" not in globals():
    raise NameError("SENDERS and/or RECEIVERS not found")

COUNTS_LAYER = "unnormalized_counts"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"
BATCH_KEY = "orig.ident"

if COUNTS_LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{COUNTS_LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

for k in [CT_KEY, GROUP_KEY, SAMPLE_KEY, BATCH_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found. Available obs cols (first 50): {list(adata_sc.obs.columns)[:50]}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

def drop_bad_ct(ct_list):
    out = []
    for ct in ct_list:
        s = str(ct)
        if s.startswith("Doublet"):
            continue
        if s.startswith("Other"):
            continue
        out.append(s)
    return sorted(set(out))

senders_py = drop_bad_ct(list(SENDERS))
receivers_py = drop_bad_ct(list(RECEIVERS))

senders_std = [std_ct(x) for x in senders_py]
receivers_std = [std_ct(x) for x in receivers_py]
panel_ct_std = sorted(set(senders_std) | set(receivers_std))

ct_std = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

m = grp.isin(["TIL_low", "TIL_high"]) & ct_std.isin(panel_ct_std)
n_keep = int(m.sum())
if n_keep == 0:
    raise ValueError("No cells left after filtering to (TIL_low/TIL_high) and sender/receiver panel celltypes.")

adata_sub = adata_sc[m].copy()

X = adata_sub.layers[COUNTS_LAYER]
if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()

EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
zone_dir = ZONE.replace("+", "plus").replace("-", "minus")  # CHANGED: derive from ZONE
out_dir = EFS_ROOT / "results" / "visium" / "til_zones" / "nichenet_filtering" / f"mnn_{zone_dir}_input"  # CHANGED: use zone_dir
out_dir.mkdir(parents=True, exist_ok=True)

for fname in ["counts.mtx", "features.tsv", "barcodes.tsv", "metadata.csv", "senders.txt", "receivers.txt"]:
    fpath = out_dir / fname
    if fpath.exists():
        fpath.unlink()

counts_gene_cell = X.T.tocsc()
mmwrite(str(out_dir / "counts.mtx"), counts_gene_cell)

genes = pd.Series(adata_sub.var_names.astype(str))
genes.to_csv(out_dir / "features.tsv", sep="\t", index=False, header=False)

cells = pd.Series(adata_sub.obs_names.astype(str))
cells.to_csv(out_dir / "barcodes.tsv", sep="\t", index=False, header=False)

ct_sub_std = adata_sub.obs[CT_KEY].astype(str).map(std_ct).values

meta = pd.DataFrame(
    {
        "cell": cells.values,
        "celltype": ct_sub_std,
        "sample": adata_sub.obs[SAMPLE_KEY].astype(str).values,
        "group": adata_sub.obs[GROUP_KEY].astype(str).values,
        "batch": adata_sub.obs[BATCH_KEY].astype(str).values,
    }
)
meta.to_csv(out_dir / "metadata.csv", index=False)

pd.Series(senders_std).to_csv(out_dir / "senders.txt", index=False, header=False)
pd.Series(receivers_std).to_csv(out_dir / "receivers.txt", index=False, header=False)

# ADDED: Export for R cell
SCE_INPUT_DIR = f"mnn_{zone_dir}_input"

print("Exported for SCE:")
print("  out_dir:", str(out_dir))
print("  genes x cells:", counts_gene_cell.shape[0], "x", counts_gene_cell.shape[1])
print("  nnz:", int(counts_gene_cell.nnz))
print("  groups:", dict(pd.Series(meta["group"]).value_counts()))
print("  celltypes in panel (senders, receivers):", len(senders_std), ",", len(receivers_std))

### Build SCE (R)

In [ ]:
%%R -i SCE_INPUT_DIR
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(S4Vectors)
  library(dplyr)
  library(readr)
  library(multinichenetr)
})

EFS_ROOT  <- "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd"
in_dir <- file.path(EFS_ROOT, "results", "visium", "til_zones", "nichenet_filtering", SCE_INPUT_DIR)  # CHANGED: use variable
stopifnot(dir.exists(in_dir))

counts <- readMM(file.path(in_dir, "counts.mtx"))
counts <- as(counts, "dgCMatrix")

feat <- read_tsv(file.path(in_dir, "features.tsv"), col_names = FALSE, show_col_types = FALSE)
genes <- if (ncol(feat) == 1) feat[[1]] else feat[[2]]

cells <- read_tsv(file.path(in_dir, "barcodes.tsv"), col_names = FALSE, show_col_types = FALSE)[[1]]
meta  <- read_csv(file.path(in_dir, "metadata.csv"), show_col_types = FALSE)

stopifnot(nrow(counts) == length(genes))
stopifnot(ncol(counts) == length(cells))

rownames(counts) <- genes
colnames(counts) <- cells

stopifnot("cell" %in% colnames(meta))
meta <- meta %>% tibble::column_to_rownames("cell")
meta <- meta[colnames(counts), , drop = FALSE]

stopifnot(all(c("celltype", "sample", "group", "batch") %in% colnames(meta)))

meta$celltype_mnn <- make.names(meta$celltype)

sce <- SingleCellExperiment(
  assays = list(counts = counts),
  colData = S4Vectors::DataFrame(meta),
  rowData = S4Vectors::DataFrame(gene = rownames(counts))
)

stopifnot("sample" %in% colnames(colData(sce)))
stopifnot("celltype_mnn" %in% colnames(colData(sce)))
stopifnot("group" %in% colnames(colData(sce)))

colData(sce)$sample_id <- as.character(colData(sce)$sample)
colData(sce)$celltype  <- as.character(colData(sce)$celltype_mnn)

if (anyNA(colData(sce)$sample_id)) stop("sample_id has NA values (check sample in metadata.csv).")
if (anyNA(colData(sce)$celltype))  stop("celltype has NA values (check celltype in metadata.csv).")

senders_oi   <- make.names(readLines(file.path(in_dir, "senders.txt")))
receivers_oi <- make.names(readLines(file.path(in_dir, "receivers.txt")))

ct_levels <- unique(colData(sce)$celltype)
missing_senders   <- setdiff(senders_oi, ct_levels)
missing_receivers <- setdiff(receivers_oi, ct_levels)
if (length(missing_senders) > 0) {
  stop("Senders not found in sce$celltype: ", paste(missing_senders, collapse = ", "))
}
if (length(missing_receivers) > 0) {
  stop("Receivers not found in sce$celltype: ", paste(missing_receivers, collapse = ", "))
}

cat("SCE:", dim(sce)[1], "genes x", dim(sce)[2], "cells\n")
cat("Groups (group):\n")
print(table(colData(sce)$group, useNA = "ifany"))
cat("\nCelltypes (celltype) top 15:\n")
print(sort(table(colData(sce)$celltype), decreasing = TRUE)[1:15])

cat("\nSenders (n=", length(senders_oi), "):\n", sep = "")
print(senders_oi)
cat("\nReceivers (n=", length(receivers_oi), "):\n", sep = "")
print(receivers_oi)

stopifnot(exists("lr_network"), exists("ligand_target_matrix"), exists("weighted_networks"))
cat("\nPriors OK:\n")
cat("  lr_network dim:", paste(dim(lr_network), collapse = " x "), "\n")
cat("  ligand_target_matrix dim:", paste(dim(ligand_target_matrix), collapse = " x "), "\n")
cat("  weighted_networks class:", paste(class(weighted_networks), collapse = ","), "\n")

### Compute ligand activity per direction

In [ ]:
# Make R-safe copies (rpy2 cannot convert Python sets)
targets_high_r = sorted(list(targets_high))
targets_low_r  = sorted(list(targets_low))

background_high_r = list(background_high)
background_low_r  = list(background_low)
ligands_high_r    = list(ligands_high)
ligands_low_r     = list(ligands_low)

print("R-ready objects:")
print("  targets_high_r:", type(targets_high_r), len(targets_high_r))
print("  targets_low_r :", type(targets_low_r),  len(targets_low_r))
print("  background_high_r:", type(background_high_r), len(background_high_r))
print("  background_low_r :", type(background_low_r),  len(background_low_r))
print("  ligands_high_r:", type(ligands_high_r), len(ligands_high_r))
print("  ligands_low_r :", type(ligands_low_r),  len(ligands_low_r))

In [ ]:
%%R -i targets_high_r -i targets_low_r -i background_high_r -i background_low_r -i ligands_high_r -i ligands_low_r
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
  library(nichenetr)
})

stopifnot(exists("ligand_target_matrix"))

to_chr_unique <- function(x) {
  unique(as.character(unlist(x, use.names = FALSE)))
}

targets_high_chr    <- to_chr_unique(targets_high_r)
targets_low_chr     <- to_chr_unique(targets_low_r)
background_high_chr <- to_chr_unique(background_high_r)
background_low_chr  <- to_chr_unique(background_low_r)
ligands_high_chr    <- to_chr_unique(ligands_high_r)
ligands_low_chr     <- to_chr_unique(ligands_low_r)

ltm <- ligand_target_matrix
ltm_ligands <- rownames(ltm)
ltm_targets <- colnames(ltm)

targets_high_use <- intersect(targets_high_chr, ltm_targets)
targets_low_use  <- intersect(targets_low_chr,  ltm_targets)

bg_high_use <- intersect(background_high_chr, ltm_targets)
bg_low_use  <- intersect(background_low_chr,  ltm_targets)

lig_high_use <- intersect(ligands_high_chr, ltm_ligands)
lig_low_use  <- intersect(ligands_low_chr,  ltm_ligands)

get_top_ltm_ligands <- function(targets_use, ltm, k_per_target) {
  out <- character()
  if (length(targets_use) == 0) return(out)

  for (t in targets_use) {
    w <- ltm[, t, drop = TRUE]
    w <- w[w > 0]
    if (length(w) == 0) next
    top <- sort(w, decreasing = TRUE)
    if (length(top) > k_per_target) top <- top[seq_len(k_per_target)]
    out <- c(out, names(top))
  }
  unique(out)
}

LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY <- 10

top_ltm_high <- get_top_ltm_ligands(targets_high_use, ltm, LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY)
top_ltm_low  <- get_top_ltm_ligands(targets_low_use,  ltm, LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY)

potential_high <- union(lig_high_use, top_ltm_high)
potential_low  <- union(lig_low_use,  top_ltm_low)

cat("Ligand activity inputs after LTM intersection + LTM augmentation:\n")
cat("  LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY=", LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY, "\n", sep = "")
cat("  TIL_high: targets=", length(targets_high_use),
    " background=", length(bg_high_use),
    " sender_panel_ligs=", length(lig_high_use),
    " ltm_top_ligs=", length(top_ltm_high),
    " potential_ligs=", length(potential_high), "\n", sep = "")
cat("  TIL_low : targets=", length(targets_low_use),
    " background=", length(bg_low_use),
    " sender_panel_ligs=", length(lig_low_use),
    " ltm_top_ligs=", length(top_ltm_low),
    " potential_ligs=", length(potential_low), "\n\n", sep = "")

if (length(targets_high_use) < 5) stop("Too few TIL_high targets after intersecting with ligand_target_matrix columns.")
if (length(targets_low_use)  < 5) stop("Too few TIL_low targets after intersecting with ligand_target_matrix columns.")
if (length(bg_high_use) < 50) stop("Too few TIL_high background genes after intersecting with ligand_target_matrix columns.")
if (length(bg_low_use)  < 50) stop("Too few TIL_low background genes after intersecting with ligand_target_matrix columns.")
if (length(potential_high) < 20) stop("Too few TIL_high potential ligands after augmentation.")
if (length(potential_low)  < 20) stop("Too few TIL_low potential ligands after augmentation.")

ligand_activities_high <- predict_ligand_activities(
  geneset = targets_high_use,
  background_expressed_genes = bg_high_use,
  potential_ligands = potential_high,
  ligand_target_matrix = ltm
) %>%
  as_tibble() %>%
  mutate(
    direction = "TIL_high",
    test_ligand = as.character(test_ligand),
    aupr_corrected = as.numeric(aupr_corrected),
    in_sender_panel = test_ligand %in% lig_high_use,
    in_ltm_top = test_ligand %in% top_ltm_high
  ) %>%
  arrange(desc(aupr_corrected), test_ligand) %>%
  mutate(rank = dplyr::row_number()) %>%
  select(test_ligand, direction, aupr_corrected, rank, in_sender_panel, in_ltm_top, everything())

ligand_activities_low <- predict_ligand_activities(
  geneset = targets_low_use,
  background_expressed_genes = bg_low_use,
  potential_ligands = potential_low,
  ligand_target_matrix = ltm
) %>%
  as_tibble() %>%
  mutate(
    direction = "TIL_low",
    test_ligand = as.character(test_ligand),
    aupr_corrected = as.numeric(aupr_corrected),
    in_sender_panel = test_ligand %in% lig_low_use,
    in_ltm_top = test_ligand %in% top_ltm_low
  ) %>%
  arrange(desc(aupr_corrected), test_ligand) %>%
  mutate(rank = dplyr::row_number()) %>%
  select(test_ligand, direction, aupr_corrected, rank, in_sender_panel, in_ltm_top, everything())

ligand_activities <- bind_rows(ligand_activities_high, ligand_activities_low)

cat("Ligand activities computed:\n")
cat("  rows:", nrow(ligand_activities), "\n")
cat("  TIL_high:", nrow(ligand_activities_high), "\n")
cat("  TIL_low :", nrow(ligand_activities_low), "\n\n")

cat("Top 10 (TIL_high):\n")
print(ligand_activities_high %>% slice_head(n = 10) %>% select(test_ligand, aupr_corrected, rank, in_sender_panel, in_ltm_top))
cat("\nTop 10 (TIL_low):\n")
print(ligand_activities_low %>% slice_head(n = 10) %>% select(test_ligand, aupr_corrected, rank, in_sender_panel, in_ltm_top))

### Ligand-receptor DE per cell type

In [ ]:
# Pass zone prefix to R for file naming
zone_prefix = ZONE.replace("+", "plus").replace("-", "minus")
print(f"Zone prefix for R: {zone_prefix}")

In [ ]:
%%R -i zone_prefix
# LR-DE: edgeR pseudobulk per celltype, TIL_high vs TIL_low
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(SummarizedExperiment)
  library(S4Vectors)
  library(dplyr)
  library(tibble)
  library(readr)
  library(edgeR)
})

stopifnot(exists("sce"), exists("senders_oi"), exists("receivers_oi"))
stopifnot(exists("lr_network"), exists("ligand_target_matrix"))
stopifnot(exists("ligand_activities"))
stopifnot("counts" %in% SummarizedExperiment::assayNames(sce))

EFS_ROOT  <- "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd"
FILTER_DIR <- file.path(EFS_ROOT, "results", "visium", "til_zones", "nichenet_filtering")
dir.create(FILTER_DIR, recursive = TRUE, showWarnings = FALSE)

GROUPS <- c("TIL_low","TIL_high")

MIN_CELLS_PER_SAMPLE  <- 5
MIN_SAMPLES_PER_GROUP <- 3

CT_COL     <- "celltype"
GROUP_COL  <- "group"
SAMPLE_COL <- "sample_id"

cd0 <- as.data.frame(colData(sce))
stopifnot(CT_COL %in% colnames(cd0), GROUP_COL %in% colnames(cd0), SAMPLE_COL %in% colnames(cd0))

cd0[[CT_COL]]     <- as.character(cd0[[CT_COL]])
cd0[[GROUP_COL]]  <- as.character(cd0[[GROUP_COL]])
cd0[[SAMPLE_COL]] <- as.character(cd0[[SAMPLE_COL]])

bad_sample <- is.na(cd0[[SAMPLE_COL]]) | cd0[[SAMPLE_COL]] == ""
if (any(bad_sample)) {
  ex <- unique(cd0[[SAMPLE_COL]][bad_sample])[1:min(10, sum(bad_sample))]
  stop(paste0("sample_id contains NA/empty values (n=", sum(bad_sample), "). Examples: ", paste(ex, collapse=", ")))
}

bad_group <- is.na(cd0[[GROUP_COL]]) | !(cd0[[GROUP_COL]] %in% GROUPS)
if (any(bad_group)) {
  ex <- unique(cd0[[GROUP_COL]][bad_group])[1:min(10, sum(bad_group))]
  stop(paste0("group has NA or unexpected labels (n=", sum(bad_group), "). Examples: ", paste(ex, collapse=", ")))
}

cat("Using columns:\n")
cat("  celltype:", CT_COL, "\n")
cat("  group   :", GROUP_COL, "\n")
cat("  sample  :", SAMPLE_COL, "\n")
cat("Cells:", nrow(cd0), "\n\n")

cat("Group counts in sce:\n")
print(table(cd0[[GROUP_COL]], useNA = "ifany"))
cat("\n")

assays_avail <- assayNames(sce)
if (!("counts" %in% assays_avail)) {
  stop(paste0("sce is missing assay 'counts'. Available assays: ", paste(assays_avail, collapse=", ")))
}
counts_all <- assay(sce, "counts")
if (!is(counts_all, "dgCMatrix")) counts_all <- as(counts_all, "CsparseMatrix")
counts_all <- as(counts_all, "dgCMatrix")

cat("Counts matrix:\n")
cat("  class:", class(counts_all)[1], "\n")
cat("  dim  :", nrow(counts_all), "x", ncol(counts_all), "\n\n")

cat("Ligand activities directions:\n")
print(table(as.character(ligand_activities$direction), useNA="ifany"))
cat("\n")

# Harmonize lr_network columns
lr_cols <- colnames(lr_network)
lig_col <- if ("from" %in% lr_cols) "from" else if ("ligand" %in% lr_cols) "ligand" else stop("lr_network missing ligand column (from/ligand).")
rec_col <- if ("to" %in% lr_cols) "to" else if ("receptor" %in% lr_cols) "receptor" else stop("lr_network missing receptor column (to/receptor).")

lr_tbl <- lr_network %>%
  dplyr::distinct(.data[[lig_col]], .data[[rec_col]])
colnames(lr_tbl) <- c("from","to")

# Feasible ligands: use NicheNet activity list, constrained to LTM rows + sce genes
ltm <- ligand_target_matrix
ltm_ligands <- rownames(ltm)
ltm_targets <- colnames(ltm)

feasible_ligands <- intersect(
  unique(as.character(ligand_activities$test_ligand)),
  intersect(ltm_ligands, rownames(counts_all))
)

# Feasible receptors: receptors paired to feasible ligands, constrained to sce genes
feasible_receptors <- intersect(
  unique(lr_tbl$to[lr_tbl$from %in% feasible_ligands]),
  rownames(counts_all)
)

cat("DE universes:\n")
cat("  feasible ligands   :", length(feasible_ligands), "\n")
cat("  feasible receptors :", length(feasible_receptors), "\n\n")

if (length(feasible_ligands) < 20) stop("Too few feasible ligands after intersecting activity ligands with LTM rows and sce genes.")
if (length(feasible_receptors) < 20) stop("Too few feasible receptors after intersecting LR receptors with sce genes.")

# Fast pseudobulk using sparse model matrix
pseudobulk_by_sample <- function(counts_gene_cell, sample_vec) {
  sample_f <- factor(sample_vec)
  M <- Matrix::sparse.model.matrix(~0 + sample_f)
  colnames(M) <- levels(sample_f)
  pb <- counts_gene_cell %*% M
  as(pb, "dgCMatrix")
}

run_edger_for_celltype <- function(celltype_name, genes_keep, mode_label) {
  tryCatch({

    idx_ct <- which(cd0[[CT_COL]] == celltype_name & cd0[[GROUP_COL]] %in% GROUPS)
    if (length(idx_ct) == 0) {
      return(list(status="skip", reason="no cells after filtering", de=NULL, meta=NULL))
    }

    meta_ct <- cd0[idx_ct, , drop=FALSE]

    cells_per_sample <- meta_ct %>%
      dplyr::count(.data[[SAMPLE_COL]], name="n_cells")

    keep_samples <- cells_per_sample %>%
      dplyr::filter(n_cells >= MIN_CELLS_PER_SAMPLE) %>%
      dplyr::pull(.data[[SAMPLE_COL]])

    if (length(keep_samples) == 0) {
      return(list(status="skip", reason=paste0("no samples with >= ", MIN_CELLS_PER_SAMPLE, " cells"), de=NULL, meta=cells_per_sample))
    }

    keep_mask <- meta_ct[[SAMPLE_COL]] %in% keep_samples
    meta_ct <- meta_ct[keep_mask, , drop=FALSE]
    idx_ct  <- idx_ct[keep_mask]

    sample_group_tbl <- meta_ct %>%
      dplyr::group_by(.data[[SAMPLE_COL]]) %>%
      dplyr::summarise(
        n_cells = dplyr::n(),
        n_groups = dplyr::n_distinct(.data[[GROUP_COL]]),
        group = dplyr::first(.data[[GROUP_COL]]),
        .groups="drop"
      )

    bad <- sample_group_tbl %>% dplyr::filter(n_groups > 1)
    if (nrow(bad) > 0) {
      return(list(status="skip", reason="sample maps to >1 group within this celltype", de=NULL, meta=sample_group_tbl))
    }

    n_by_group <- sample_group_tbl %>% dplyr::count(group, name="n_samples")
    n_low  <- n_by_group %>% dplyr::filter(group=="TIL_low")  %>% dplyr::pull(n_samples)
    n_high <- n_by_group %>% dplyr::filter(group=="TIL_high") %>% dplyr::pull(n_samples)
    n_low  <- ifelse(length(n_low)==0, 0, n_low)
    n_high <- ifelse(length(n_high)==0, 0, n_high)

    if (n_low < MIN_SAMPLES_PER_GROUP || n_high < MIN_SAMPLES_PER_GROUP) {
      return(list(
        status="skip",
        reason=paste0("too few samples per group (low=", n_low, ", high=", n_high, ")"),
        de=NULL,
        meta=sample_group_tbl
      ))
    }

    genes_keep <- intersect(as.character(genes_keep), rownames(counts_all))
    if (length(genes_keep) < 10) {
      return(list(status="skip", reason="too few genes in universe after intersecting with sce genes", de=NULL, meta=sample_group_tbl))
    }

    counts_sub <- counts_all[genes_keep, idx_ct, drop=FALSE]
    pb <- pseudobulk_by_sample(counts_sub, meta_ct[[SAMPLE_COL]])

    group_vec <- sample_group_tbl$group
    names(group_vec) <- sample_group_tbl[[SAMPLE_COL]]
    group_vec <- group_vec[colnames(pb)]

    if (any(is.na(group_vec))) {
      na_samples <- colnames(pb)[is.na(group_vec)]
      return(list(
        status="error",
        reason=paste0("group_vec has NA after aligning to pseudobulk columns. Examples: ", paste(head(na_samples, 10), collapse=", ")),
        de=NULL,
        meta=sample_group_tbl
      ))
    }

    group_vec <- factor(group_vec, levels=GROUPS)

    y <- DGEList(counts = as.matrix(pb))
    design <- model.matrix(~ group_vec)

    keep <- filterByExpr(y, design=design)
    y <- y[keep, , keep.lib.sizes=FALSE]
    y <- calcNormFactors(y)

    y <- estimateDisp(y, design, robust=TRUE)
    fit <- glmQLFit(y, design, robust=TRUE)
    qlf <- glmQLFTest(fit, coef=2)

    tt <- topTags(qlf, n=Inf)$table %>%
      tibble::rownames_to_column("gene") %>%
      dplyr::as_tibble() %>%
      dplyr::transmute(
        gene,
        logFC,
        logCPM,
        F,
        PValue,
        FDR,
        celltype = celltype_name,
        mode = mode_label,
        n_samples_total = ncol(pb),
        n_TIL_low = sum(group_vec=="TIL_low"),
        n_TIL_high = sum(group_vec=="TIL_high")
      )

    meta_out <- sample_group_tbl %>%
      dplyr::rename(sample = .data[[SAMPLE_COL]]) %>%
      dplyr::mutate(celltype = celltype_name, mode = mode_label)

    list(status="ok", reason="", de=tt, meta=meta_out)

  }, error = function(e) {
    list(status="error", reason=paste0("edgeR failed: ", conditionMessage(e)), de=NULL, meta=NULL)
  })
}

run_panel <- function(celltypes_vec, genes_keep, mode_label) {
  de_list <- list()
  meta_list <- list()
  status_rows <- list()

  for (ct in celltypes_vec) {
    res <- run_edger_for_celltype(ct, genes_keep, mode_label)
    status_rows[[ct]] <- tibble(
      mode = mode_label,
      celltype = ct,
      status = res$status,
      reason = res$reason
    )
    if (identical(res$status, "ok")) {
      de_list[[ct]] <- res$de
      meta_list[[ct]] <- res$meta
    } else {
      if (!is.null(res$meta)) meta_list[[ct]] <- res$meta
    }
  }

  list(
    status = dplyr::bind_rows(status_rows),
    de = if (length(de_list)) dplyr::bind_rows(de_list) else tibble(),
    meta = if (length(meta_list)) dplyr::bind_rows(meta_list) else tibble()
  )
}

# Run
sender_res   <- run_panel(senders_oi,   feasible_ligands,   "sender_ligand")
receiver_res <- run_panel(receivers_oi, feasible_receptors, "receiver_receptor")

# CHANGED: Use zone_prefix for output file names
out_sender_de     <- file.path(FILTER_DIR, paste0(zone_prefix, "_sender_ligand_DE_edger_by_celltype_TILhigh_vs_TILlow_activityLigs_LTMrows.csv"))
out_receiver_de   <- file.path(FILTER_DIR, paste0(zone_prefix, "_receiver_receptor_DE_edger_by_celltype_TILhigh_vs_TILlow_LRreceptorsOfActivityLigs.csv"))
out_sender_meta   <- file.path(FILTER_DIR, paste0(zone_prefix, "_sender_pseudobulk_samples_by_celltype.csv"))
out_receiver_meta <- file.path(FILTER_DIR, paste0(zone_prefix, "_receiver_pseudobulk_samples_by_celltype.csv"))
out_status        <- file.path(FILTER_DIR, paste0(zone_prefix, "_LR_DE_status_by_celltype.csv"))

readr::write_csv(sender_res$de, out_sender_de)
readr::write_csv(receiver_res$de, out_receiver_de)
readr::write_csv(sender_res$meta, out_sender_meta)
readr::write_csv(receiver_res$meta, out_receiver_meta)
readr::write_csv(dplyr::bind_rows(sender_res$status, receiver_res$status), out_status)

cat("Wrote:\n")
cat(" ", out_sender_de, "\n")
cat(" ", out_receiver_de, "\n")
cat(" ", out_sender_meta, "\n")
cat(" ", out_receiver_meta, "\n")
cat(" ", out_status, "\n\n")

cat("Celltypes with OK DE (senders):", sum(sender_res$status$status=="ok"), "/", length(senders_oi), "\n")
cat("Celltypes with OK DE (receivers):", sum(receiver_res$status$status=="ok"), "/", length(receivers_oi), "\n\n")

if (any(sender_res$status$status=="error") || any(receiver_res$status$status=="error")) {
  cat("Errors detected in some celltypes:\n")
  print(dplyr::bind_rows(sender_res$status, receiver_res$status) %>% dplyr::filter(status=="error"))
  cat("\n")
}

if (nrow(sender_res$de) > 0) {
  cat("Top 10 ligand DE rows overall (by FDR):\n")
  print(sender_res$de %>% dplyr::arrange(FDR) %>% dplyr::slice_head(n=10))
  cat("\n")
}
if (nrow(receiver_res$de) > 0) {
  cat("Top 10 receptor DE rows overall (by FDR):\n")
  print(receiver_res$de %>% dplyr::arrange(FDR) %>% dplyr::slice_head(n=10))
  cat("\n")
}

### Export results to python

In [ ]:
%%R -o ligand_activities -o sender_de -o receiver_de
# Export R results to Python (rpy2)

stopifnot(exists("ligand_activities"))
stopifnot(exists("sender_res"), exists("receiver_res"))

# Pull DE tables out of the result bundles
sender_de   <- sender_res$de
receiver_de <- receiver_res$de

# Ensure base data.frame for rpy2 conversion stability
ligand_activities <- as.data.frame(ligand_activities)
sender_de   <- as.data.frame(sender_de)
receiver_de <- as.data.frame(receiver_de)

# Minimal sanity checks for downstream Cell 5 expectations
req_act <- c("test_ligand", "direction", "aupr_corrected", "rank")
req_de  <- c("gene", "celltype", "logFC", "FDR")

missing_act <- setdiff(req_act, colnames(ligand_activities))
missing_sde <- setdiff(req_de,  colnames(sender_de))
missing_rde <- setdiff(req_de,  colnames(receiver_de))

if (length(missing_act) > 0) stop(paste0("ligand_activities missing: ", paste(missing_act, collapse=", ")))
if (length(missing_sde) > 0) stop(paste0("sender_de missing: ", paste(missing_sde, collapse=", ")))
if (length(missing_rde) > 0) stop(paste0("receiver_de missing: ", paste(missing_rde, collapse=", ")))

cat("Export objects prepared:\n")
cat("  ligand_activities rows:", nrow(ligand_activities), "\n")
cat("  sender_de rows       :", nrow(sender_de), "\n")
cat("  receiver_de rows     :", nrow(receiver_de), "\n")

### Construct axis scores

In [ ]:
# %% [markdown]
# ### Construct axis scores

# %%
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path

# === CLEAR STALE VARIABLES ===
for _v in ["all_axes", "top_axes", "sender_candidates", "ligand_sender_gating", 
           "base", "axes", "trm_high", "trm_low", "ltm_df", "act_by_dir",
           "allowed_ligands_by_dir", "lr_use_by_dir", "receptors_by_dir"]:
    if _v in globals():
        del globals()[_v]

# === VERIFY INPUTS ===
required = ["target_receiver_map_high", "target_receiver_map_low", "SENDERS", "RECEIVERS",
            "ligand_activities", "sender_de", "receiver_de", "lr_network", 
            "ligand_target_matrix", "de_df", "OUT_DIR",
            "TOP_LIGANDS_PER_DIRECTION", "LTM_TOPK_PER_TARGET", "TOP_AXES_PER_TARGET",
            "TOP_LIGANDS_PER_TARGET", "TOP_AXES_PER_LIGAND_PER_TARGET", "SENDER_TOPK_PER_LIGAND"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing: {missing}")

print("INPUT CHECK:")
print(f"  target_receiver_map_high: {len(target_receiver_map_high)} rows, {target_receiver_map_high['target'].nunique()} targets")
print(f"  target_receiver_map_low: {len(target_receiver_map_low)} rows, {target_receiver_map_low['target'].nunique()} targets")

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"

MIN_PCT_EXPRESSED_LIGAND = 0.10
MIN_SAMPLES_EXPRESSING_LIGAND = 3

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = sorted({std_ct(x) for x in SENDERS})
receivers_std = sorted({std_ct(x) for x in RECEIVERS})

TOP_LIGANDS_PER_DIRECTION = int(TOP_LIGANDS_PER_DIRECTION)
LTM_TOPK_PER_TARGET = int(LTM_TOPK_PER_TARGET)
TOP_AXES_PER_TARGET = int(TOP_AXES_PER_TARGET)
TOP_LIGANDS_PER_TARGET = int(TOP_LIGANDS_PER_TARGET)
TOP_AXES_PER_LIGAND_PER_TARGET = int(TOP_AXES_PER_LIGAND_PER_TARGET)
SENDER_TOPK_PER_LIGAND = int(SENDER_TOPK_PER_LIGAND) if SENDER_TOPK_PER_LIGAND is not None else None

LIGANDS_FROM_LTM_PER_TARGET = 10
KEEP_LIGANDS = []

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nSELECTION CONTROLS:")
print(f"  TOP_LIGANDS_PER_DIRECTION: {TOP_LIGANDS_PER_DIRECTION}")
print(f"  LTM_TOPK_PER_TARGET: {LTM_TOPK_PER_TARGET}")
print(f"  TOP_AXES_PER_TARGET: {TOP_AXES_PER_TARGET}")
print(f"  TOP_LIGANDS_PER_TARGET: {TOP_LIGANDS_PER_TARGET}")
print(f"  TOP_AXES_PER_LIGAND_PER_TARGET: {TOP_AXES_PER_LIGAND_PER_TARGET}")
print(f"  SENDER_TOPK_PER_LIGAND: {SENDER_TOPK_PER_LIGAND}")

# === COPY INPUTS ===
ligand_activities = pd.DataFrame(ligand_activities).copy()
sender_de = pd.DataFrame(sender_de).copy()
receiver_de = pd.DataFrame(receiver_de).copy()

ligand_activities["test_ligand"] = ligand_activities["test_ligand"].astype(str)
ligand_activities["direction"] = ligand_activities["direction"].astype(str)
ligand_activities["rank"] = pd.to_numeric(ligand_activities["rank"], errors="coerce")
ligand_activities["aupr_corrected"] = pd.to_numeric(ligand_activities["aupr_corrected"], errors="coerce")

for df_ in (sender_de, receiver_de):
    df_["gene"] = df_["gene"].astype(str)
    df_["celltype"] = df_["celltype"].astype(str)
    df_["logFC"] = pd.to_numeric(df_["logFC"], errors="coerce")
    df_["FDR"] = pd.to_numeric(df_["FDR"], errors="coerce")

lr_df0 = pd.DataFrame(lr_network).copy()
cols = set(lr_df0.columns.astype(str))
if {"from", "to"}.issubset(cols):
    lr_df = lr_df0.rename(columns={"from": "ligand", "to": "receptor"})[["ligand", "receptor"]].copy()
elif {"ligand", "receptor"}.issubset(cols):
    lr_df = lr_df0[["ligand", "receptor"]].copy()
else:
    raise KeyError(f"lr_network columns not recognized: {list(lr_df0.columns)}")
lr_df["ligand"] = lr_df["ligand"].astype(str)
lr_df["receptor"] = lr_df["receptor"].astype(str)
lr_df = lr_df.drop_duplicates()

# Fresh copy of TRMs
trm_high = pd.DataFrame(target_receiver_map_high).copy()
trm_low = pd.DataFrame(target_receiver_map_low).copy()

for trm in (trm_high, trm_low):
    trm["target"] = trm["target"].astype(str)
    trm["receiver_celltype"] = trm["receiver_celltype"].astype(str)
    for col in ["target_pct_expr", "target_mean_expr", "receiver_rank_score", "panel_coverage_ratio"]:
        if col in trm.columns:
            trm[col] = pd.to_numeric(trm[col], errors="coerce")

ltm_df_full = pd.DataFrame(
    ligand_target_matrix.values,
    index=pd.Index(ligand_target_matrix.index.astype(str), name="ligand"),
    columns=pd.Index(ligand_target_matrix.columns.astype(str), name="target"),
)

def get_ltm_ligands_for_targets(targets, ltm_df, k):
    targets = [t for t in targets if t in ltm_df.columns]
    ligs = set()
    for t in targets:
        w = ltm_df[t]
        w = w[w > 0].nlargest(k)
        ligs.update(w.index.astype(str))
    return ligs

# === ACTIVITY MAPS ===
act_by_dir = {}
for direction in ["TIL_high", "TIL_low"]:
    act = ligand_activities[ligand_activities["direction"] == direction].dropna(subset=["rank"])
    act = act.sort_values("rank")
    act_by_dir[direction] = {
        "top_ligs": act.nsmallest(TOP_LIGANDS_PER_DIRECTION, "rank")["test_ligand"].tolist(),
        "activity_map": act.set_index("test_ligand")["aupr_corrected"].to_dict(),
        "rank_map": act.set_index("test_ligand")["rank"].to_dict(),
    }

# === LIGAND UNIVERSE ===
allowed_ligands_by_dir = {}
for direction, trm in [("TIL_high", trm_high), ("TIL_low", trm_low)]:
    top_act = set(act_by_dir[direction]["top_ligs"])
    dir_targets = sorted(set(trm["target"]) & set(ltm_df_full.columns))
    top_ltm = get_ltm_ligands_for_targets(dir_targets, ltm_df_full, LIGANDS_FROM_LTM_PER_TARGET)
    allowed_ligands_by_dir[direction] = top_act | top_ltm | set(KEEP_LIGANDS)
    print(f"{direction} ligand universe: {len(allowed_ligands_by_dir[direction])}")

need_ligs = sorted(allowed_ligands_by_dir["TIL_high"] | allowed_ligands_by_dir["TIL_low"])
need_tgts = sorted(set(trm_high["target"]) | set(trm_low["target"]))
use_ligs = sorted(set(ltm_df_full.index) & set(need_ligs))
use_tgts = sorted(set(ltm_df_full.columns) & set(need_tgts))
ltm_df = ltm_df_full.loc[use_ligs, use_tgts].copy()

print(f"LTM subset: {len(use_ligs)} ligands x {len(use_tgts)} targets")

# === SENDER GATING ===
ct_std = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)
sample_ids = adata_sc.obs[SAMPLE_KEY].astype(str)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)
ligs_in_var = sorted(set(use_ligs) & var_set)

sender_rows = []
for direction in ["TIL_high", "TIL_low"]:
    dir_ligs = sorted(allowed_ligands_by_dir[direction] & set(ligs_in_var))
    if not dir_ligs:
        continue
    lig_idx = var_names.get_indexer(dir_ligs)

    for sct in senders_std:
        m = (grp == direction).to_numpy() & (ct_std == sct).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue

        sample_sub = sample_ids.iloc[idx].values
        n_samples = len(np.unique(sample_sub))
        Xsub = X[idx, :][:, lig_idx]
        pct = np.asarray(Xsub.getnnz(axis=0)).ravel() / idx.size
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        Xsub_dense = Xsub.toarray()

        for i, lig in enumerate(dir_ligs):
            n_samp_expr = len(set(sample_sub[Xsub_dense[:, i] > 0]))
            if pct[i] >= MIN_PCT_EXPRESSED_LIGAND and n_samp_expr >= MIN_SAMPLES_EXPRESSING_LIGAND:
                sender_rows.append({
                    "direction": direction, "ligand": lig, "sender_celltype": sct,
                    "pct_expr": float(pct[i]), "mean_expr": float(mean_expr[i]),
                    "n_samples_expressing": n_samp_expr,
                    "sample_frac": n_samp_expr / n_samples if n_samples > 0 else 0,
                })

sender_gating = pd.DataFrame(sender_rows)

# Sender specificity and rank score
for direction in ["TIL_high", "TIL_low"]:
    mask = sender_gating["direction"] == direction
    if mask.sum() == 0:
        continue
    lig_mean = sender_gating.loc[mask].groupby("ligand")["mean_expr"].transform("mean").replace(0, np.nan)
    sender_gating.loc[mask, "specificity"] = (sender_gating.loc[mask, "mean_expr"] / lig_mean).fillna(0)

sender_gating["_mr"] = sender_gating.groupby(["direction", "ligand"])["mean_expr"].rank(ascending=False)
sender_gating["_pr"] = sender_gating.groupby(["direction", "ligand"])["pct_expr"].rank(ascending=False)
sender_gating["_sfr"] = sender_gating.groupby(["direction", "ligand"])["sample_frac"].rank(ascending=False)
sender_gating["_spr"] = sender_gating.groupby(["direction", "ligand"])["specificity"].rank(ascending=False)
sender_gating["sender_rank_score"] = 0.35/sender_gating["_mr"] + 0.25/sender_gating["_pr"] + 0.20/sender_gating["_sfr"] + 0.20/sender_gating["_spr"]
sender_gating = sender_gating.drop(columns=["_mr", "_pr", "_sfr", "_spr"])

# Merge sender DE
sender_de_use = sender_de.rename(columns={"gene": "ligand", "celltype": "sender_celltype", "logFC": "ligand_logFC", "FDR": "ligand_FDR"})
sender_candidates = sender_gating.merge(sender_de_use[["ligand", "sender_celltype", "ligand_logFC", "ligand_FDR"]], on=["ligand", "sender_celltype"], how="left")

# Concordance bonus
exp_sign = np.where(sender_candidates["direction"] == "TIL_high", 1, -1)
conc = sender_candidates["ligand_logFC"].notna() & ((exp_sign * sender_candidates["ligand_logFC"]) > 0)
sender_candidates.loc[conc, "sender_rank_score"] += 0.05

sender_candidates = sender_candidates.sort_values(["direction", "ligand", "sender_rank_score"], ascending=[True, True, False])
if SENDER_TOPK_PER_LIGAND is not None:
    sender_candidates = sender_candidates.groupby(["direction", "ligand"], as_index=False).head(SENDER_TOPK_PER_LIGAND)

print(f"Sender candidates: {len(sender_candidates)} rows")

# === LR BY DIRECTION ===
lr_by_dir = {}
for d in ["TIL_high", "TIL_low"]:
    lr_by_dir[d] = lr_df[lr_df["ligand"].isin(allowed_ligands_by_dir[d])].copy()
    lr_by_dir[d]["direction"] = d

# === LTM TOP-K ===
def ltm_topk(direction, trm):
    tgts = sorted(set(trm["target"]) & set(ltm_df.columns))
    elig_ligs = sorted(set(lr_by_dir[direction]["ligand"]) & set(ltm_df.index))
    if not tgts or not elig_ligs:
        return pd.DataFrame(columns=["direction", "target", "ligand", "ltm_weight"])
    sub = ltm_df.loc[elig_ligs, tgts]
    rows = []
    for t in tgts:
        w = sub[t]
        w = w[w > 0].nlargest(LTM_TOPK_PER_TARGET)
        for lig, wt in w.items():
            rows.append({"direction": direction, "target": t, "ligand": str(lig), "ltm_weight": float(wt)})
    return pd.DataFrame(rows)

topk_high = ltm_topk("TIL_high", trm_high)
topk_low = ltm_topk("TIL_low", trm_low)

# === MERGE TRM ===
trm_cols = ["target", "receiver_celltype", "target_pct_expr", "target_mean_expr", "receiver_rank_score",
            "global_best_celltype", "panel_coverage_ratio", "coverage_flag"]
trm_cols_high = [c for c in trm_cols if c in trm_high.columns]
trm_cols_low = [c for c in trm_cols if c in trm_low.columns]

base_high = topk_high.merge(trm_high[trm_cols_high], on="target", how="inner")
base_low = topk_low.merge(trm_low[trm_cols_low], on="target", how="inner")
base = pd.concat([base_high, base_low], ignore_index=True)

print(f"Base: {len(base)} rows (high={len(base_high)}, low={len(base_low)})")

# Attach activity
base["ligand_activity"] = np.nan
base["ligand_activity_rank"] = np.nan
for d in ["TIL_high", "TIL_low"]:
    m = base["direction"] == d
    base.loc[m, "ligand_activity"] = base.loc[m, "ligand"].map(act_by_dir[d]["activity_map"])
    base.loc[m, "ligand_activity_rank"] = base.loc[m, "ligand"].map(act_by_dir[d]["rank_map"])

# Attach sender
base = base.merge(sender_candidates, on=["direction", "ligand"], how="inner")
print(f"After sender merge: {len(base)} rows")

# Attach receptor
lr_dir = pd.concat([lr_by_dir["TIL_high"], lr_by_dir["TIL_low"]], ignore_index=True)
axes = base.merge(lr_dir, on=["direction", "ligand"], how="inner")

receiver_tbl = receiver_de.rename(columns={"celltype": "receiver_celltype", "gene": "receptor", "logFC": "receptor_logFC", "FDR": "receptor_FDR"})
axes = axes.merge(receiver_tbl[["receiver_celltype", "receptor", "receptor_logFC", "receptor_FDR"]], on=["receiver_celltype", "receptor"], how="left")

# Attach target DE
de_use = de_df[["gene", "logFC", "FDR"]].copy()
de_use.columns = ["target", "target_logFC", "target_FDR"]
axes = axes.merge(de_use, on="target", how="left")

axes = axes.dropna(subset=["receptor_logFC"])
print(f"After receptor filter: {len(axes)} axes")

# === SCORE ===
def score_axes(df):
    df = df.copy()
    for d in ["TIL_high", "TIL_low"]:
        m = df["direction"] == d
        if m.sum() == 0:
            continue
        ranks = df.loc[m, "ligand_activity_rank"]
        max_r = ranks.max()
        df.loc[m, "activity_pctl"] = 1 - (ranks - 1) / max_r if max_r > 1 else 1.0
        df.loc[m, "ltm_pctl"] = df.loc[m, "ltm_weight"].rank(pct=True)
        if "receiver_rank_score" in df.columns:
            df.loc[m, "receiver_rank_pctl"] = df.loc[m, "receiver_rank_score"].rank(pct=True)
        else:
            df.loc[m, "receiver_rank_pctl"] = 0.5

    exp_sign = np.where(df["direction"] == "TIL_high", 1, -1)
    df["ligand_concordant"] = (exp_sign * df["ligand_logFC"]) > 0
    df["receptor_concordant"] = (exp_sign * df["receptor_logFC"]) > 0

    df["axis_score"] = (
        0.35 * df["activity_pctl"].fillna(0) +
        0.25 * df["ltm_pctl"].fillna(0) +
        0.15 * df["receiver_rank_pctl"].fillna(0) +
        0.10 * df["ligand_concordant"].astype(float) +
        0.15 * df["receptor_concordant"].astype(float)
    )
    return df

all_axes = score_axes(axes)
all_axes = all_axes.sort_values(["direction", "target", "axis_score"], ascending=[True, True, False])
all_axes["axis_rank"] = all_axes.groupby(["direction", "target"]).cumcount() + 1
all_axes = all_axes.drop_duplicates(subset=["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"])

# === TOP AXES SELECTION ===
lig_best = all_axes.groupby(["direction", "target", "ligand"]).agg(
    best_score=("axis_score", "max"), best_ltm=("ltm_weight", "max"), best_act=("ligand_activity", "max"), n=("axis_score", "size")
).reset_index()
lig_best = lig_best.sort_values(["direction", "target", "best_score", "best_ltm", "best_act"], ascending=[True, True, False, False, False])
lig_best["lig_rank"] = lig_best.groupby(["direction", "target"]).cumcount() + 1
keep_ligs = lig_best[lig_best["lig_rank"] <= TOP_LIGANDS_PER_TARGET][["direction", "target", "ligand"]]

sel = all_axes.merge(keep_ligs, on=["direction", "target", "ligand"], how="inner")
sel = sel.sort_values(["direction", "target", "ligand", "axis_score"], ascending=[True, True, True, False])
sel["ax_in_lig"] = sel.groupby(["direction", "target", "ligand"]).cumcount() + 1
sel = sel[sel["ax_in_lig"] <= TOP_AXES_PER_LIGAND_PER_TARGET]
sel = sel.sort_values(["direction", "target", "axis_score"], ascending=[True, True, False])
sel["ax_final"] = sel.groupby(["direction", "target"]).cumcount() + 1
top_axes = sel[sel["ax_final"] <= TOP_AXES_PER_TARGET].copy()
top_axes["axis_rank"] = top_axes.groupby(["direction", "target"]).cumcount() + 1

# === SUMMARY ===
print(f"\nAXES BUILT:")
print(f"  all_axes: {len(all_axes)} rows, {all_axes['target'].nunique()} targets")
print(f"  top_axes: {len(top_axes)} rows, {top_axes['target'].nunique()} targets")

for d in ["TIL_high", "TIL_low"]:
    sub = all_axes[all_axes["direction"] == d]
    print(f"\n  {d}: {len(sub)} axes, {sub['target'].nunique()} targets, {sub['ligand'].nunique()} ligands")

# === SAVE ===
all_axes.to_csv(OUT_DIR / "all_axes.csv", index=False)
top_axes.to_csv(OUT_DIR / "top_axes.csv", index=False)
print(f"\n✓ Saved to {OUT_DIR}")

### Summarize axis scores for plot selection

In [ ]:
# %% [markdown]
# ### Summarize axis scores for plot selection

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib

# === CLEAR STALE VARIABLES ===
for _v in ["df", "df_all", "df_top", "axes_catalog", "axes_shortlist", "short", "short2"]:
    if _v in globals():
        del globals()[_v]

# === VERIFY INPUTS ===
if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR not found.")
if "top_axes" not in globals():
    raise NameError("top_axes not found. Run Cell 5 first.")

OUT_DIR = Path(OUT_DIR)

print("INPUT CHECK:")
print(f"  top_axes: {len(top_axes)} rows, {top_axes['target'].nunique()} targets")
if "all_axes" in globals():
    print(f"  all_axes: {len(all_axes)} rows (for reference, not used in catalog)")

# Use top_axes (filtered) for catalog, NOT all_axes
df = pd.DataFrame(top_axes).copy()

req_cols = {"direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
            "target_logFC", "target_FDR", "ligand_activity", "ltm_weight",
            "ligand_concordant", "receptor_concordant", "axis_score", "axis_rank"}
missing = req_cols - set(df.columns)
if missing:
    raise KeyError(f"top_axes missing columns: {sorted(missing)}")

# Normalize dtypes
for c in ["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"]:
    df[c] = df[c].astype(str)
for c in ["axis_score", "axis_rank", "target_logFC", "target_FDR", "ligand_activity", "ltm_weight"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["ligand_concordant"] = df["ligand_concordant"].astype(bool)
df["receptor_concordant"] = df["receptor_concordant"].astype(bool)

if "receiver_rank_score" in df.columns:
    df["receiver_rank_score"] = pd.to_numeric(df["receiver_rank_score"], errors="coerce")
if "panel_coverage_ratio" in df.columns:
    df["panel_coverage_ratio"] = pd.to_numeric(df["panel_coverage_ratio"], errors="coerce")

# === FILTER 1: positive activity ===
n_before = len(df)
df = df[df["ligand_activity"] > 0].copy()
print(f"\nFilter: ligand_activity > 0: {n_before} -> {len(df)} axes")

# === ABUNDANCE ANNOTATIONS ===
def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

abundance_available = False
if "EFS_ROOT" in globals() and "ZONE" in globals():
    ab_path = Path(EFS_ROOT) / "results/visium/til_zones/celltype_zone_specificity_curated/celltype_zone_specificity_level4_curated.csv"

    if ab_path.exists():
        ab = pd.read_csv(ab_path)
        ab["group"] = ab["group"].astype(str)
        ab["zone"] = ab["zone"].astype(str)
        ab["celltype_std"] = ab["celltype_level4_curated"].astype(str).map(std_ct)
        ab_zone = ab[ab["zone"] == str(ZONE)].copy()

        delta_tbl = ab_zone.pivot_table(index="celltype_std", columns="group", values="mean_zone", aggfunc="first").reset_index()
        if "TIL_high" in delta_tbl.columns and "TIL_low" in delta_tbl.columns:
            delta_tbl["mean_zone_high_minus_low"] = pd.to_numeric(delta_tbl["TIL_high"], errors="coerce") - pd.to_numeric(delta_tbl["TIL_low"], errors="coerce")
            delta_tbl = delta_tbl[["celltype_std", "mean_zone_high_minus_low"]]
        else:
            delta_tbl = pd.DataFrame({"celltype_std": ab_zone["celltype_std"].unique(), "mean_zone_high_minus_low": np.nan})

        df["sender_celltype_std"] = df["sender_celltype"].map(std_ct)
        df["receiver_celltype_std"] = df["receiver_celltype"].map(std_ct)

        # Sender abundance
        sender_ab = ab_zone.rename(columns={"celltype_std": "sender_celltype_std", "mean_zone": "sender_mean_zone"})
        sender_ab = sender_ab[["group", "sender_celltype_std", "sender_mean_zone"]].copy()
        df = df.merge(sender_ab, left_on=["direction", "sender_celltype_std"], right_on=["group", "sender_celltype_std"], how="left")
        df = df.drop(columns=["group"], errors="ignore")

        # Receiver abundance
        receiver_ab = ab_zone.rename(columns={"celltype_std": "receiver_celltype_std", "mean_zone": "receiver_mean_zone"})
        receiver_ab = receiver_ab[["group", "receiver_celltype_std", "receiver_mean_zone"]].copy()
        df = df.merge(receiver_ab, left_on=["direction", "receiver_celltype_std"], right_on=["group", "receiver_celltype_std"], how="left")
        df = df.drop(columns=["group"], errors="ignore")

        # Delta
        delta_sender = delta_tbl.rename(columns={"celltype_std": "sender_celltype_std", "mean_zone_high_minus_low": "sender_mean_zone_high_minus_low"})
        delta_receiver = delta_tbl.rename(columns={"celltype_std": "receiver_celltype_std", "mean_zone_high_minus_low": "receiver_mean_zone_high_minus_low"})
        df = df.merge(delta_sender, on="sender_celltype_std", how="left")
        df = df.merge(delta_receiver, on="receiver_celltype_std", how="left")

        print(f"Abundance annotations loaded from: {ab_path}")
        abundance_available = True
    else:
        print(f"Abundance CSV not found: {ab_path}")
else:
    print("EFS_ROOT or ZONE not set, skipping abundance annotations")

# === FILTER 2: concordance/abundance ===
ABUNDANCE_THRESHOLD = 0.05
n_before = len(df)

if abundance_available:
    high_mask = df["direction"] == "TIL_high"
    low_mask = df["direction"] == "TIL_low"
    high_keep = df["ligand_concordant"] | (df["sender_mean_zone_high_minus_low"] > ABUNDANCE_THRESHOLD)
    low_keep = df["ligand_concordant"] | (df["sender_mean_zone_high_minus_low"] < -ABUNDANCE_THRESHOLD)
    df = df[(high_mask & high_keep) | (low_mask & low_keep)].copy()
else:
    df = df[df["ligand_concordant"]].copy()

print(f"Filter: concordance/abundance: {n_before} -> {len(df)} axes")

# === AXIS ID ===
key_cols = ["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"]
df = df.drop_duplicates(subset=key_cols).copy()

def axis_hash_id(row):
    s = "|".join(str(row[c]) for c in key_cols)
    h = hashlib.md5(s.encode()).hexdigest()[:6].upper()
    return ("H" if row["direction"] == "TIL_high" else "L") + h

df["axis_id"] = df.apply(axis_hash_id, axis=1)

# === RANK SCORE ===
df["de_strength"] = -np.log10(df["target_FDR"].clip(lower=1e-300))
df["rank_score"] = df["axis_score"] * df["de_strength"] * df["target_logFC"].abs()

# === VERIFY AXES PER TARGET ===
axes_per_target = df.groupby(["direction", "target"]).size()
print(f"\nAxes per target: min={axes_per_target.min()}, max={axes_per_target.max()}, mean={axes_per_target.mean():.1f}")

# === CATALOG COLUMNS ===
catalog_cols = ["axis_id", "direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
                "axis_score", "axis_rank", "rank_score", "target_logFC", "target_FDR",
                "ligand_activity", "ltm_weight", "ligand_concordant", "receptor_concordant"]

# Add optional columns if present
for c in ["receiver_rank_score", "global_best_celltype", "global_best_n_cells", "panel_coverage_ratio", "coverage_flag",
          "sender_mean_zone", "receiver_mean_zone", "sender_mean_zone_high_minus_low", "receiver_mean_zone_high_minus_low"]:
    if c in df.columns:
        catalog_cols.append(c)

axes_catalog = df[[c for c in catalog_cols if c in df.columns]].copy()
axes_catalog.to_csv(OUT_DIR / "axes_catalog.csv", index=False)

# === SHORTLIST (further filtered subset) ===
AXIS_SCORE_MIN = 0.5
MAX_AXIS_RANK_PER_TARGET = 3
TARGET_FDR_MAX = 0.05
TARGET_LOGFC_MIN = 0.25
AXES_PER_TARGET_DE_BAL = 2

short = df[(df["axis_score"] > AXIS_SCORE_MIN) & (df["axis_rank"] <= MAX_AXIS_RANK_PER_TARGET)].copy()
short = short.dropna(subset=["target_FDR", "target_logFC", "rank_score"])
short = short[short["target_FDR"] <= TARGET_FDR_MAX]

high = short[(short["direction"] == "TIL_high") & (short["target_logFC"] >= TARGET_LOGFC_MIN)]
low = short[(short["direction"] == "TIL_low") & (short["target_logFC"] <= -TARGET_LOGFC_MIN)]
short = pd.concat([high, low])
short = short.sort_values(["direction", "target", "rank_score"], ascending=[True, True, False])
axes_shortlist = short.groupby(["direction", "target"], as_index=False).head(AXES_PER_TARGET_DE_BAL)
axes_shortlist = axes_shortlist[[c for c in catalog_cols if c in axes_shortlist.columns]]
axes_shortlist.to_csv(OUT_DIR / "axes_shortlist.csv", index=False)

# === SUMMARY ===
print(f"\nCATALOG SUMMARY:")
for d in ["TIL_high", "TIL_low"]:
    sub = df[df["direction"] == d]
    print(f"  {d}: {len(sub)} axes, {sub['target'].nunique()} targets, {sub['ligand'].nunique()} ligands")

if "coverage_flag" in df.columns:
    print(f"\nPANEL COVERAGE:")
    for d in ["TIL_high", "TIL_low"]:
        sub = df[df["direction"] == d]
        if sub.empty:
            continue
        n_ok = (sub["coverage_flag"] == "OK").sum()
        n_low = (sub["coverage_flag"] == "LOW_COVERAGE").sum()
        print(f"  {d}: OK={n_ok}, LOW_COVERAGE={n_low}")

print(f"\nSHORTLIST: {len(axes_shortlist)} axes, {axes_shortlist['target'].nunique()} targets")

print(f"\nWROTE:")
print(f"  {OUT_DIR / 'axes_catalog.csv'}")
print(f"  {OUT_DIR / 'axes_shortlist.csv'}")

# === DISPLAY ===
display_cols = ["axis_id", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
                "axis_score", "rank_score", "target_logFC"]
if "coverage_flag" in axes_catalog.columns:
    display_cols.append("coverage_flag")
display_cols = [c for c in display_cols if c in axes_catalog.columns]

print(f"\nTOP 15 TIL_high:")
display(axes_catalog[axes_catalog["direction"] == "TIL_high"].nlargest(15, "rank_score")[display_cols])

print(f"\nTOP 15 TIL_low:")
display(axes_catalog[axes_catalog["direction"] == "TIL_low"].nlargest(15, "rank_score")[display_cols])

# === WARNINGS ===
if "coverage_flag" in axes_shortlist.columns:
    low_cov = axes_shortlist[axes_shortlist["coverage_flag"] == "LOW_COVERAGE"]
    if len(low_cov) > 0:
        print(f"\n⚠️ WARNING: {len(low_cov)} shortlist axes have LOW_COVERAGE:")
        for _, r in low_cov.iterrows():
            gb_ct = r.get('global_best_celltype', 'N/A')
            gb_n = r.get('global_best_n_cells', 'N/A')
            print(f"  {r['axis_id']}: {r['target']} (global_best={gb_ct}, n={gb_n})")

print("\n✓ Done")

### Axes for TIL+ Circos plot

In [ ]:
# Cell 7: Enter selected axis IDs (uses current run's catalog on disk)

import pandas as pd
from pathlib import Path

if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR not found.")

OUT_DIR = Path(OUT_DIR)
catalog_path = OUT_DIR / "axes_catalog.csv"
if not catalog_path.exists():
    raise FileNotFoundError(f"Missing catalog: {catalog_path}. Run Cell 6 first.")

axes_catalog = pd.read_csv(catalog_path)

if "axis_id" not in axes_catalog.columns:
    raise KeyError(f"axes_catalog.csv missing axis_id. Columns: {list(axes_catalog.columns)}")

# Paste your chosen IDs here (from this run's axes_catalog.csv)
SELECT_TIL_HIGH = [
    "H266533","HB550D3","H3198DC","H238F85","HF15A68","H75620B","H954112","H3252F3","HD9A50C",
    "H3F1F25","HD01664","H5281DC","HAE1D7E","H22952A","H55378B","HB0DA7B","H1E8A4B","H8D3623",
    "H03C0E5","H9F9667","H8A7D5A"
]

SELECT_TIL_LOW = [
    "LEA62B3","L3C9EF8","L2BC69A","LAA8A3D","L8796B1","L97207D","LA2CA39","L4D8958","LC8A51E",
    "LEEBD09","LA508E3","L1E50D3","LC88442","LE766E7","L6F3530","L0B8EF2","L7D7161",
    "L38695B","LB3FE3E","L3FCB33","L3D9965","L5E369B","L106858","L8F5A7D","L0AC580","L92F723",
    "LC5C55E","L5F1DFE","L15DD73","L772FFC","L0CFB5F","L517C1F","L257994"
]

# Normalize and dedupe while preserving input order
raw_ids = [str(x).strip() for x in (SELECT_TIL_HIGH + SELECT_TIL_LOW) if str(x).strip()]
if len(raw_ids) == 0:
    raise ValueError("No axis IDs provided. Fill SELECT_TIL_HIGH and/or SELECT_TIL_LOW.")

seen = set()
selected_ids = []
for x in raw_ids:
    if x not in seen:
        selected_ids.append(x)
        seen.add(x)

catalog_ids = axes_catalog["axis_id"].astype(str)
missing = sorted(set(selected_ids) - set(catalog_ids))
if missing:
    raise ValueError(
        "Some axis IDs were not found in the current run's catalog.\n"
        "This usually means you copied IDs from a different run.\n\n"
        "Missing:\n" + "\n".join(missing)
    )

sel = axes_catalog[catalog_ids.isin(selected_ids)].copy()

# Preserve the order you typed them
order = {aid: i for i, aid in enumerate(selected_ids)}
sel["input_order"] = sel["axis_id"].astype(str).map(order)
sel = sel.sort_values(["input_order"]).drop(columns=["input_order"])

# Sanity: ensure selected HIGH/LOW match your lists
if "direction" in sel.columns:
    bad_high = sel[(sel["axis_id"].isin(SELECT_TIL_HIGH)) & (sel["direction"] != "TIL_high")]
    bad_low  = sel[(sel["axis_id"].isin(SELECT_TIL_LOW))  & (sel["direction"] != "TIL_low")]
    if len(bad_high) > 0 or len(bad_low) > 0:
        msg = []
        if len(bad_high) > 0:
            msg.append("Some IDs in SELECT_TIL_HIGH are not TIL_high:\n" + "\n".join(bad_high["axis_id"].astype(str).tolist()))
        if len(bad_low) > 0:
            msg.append("Some IDs in SELECT_TIL_LOW are not TIL_low:\n" + "\n".join(bad_low["axis_id"].astype(str).tolist()))
        raise ValueError("\n\n".join(msg))

out_path = OUT_DIR / "axes_selected.csv"
sel.to_csv(out_path, index=False)

print("Selected axes:", len(sel))
if "direction" in sel.columns:
    print("  HIGH:", int((sel["direction"] == "TIL_high").sum()))
    print("  LOW :", int((sel["direction"] == "TIL_low").sum()))
print("Wrote:", str(out_path))

show_cols = [c for c in [
    "axis_id","direction","target","ligand","receptor",
    "sender_celltype","receiver_celltype",
    "axis_score","rank_score","target_logFC","target_FDR",
    "ligand_activity","ltm_weight",
    "ligand_concordant","receptor_concordant",
] if c in sel.columns]

display(sel[show_cols])

# Border Zone 1 (BZ1) NicheNet

### Define BZ1 senders and receivers 

In [ ]:
def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

TILPLUS_GROUP = "TIL_high"

SENDERS = [
    "Malignant_AC-like_IFN_response",
    "Malignant_MES-hypoxia",
    "Malignant_AC2",
    "Astrocyte_IFN_response",
    "Macrophage_IFN_response",
    "Macrophage_C1Q_pos",
    "Macrophage_PVM",
    "Microglia_IFN_response"
]

RECEIVERS = [
    "Endothelial_tip", "Endothelial_arterial", "Pericyte",
    "Perivascular_fibroblast_type-I",
    "Perivascular_fibroblast_type-II",
    "Perivascular_fibroblast_type-III",
    "Pericyte_fibro-like", "VSMC",
    "Macrophage_IFN_response", "Microglia_IFN_response", "Macrophage_C1Q_pos","Macrophage_PVM"
]

SENDERS_STD = [std_ct(x) for x in SENDERS]
RECEIVERS_STD = [std_ct(x) for x in RECEIVERS]

tilplus_panel = {
    "group": TILPLUS_GROUP,
    "senders": SENDERS_STD,
    "receivers": RECEIVERS_STD,
}

print("BZ1 panel defined:")
print("  group:", tilplus_panel["group"])
print("  senders:", len(tilplus_panel["senders"]))
print("  receivers:", len(tilplus_panel["receivers"]))

### GSEA of HIGH and LOW BZ1 programs

In [ ]:
# %% [markdown]
# ### ULM pathway enrichment scoring

# %%
import numpy as np
import pandas as pd
from pathlib import Path
import requests
import decoupler as dc

# User config
ZONE = "BZ1"  # Options: "TIL+", "BZ1", "BZ2"

# Paths
cache_dir = Path.home() / ".cache" / "msigdb"
cache_dir.mkdir(parents=True, exist_ok=True)

EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
DE_DIR = EFS_ROOT / "results/visium/til_zones/deg_pseudobulk_edger"
OUT_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_filtering"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Zone-specific DE file
visium_de_file = DE_DIR / f"DE_edger_zone_{ZONE}_TIL_high_vs_TIL_low.csv"

print(f"ZONE: {ZONE}")
print(f"DE file: {visium_de_file.name}")

if not visium_de_file.exists():
    raise FileNotFoundError(f"Missing: {visium_de_file}")

print("Input file found")

# MSigDB GMT download and parse
urls = {
    "hallmark": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.symbols.gmt",
    "kegg":     "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.kegg_medicus.v2023.2.Hs.symbols.gmt",
    "biocarta": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.biocarta.v2023.2.Hs.symbols.gmt",
}
paths = {k: cache_dir / Path(v).name for k, v in urls.items()}

def download_text(url: str, out_path: Path, timeout: int = 120) -> None:
    if out_path.exists() and out_path.stat().st_size > 0:
        return
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    out_path.write_text(r.text)

def parse_gmt(gmt_path: Path) -> dict:
    gene_sets = {}
    with gmt_path.open("r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                gene_sets[parts[0]] = [g for g in parts[2:] if g]
    return gene_sets

for k, url in urls.items():
    download_text(url, paths[k])

hallmark_sets = parse_gmt(paths["hallmark"])
kegg_sets = parse_gmt(paths["kegg"])
biocarta_sets = parse_gmt(paths["biocarta"])

def gmt_to_net(gene_sets: dict) -> pd.DataFrame:
    rows = [(gs, g, 1.0) for gs, genes in gene_sets.items() for g in genes]
    return pd.DataFrame(rows, columns=["source", "target", "weight"])

net = pd.concat(
    [gmt_to_net(hallmark_sets), gmt_to_net(kegg_sets), gmt_to_net(biocarta_sets)],
    ignore_index=True
)

# Load full DE results (all genes)
visium_de = pd.read_csv(visium_de_file)
for c in ["gene", "logFC"]:
    if c not in visium_de.columns:
        raise ValueError(f"Missing column '{c}' in {visium_de_file.name}")

visium_de["gene"] = visium_de["gene"].astype(str)
visium_de["logFC"] = pd.to_numeric(visium_de["logFC"], errors="coerce")
visium_de = visium_de.dropna(subset=["logFC"])
visium_de = visium_de.drop_duplicates(subset=["gene"], keep="first")

print(f"Genes in DE results: {len(visium_de)}")

# BH FDR correction
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    p0 = p[ok]
    n = p0.size
    order = np.argsort(p0)
    ranked = p0[order]
    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty_like(p0)
    out[order] = np.clip(adj, 0, 1)
    q[ok] = out
    return q

# Run ULM on all genes
# Positive logFC = upregulated in TIL_high
# Negative logFC = upregulated in TIL_low
def run_ulm_all_genes(de_df: pd.DataFrame, label: str) -> pd.DataFrame:
    w = de_df.set_index("gene")["logFC"].astype(float)
    mat = w.to_frame().T
    mat.index = [label]

    est, pvs = dc.run_ulm(
        mat=mat,
        net=net,
        source="source",
        target="target",
        weight="weight",
        min_n=5,
        verbose=False,
    )

    res = pd.DataFrame({
        "gene_set": est.columns.astype(str),
        "score": est.iloc[0].to_numpy(dtype=float),
        "pval": pvs.iloc[0].to_numpy(dtype=float),
    })
    res["fdr"] = bh_fdr(res["pval"].to_numpy())
    res["collection"] = np.select(
        [
            res["gene_set"].str.startswith("HALLMARK_"),
            res["gene_set"].str.startswith("KEGG_"),
            res["gene_set"].str.startswith("BIOCARTA_"),
        ],
        ["Hallmark", "KEGG", "BioCarta"],
        default="Other",
    )
    return res

ulm_results = run_ulm_all_genes(visium_de, label=f"{ZONE}_TIL_high_vs_low")

# Positive score = enriched in TIL_high direction
# Negative score = enriched in TIL_low direction
ulm_high = ulm_results[ulm_results["score"] > 0].sort_values("score", ascending=False).copy()
ulm_low = ulm_results[ulm_results["score"] < 0].sort_values("score", ascending=True).copy()

# Save outputs
zone_file_prefix = ZONE.replace("+", "plus").replace("-", "minus")
out_all = OUT_DIR / f"{zone_file_prefix}_ULM_all_pathways.csv"
out_high = OUT_DIR / f"{zone_file_prefix}_ULM_TIL_high_enriched.csv"
out_low = OUT_DIR / f"{zone_file_prefix}_ULM_TIL_low_enriched.csv"

ulm_results.sort_values("score", ascending=False).to_csv(out_all, index=False)
ulm_high.to_csv(out_high, index=False)
ulm_low.to_csv(out_low, index=False)

print(f"\nGene sets: Hallmark={len(hallmark_sets)}, KEGG={len(kegg_sets)}, BioCarta={len(biocarta_sets)}")
print(f"Network: {net['source'].nunique()} sets, {net['target'].nunique()} genes")
print(f"\nWrote:")
print(f"  {out_all}")
print(f"  {out_high}")
print(f"  {out_low}")

# Display top results
def show_top(res: pd.DataFrame, title: str, n: int = 20) -> None:
    if res.empty:
        print(f"\n{title}: no results")
        return
    top = res.head(n).copy()
    top["score"] = top["score"].round(3)
    top["pval"] = top["pval"].map(lambda x: f"{x:.2e}" if np.isfinite(x) else "nan")
    top["fdr"] = top["fdr"].map(lambda x: f"{x:.2e}" if np.isfinite(x) else "nan")
    print(f"\n{title}")
    print(top[["gene_set", "collection", "score", "pval", "fdr"]].to_string(index=False))

show_top(ulm_high, f"Top pathways enriched in TIL_high ({ZONE})")
show_top(ulm_low, f"Top pathways enriched in TIL_low ({ZONE})")

### Define pathway filters

In [ ]:
# %% [markdown]
# ### Build pathway gene filter

# %%
# Select pathways from any collection (Hallmark, KEGG, BioCarta) plus custom gene lists
# Requires hallmark_sets, kegg_sets, biocarta_sets from ULM cell

# Selected pathways (can include HALLMARK_, KEGG_, or BIOCARTA_ prefixes)
selected_pathways = [
    # Hallmark
    "HALLMARK_ANGIOGENESIS",
    "HALLMARK_COMPLEMENT",
    "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
    "HALLMARK_HYPOXIA",
    "HALLMARK_APOPTOSIS",
    "HALLMARK_IL6_JAK_STAT3_SIGNALING",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE",
    "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "HALLMARK_TGF_BETA_SIGNALING",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "HALLMARK_IL2_STAT5_SIGNALING",
    "HALLMARK_APICAL_JUNCTION",
    "KEGG_MEDICUS_REFERENCE_CYTOKINE_JAK_STAT_SIGNALING_PATHWAY",
    "BIOCARTA_TCYTOTOXIC_PATHWAY",
    "KEGG_MEDICUS_REFERENCE_WNT_SIGNALING_MODULATION_WNT_ACYLATION",
    "BIOCARTA_PDGF_PATHWAY"
]

# Custom gene lists (not from MSigDB)
ecm_genes_brain = [
    # Collagens
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
    "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
    "COL7A1", "COL8A1", "COL10A1", "COL11A1", "COL12A1",
    "COL14A1", "COL15A1", "COL16A1", "COL18A1",
    
    # ECM glycoproteins
    "FN1", "FBN1", "FBN2",
    "TNC", "TNR", "TNXB",
    "SPARC", "POSTN", "SPP1",
    "LAMA1", "LAMA2", "LAMA3", "LAMA4", "LAMA5",
    "LAMB1", "LAMB2", "LAMB3",
    "LAMC1", "LAMC2", "LAMC3",
    "THBS1", "THBS2", "THBS4",
    "VTN", "NID1", "NID2",
    "FBLN1", "FBLN2", "FBLN5",
    
    # Proteoglycans
    "VCAN", "ACAN", "BCAN", "NCAN",
    "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN",
    "HSPG2", "AGRN",
    "GPC1", "GPC2", "GPC3",
    "SDC1", "SDC4",
    "CSPG4", "CSPG5", "PTPRZ1",
    "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",
    
    # Proteases
    "MMP1", "MMP2", "MMP3", "MMP7", "MMP9", "MMP11", "MMP12", "MMP13", "MMP14", "MMP16",
    "ADAMTS1", "ADAMTS4", "ADAMTS5", "ADAMTS8",
    "ADAM8",
    "HPSE",
    "PLAU", "PLAT",
    
    # Protease inhibitors
    "TIMP1", "TIMP2", "TIMP3",
    "SERPINE1",
    
    # Crosslinking enzymes
    "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4",
    "TGM2",
    "P4HA1", "P4HA2", "P4HA3",
    "PLOD1", "PLOD2", "PLOD3",
    
    # Hyaluronan metabolism
    "HAS1", "HAS2", "HAS3",
    "HYAL1", "HYAL2",
]

# Combine all pathway sets into one lookup
all_pathway_sets = {}
if "hallmark_sets" in globals():
    all_pathway_sets.update(hallmark_sets)
if "kegg_sets" in globals():
    all_pathway_sets.update(kegg_sets)
if "biocarta_sets" in globals():
    all_pathway_sets.update(biocarta_sets)

if not all_pathway_sets:
    raise RuntimeError("No pathway sets found. Run ULM scoring cell first.")

print(f"Available pathway sets: {len(all_pathway_sets)}")

# Validate selected pathways exist
missing = [p for p in selected_pathways if p not in all_pathway_sets]
if missing:
    print("\nMissing pathways:")
    for m in missing:
        print(f"  {m}")
    raise RuntimeError(f"{len(missing)} selected pathways not found in available sets")

# Build gene filter
pathway_genes = set(ecm_genes_brain)
for pathway in selected_pathways:
    pathway_genes.update(all_pathway_sets[pathway])

PATHWAY_FILTER = sorted(pathway_genes)

print(f"\nSelected pathways: {len(selected_pathways)}")
print(f"Custom gene lists: 1 (ecm_genes_brain: {len(ecm_genes_brain)} genes)")
print(f"PATHWAY_FILTER: {len(PATHWAY_FILTER)} unique genes")

### Config

In [ ]:
# Cell 1: Configuration

from pathlib import Path

ZONE = "BZ1"

# Set this to False for unbiased targets
USE_PATHWAY_FILTER = True

# If USE_PATHWAY_FILTER is True, PATHWAY_FILTER must already exist as an iterable of gene symbols
if USE_PATHWAY_FILTER:
    if "PATHWAY_FILTER" not in globals():
        raise NameError("USE_PATHWAY_FILTER=True but PATHWAY_FILTER is not defined in the notebook.")
    PATHWAY_FILTER = set(map(str, PATHWAY_FILTER))
else:
    PATHWAY_FILTER = None

FDR_CUTOFF = 0.05
LOGFC_CUTOFF = 0.25

# Axis construction and selection controls
TOP_LIGANDS_PER_DIRECTION = 50
LTM_TOPK_PER_TARGET = 7
TOP_AXES_PER_TARGET = 7
SENDER_TOPK_PER_LIGAND = 7  # set to None to keep all passing senders

# Ligand diversity controls within a target (used in Cell 5)
TOP_LIGANDS_PER_TARGET = 7
TOP_AXES_PER_LIGAND_PER_TARGET = 7

# Catalog truncation control (used in Cell 6)
CATALOG_MAX_AXES_PER_TARGET = 7  # set to None for no truncation, or increase to browse more

# Expression threshold for target->receiver mapping
FRACTION_CUTOFF = 0.05

# IMPORTANT: keep this consistent with other cells (they use /mnt/...)
EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
if not EFS_ROOT.exists():
    raise FileNotFoundError(f"EFS_ROOT does not exist: {EFS_ROOT}")

GATING_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_gating"
OUT_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_analysis" / ZONE.replace("+", "plus").replace("-", "minus")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Zone: {ZONE}")
print(f"Pathway filter: {'None (unbiased)' if PATHWAY_FILTER is None else f'{len(PATHWAY_FILTER)} genes'}")
print(f"FDR_CUTOFF: {FDR_CUTOFF}")
print(f"LOGFC_CUTOFF: {LOGFC_CUTOFF}")
print(f"FRACTION_CUTOFF: {FRACTION_CUTOFF}")
print(f"Output: {OUT_DIR}")

print(f"TOP_LIGANDS_PER_DIRECTION: {TOP_LIGANDS_PER_DIRECTION}")
print(f"LTM_TOPK_PER_TARGET: {LTM_TOPK_PER_TARGET}")
print(f"TOP_AXES_PER_TARGET: {TOP_AXES_PER_TARGET}")
print(f"SENDER_TOPK_PER_LIGAND: {SENDER_TOPK_PER_LIGAND}")

print(f"TOP_LIGANDS_PER_TARGET: {TOP_LIGANDS_PER_TARGET}")
print(f"TOP_AXES_PER_LIGAND_PER_TARGET: {TOP_AXES_PER_LIGAND_PER_TARGET}")
print(f"CATALOG_MAX_AXES_PER_TARGET: {CATALOG_MAX_AXES_PER_TARGET}")

### Zone background gene universe

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Defines these as sorted Python lists of gene symbols for downstream %%R cells:
#   background_high, background_low
#   ligands_high, ligands_low
#   receptors_high, receptors_low
#
# Also keeps a structured dict for inspection:
#   universes_panel

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
MIN_PCT_EXPRESSED = 0.10

required = ["adata_sc", "nichenet_priors", "SENDERS", "RECEIVERS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required variables: {missing}")

if LAYER not in adata_sc.layers:
    raise KeyError(
        f"adata_sc.layers['{LAYER}'] not found. Available layers: {list(adata_sc.layers.keys())}"
    )
for k in [CT_KEY, GROUP_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(
            f"adata_sc.obs['{k}'] not found. Available obs columns (first 60): {list(adata_sc.obs.columns)[:60]}"
        )

for k in ["ltm_targets", "ltm_ligands", "lr_ligands", "lr_receptors"]:
    if k not in nichenet_priors:
        raise KeyError(f"nichenet_priors['{k}'] not found. Available keys: {list(nichenet_priors.keys())}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = set(std_ct(x) for x in SENDERS)
receivers_std = set(std_ct(x) for x in RECEIVERS)

ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

expected_groups = {"TIL_high", "TIL_low"}
groups_present = set(grp.unique())
if not expected_groups.issubset(groups_present):
    raise ValueError(f"Expected {expected_groups} in {GROUP_KEY}, found: {sorted(groups_present)}")

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

ltm_targets = set(map(str, nichenet_priors["ltm_targets"]))
ltm_ligands = set(map(str, nichenet_priors["ltm_ligands"]))
lr_ligands = set(map(str, nichenet_priors["lr_ligands"]))
lr_receptors = set(map(str, nichenet_priors["lr_receptors"]))

candidate_background = ltm_targets
candidate_ligands = ltm_ligands.intersection(lr_ligands)
candidate_receptors = lr_receptors

def expressed_genes(mask_cells: np.ndarray, candidate_genes: set, min_pct: float) -> set:
    idx = np.flatnonzero(mask_cells)
    if idx.size == 0:
        return set()

    genes_present = sorted(candidate_genes.intersection(var_set))
    if not genes_present:
        return set()

    gene_idx = var_names.get_indexer(genes_present)
    Xsub = X[idx, :][:, gene_idx]

    nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
    pct = nnz / float(idx.size)

    keep = pct >= float(min_pct)
    return set(np.asarray(genes_present, dtype=object)[keep])

universes_panel = {}

for g in ["TIL_high", "TIL_low"]:
    sender_mask = (grp == g).to_numpy() & ct.isin(senders_std).to_numpy()
    receiver_mask = (grp == g).to_numpy() & ct.isin(receivers_std).to_numpy()

    bg = expressed_genes(receiver_mask, candidate_background, MIN_PCT_EXPRESSED)
    lig = expressed_genes(sender_mask, candidate_ligands, MIN_PCT_EXPRESSED)
    rec = expressed_genes(receiver_mask, candidate_receptors, MIN_PCT_EXPRESSED)

    universes_panel[g] = {
        "background": bg,
        "ligands": lig,
        "receptors": rec,
        "n_sender_cells": int(sender_mask.sum()),
        "n_receiver_cells": int(receiver_mask.sum()),
    }

background_high = sorted(map(str, universes_panel["TIL_high"]["background"]))
background_low = sorted(map(str, universes_panel["TIL_low"]["background"]))
ligands_high = sorted(map(str, universes_panel["TIL_high"]["ligands"]))
ligands_low = sorted(map(str, universes_panel["TIL_low"]["ligands"]))
receptors_high = sorted(map(str, universes_panel["TIL_high"]["receptors"]))
receptors_low = sorted(map(str, universes_panel["TIL_low"]["receptors"]))

summary_df = pd.DataFrame(
    [
        {
            "group": g,
            "n_sender_cells": universes_panel[g]["n_sender_cells"],
            "n_receiver_cells": universes_panel[g]["n_receiver_cells"],
            "n_background": len(universes_panel[g]["background"]),
            "n_ligands": len(universes_panel[g]["ligands"]),
            "n_receptors": len(universes_panel[g]["receptors"]),
        }
        for g in ["TIL_high", "TIL_low"]
    ]
)

print(summary_df.to_string(index=False))

print("\nUniverse integrity checks:")
print("  candidate_background (ltm_targets):", len(candidate_background))
print("  candidate_ligands (ltm_ligands ∩ lr_ligands):", len(candidate_ligands))
print("  candidate_receptors (lr_receptors):", len(candidate_receptors))
print("  var genes:", len(var_names))
print("  overlap candidate_ligands with var:", len(candidate_ligands.intersection(var_set)))
print("  overlap candidate_receptors with var:", len(candidate_receptors.intersection(var_set)))

if len(background_high) == 0 or len(background_low) == 0:
    raise ValueError("Background is empty for at least one group. Check receiver panel, group labels, and MIN_PCT_EXPRESSED.")
if len(ligands_high) == 0 or len(ligands_low) == 0:
    raise ValueError("Ligands are empty for at least one group. Check sender panel, group labels, and MIN_PCT_EXPRESSED.")
if len(receptors_high) == 0 or len(receptors_low) == 0:
    raise ValueError("Receptors are empty for at least one group. Check receiver panel, group labels, and MIN_PCT_EXPRESSED.")

print("\nExported lists ready for R:")
print("  background_high:", len(background_high), "background_low:", len(background_low))
print("  ligands_high   :", len(ligands_high), "ligands_low   :", len(ligands_low))
print("  receptors_high :", len(receptors_high), "receptors_low :", len(receptors_low))

### Target gene universe

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Builds canonical objects for this session:
#   targets_high, targets_low
#   universes (dict with targets/ligands/background/receptors per direction)
# Keeps de_df for downstream axis construction (Cell 5 expects de_df).

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
ZONE_KEY = ZONE  # CHANGED: was "TIL+"

MIN_PCT_EXPRESSED = 0.10

required = [
    "adata_sc", "nichenet_priors", "SENDERS", "RECEIVERS",
    "de_tables", "PATHWAY_FILTER", "FDR_CUTOFF", "LOGFC_CUTOFF"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects/params (run upstream cells): {missing}")

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

for k in [CT_KEY, GROUP_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found. Available (first 60): {list(adata_sc.obs.columns)[:60]}")

if ZONE_KEY not in de_tables:
    raise KeyError(f"de_tables['{ZONE_KEY}'] not found. Available keys: {list(de_tables.keys())}")

for k in ["ltm_targets", "ltm_ligands", "lr_ligands", "lr_receptors"]:
    if k not in nichenet_priors:
        raise KeyError(f"nichenet_priors['{k}'] not found. Available keys: {list(nichenet_priors.keys())}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = set(std_ct(x) for x in SENDERS)
receivers_std = set(std_ct(x) for x in RECEIVERS)

ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

expected_groups = {"TIL_high", "TIL_low"}
groups_present = set(grp.unique())
if not expected_groups.issubset(groups_present):
    raise ValueError(f"Expected {expected_groups} in {GROUP_KEY}, found: {sorted(groups_present)}")

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

ltm_targets = set(map(str, nichenet_priors["ltm_targets"]))
ltm_ligands = set(map(str, nichenet_priors["ltm_ligands"]))
lr_ligands = set(map(str, nichenet_priors["lr_ligands"]))
lr_receptors = set(map(str, nichenet_priors["lr_receptors"]))

candidate_background = ltm_targets
candidate_ligands = ltm_ligands.intersection(lr_ligands)
candidate_receptors = lr_receptors

def expressed_genes(mask_cells: np.ndarray, candidate_genes: set, min_pct: float) -> set:
    idx = np.flatnonzero(mask_cells)
    if idx.size == 0:
        return set()

    genes_present = sorted(candidate_genes.intersection(var_set))
    if not genes_present:
        return set()

    gene_idx = var_names.get_indexer(genes_present)
    Xsub = X[idx, :][:, gene_idx]

    nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
    pct = nnz / float(idx.size)

    keep = pct >= float(min_pct)
    return set(np.asarray(genes_present, dtype=object)[keep])

panel = {}
for g in ["TIL_high", "TIL_low"]:
    sender_mask = (grp == g).to_numpy() & ct.isin(senders_std).to_numpy()
    receiver_mask = (grp == g).to_numpy() & ct.isin(receivers_std).to_numpy()

    panel[g] = {
        "n_sender_cells": int(sender_mask.sum()),
        "n_receiver_cells": int(receiver_mask.sum()),
        "background": expressed_genes(receiver_mask, candidate_background, MIN_PCT_EXPRESSED),
        "ligands": expressed_genes(sender_mask, candidate_ligands, MIN_PCT_EXPRESSED),
        "receptors": expressed_genes(receiver_mask, candidate_receptors, MIN_PCT_EXPRESSED),
    }

de_df = de_tables[ZONE_KEY].copy()

req = {"gene", "FDR", "logFC"}
if not req.issubset(set(de_df.columns)):
    raise KeyError(f"de_tables['{ZONE_KEY}'] must contain {sorted(req)}. Found: {list(de_df.columns)}")

de_df["gene"] = de_df["gene"].astype(str)
de_df["FDR"] = pd.to_numeric(de_df["FDR"], errors="coerce")
de_df["logFC"] = pd.to_numeric(de_df["logFC"], errors="coerce")

# DE-defined targets by direction (sign anchored to TIL_high_vs_low)
de_high = set(
    de_df.loc[
        (de_df["FDR"] <= float(FDR_CUTOFF)) & (de_df["logFC"] >= float(LOGFC_CUTOFF)),
        "gene"
    ].dropna()
)
de_low = set(
    de_df.loc[
        (de_df["FDR"] <= float(FDR_CUTOFF)) & (de_df["logFC"] <= -float(LOGFC_CUTOFF)),
        "gene"
    ].dropna()
)

# Gate to receiver-expressed background per group
targets_high = de_high.intersection(panel["TIL_high"]["background"])
targets_low = de_low.intersection(panel["TIL_low"]["background"])

targets_high_pre = set(targets_high)
targets_low_pre = set(targets_low)

# Optional pathway filter (apply after background gating)
if PATHWAY_FILTER is not None:
    pf = set(map(str, PATHWAY_FILTER))
    targets_high = targets_high.intersection(pf)
    targets_low = targets_low.intersection(pf)

universes = {
    "targets_high": sorted(map(str, targets_high)),
    "targets_low": sorted(map(str, targets_low)),
    "ligands_high": sorted(map(str, panel["TIL_high"]["ligands"])),
    "ligands_low": sorted(map(str, panel["TIL_low"]["ligands"])),
    "background_high": sorted(map(str, panel["TIL_high"]["background"])),
    "background_low": sorted(map(str, panel["TIL_low"]["background"])),
    "receptors_high": sorted(map(str, panel["TIL_high"]["receptors"])),
    "receptors_low": sorted(map(str, panel["TIL_low"]["receptors"])),
}

summary_df = pd.DataFrame(
    [
        {
            "group": g,
            "n_sender_cells": panel[g]["n_sender_cells"],
            "n_receiver_cells": panel[g]["n_receiver_cells"],
            "n_background": len(panel[g]["background"]),
            "n_ligands": len(panel[g]["ligands"]),
            "n_receptors": len(panel[g]["receptors"]),
        }
        for g in ["TIL_high", "TIL_low"]
    ]
)

print(summary_df.to_string(index=False))

print("\nTargets:")
print("  targets_high (DE∩background) pre pathway:", len(targets_high_pre), "post:", len(targets_high))
print("  targets_low  (DE∩background) pre pathway:", len(targets_low_pre),  "post:", len(targets_low))
print("  pathway filter enabled:", PATHWAY_FILTER is not None)

if PATHWAY_FILTER is not None:
    pf_set = set(map(str, PATHWAY_FILTER))
    print("  targets_high removed by PATHWAY_FILTER:", len(targets_high_pre - pf_set))
    print("  targets_low  removed by PATHWAY_FILTER:", len(targets_low_pre - pf_set))

if len(universes["targets_high"]) == 0 or len(universes["targets_low"]) == 0:
    raise ValueError(
        "Targets empty after DE/background gating and (optional) pathway filter. "
        "Check thresholds, DE orientation, and PATHWAY_FILTER."
    )

### Map targets to receiver cell types by expression fraction

In [ ]:
# %% [markdown]
# ### Map targets to receiver cell types by expression fraction

# %%
import numpy as np
import pandas as pd
import scipy.sparse as sp

# === CLEAR STALE VARIABLES ===
for _v in ["target_receiver_map_high", "target_receiver_map_low", "target_panel_coverage",
           "global_best_high", "global_best_low", "all_celltypes", "valid_celltypes", 
           "excluded_celltypes", "ct", "grp", "X", "var_names", "var_set"]:
    if _v in globals():
        del globals()[_v]

required = ["adata_sc", "targets_high", "targets_low", "RECEIVERS", "FRACTION_CUTOFF"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects/params: {missing}")

# === VERIFY INPUTS ===
print(f"INPUT CHECK:")
print(f"  targets_high: {len(targets_high)} genes")
print(f"  targets_low: {len(targets_low)} genes")
print(f"  RECEIVERS: {len(RECEIVERS)} cell types")
print(f"  FRACTION_CUTOFF: {FRACTION_CUTOFF}")

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"

MIN_SAMPLES_EXPRESSING = 3
PANEL_COVERAGE_THRESHOLD = 0.50

# Minimum cells required for a cell type to be considered in global best computation
# This prevents rare cell types (e.g., 1 cell) from dominating due to noise
MIN_CELLS_FOR_GLOBAL_BEST = 50

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found.")
for k in [CT_KEY, GROUP_KEY, SAMPLE_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found.")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

def is_garbage_ct(ct_name: str) -> bool:
    """Exclude Doublet, Other, Unknown, Unassigned cell types."""
    s = str(ct_name)
    for bad in ["Doublet", "doublet", "Other", "other", "Unknown", "Unassigned"]:
        if bad in s:
            return True
    return False

receivers_std = sorted(set(std_ct(x) for x in RECEIVERS))

# Build fresh from adata_sc
ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)
sample_ids = adata_sc.obs[SAMPLE_KEY].astype(str)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

# Get valid cell types (excluding garbage)
all_celltypes_raw = sorted(ct.unique())
valid_celltypes = [c for c in all_celltypes_raw if not is_garbage_ct(c)]
excluded_celltypes = [c for c in all_celltypes_raw if is_garbage_ct(c)]

print(f"\nCELL TYPE FILTERING:")
print(f"  Raw cell types: {len(all_celltypes_raw)}")
print(f"  Valid cell types (no Doublet/Other): {len(valid_celltypes)}")
print(f"  Excluded: {len(excluded_celltypes)}")

# Compute cell counts per cell type per group
celltype_counts = {}
for group_label in ["TIL_high", "TIL_low"]:
    celltype_counts[group_label] = {}
    for celltype in valid_celltypes:
        m = (grp == group_label).to_numpy() & (ct == celltype).to_numpy()
        n_cells = m.sum()
        celltype_counts[group_label][celltype] = n_cells

# Filter valid cell types for global best by minimum cell count
valid_celltypes_for_global = {}
for group_label in ["TIL_high", "TIL_low"]:
    valid_celltypes_for_global[group_label] = [
        c for c in valid_celltypes 
        if celltype_counts[group_label][c] >= MIN_CELLS_FOR_GLOBAL_BEST
    ]
    n_excluded_by_count = len(valid_celltypes) - len(valid_celltypes_for_global[group_label])
    print(f"  {group_label}: {len(valid_celltypes_for_global[group_label])} cell types with >= {MIN_CELLS_FOR_GLOBAL_BEST} cells ({n_excluded_by_count} excluded)")

def build_trm_for_group(group_label: str, targets_panel) -> pd.DataFrame:
    """Build target-receiver map for panel receivers only."""
    targets_panel = [str(x) for x in list(targets_panel)]
    targets_present = [t for t in targets_panel if t in var_set]
    if len(targets_present) == 0:
        raise ValueError(f"No targets found in var_names for {group_label}.")

    print(f"  {group_label}: {len(targets_present)} targets in var_names")
    
    gene_idx = var_names.get_indexer(targets_present)
    rows = []

    for rct in receivers_std:
        m = (grp == group_label).to_numpy() & (ct == rct).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue

        sample_sub = sample_ids.iloc[idx].values
        n_samples = len(np.unique(sample_sub))

        Xsub = X[idx, :][:, gene_idx]
        nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
        pct = nnz / float(idx.size)
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        Xsub_dense = Xsub.toarray()

        for i, target in enumerate(targets_present):
            gene_expr = Xsub_dense[:, i]
            n_samples_expressing = len(set(sample_sub[gene_expr > 0]))
            sample_frac = n_samples_expressing / n_samples if n_samples > 0 else 0.0

            if pct[i] >= float(FRACTION_CUTOFF) and n_samples_expressing >= MIN_SAMPLES_EXPRESSING:
                rows.append({
                    "target": target,
                    "receiver_celltype": rct,
                    "target_pct_expr": float(pct[i]),
                    "target_mean_expr": float(mean_expr[i]),
                    "n_samples": n_samples,
                    "n_samples_expressing": n_samples_expressing,
                    "sample_frac": float(sample_frac),
                })

    if not rows:
        return pd.DataFrame(columns=[
            "target", "receiver_celltype", "target_pct_expr", "target_mean_expr",
            "n_samples", "n_samples_expressing", "sample_frac", "specificity", "receiver_rank_score"
        ])

    out = pd.DataFrame(rows)

    # Specificity within panel
    target_mean_all = out.groupby("target")["target_mean_expr"].transform("mean").replace(0, np.nan)
    out["specificity"] = (out["target_mean_expr"] / target_mean_all).fillna(0)

    # Receiver rank score
    out["_mr"] = out.groupby("target")["target_mean_expr"].rank(ascending=False)
    out["_pr"] = out.groupby("target")["target_pct_expr"].rank(ascending=False)
    out["_sfr"] = out.groupby("target")["sample_frac"].rank(ascending=False)
    out["_spr"] = out.groupby("target")["specificity"].rank(ascending=False)
    out["receiver_rank_score"] = 0.35/out["_mr"] + 0.25/out["_pr"] + 0.20/out["_sfr"] + 0.20/out["_spr"]
    out = out.drop(columns=["_mr", "_pr", "_sfr", "_spr"])

    return out

def compute_global_best(group_label: str, targets_list) -> pd.DataFrame:
    """Find global best cell type per target.
    
    Only considers cell types with >= MIN_CELLS_FOR_GLOBAL_BEST cells
    to avoid noise from rare populations.
    """
    targets_list = [str(x) for x in targets_list if str(x) in var_set]
    if not targets_list:
        return pd.DataFrame(columns=["target", "global_best_celltype", "global_best_mean_expr", "global_best_n_cells"])

    gene_idx = var_names.get_indexer(targets_list)
    best = {t: (None, 0.0, 0) for t in targets_list}  # (celltype, mean_expr, n_cells)

    # Only use cell types with sufficient cells
    celltypes_to_check = valid_celltypes_for_global[group_label]
    
    for celltype in celltypes_to_check:
        m = (grp == group_label).to_numpy() & (ct == celltype).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue
        n_cells = idx.size
        Xsub = X[idx, :][:, gene_idx]
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        for i, target in enumerate(targets_list):
            if mean_expr[i] > best[target][1]:
                best[target] = (celltype, float(mean_expr[i]), n_cells)

    results = [{"target": t, "global_best_celltype": v[0], "global_best_mean_expr": v[1], "global_best_n_cells": v[2]} 
               for t, v in best.items() if v[0] is not None]
    return pd.DataFrame(results)

def add_coverage(trm: pd.DataFrame, global_best: pd.DataFrame) -> pd.DataFrame:
    if trm.empty:
        return trm
    trm = trm.merge(global_best, on="target", how="left")
    panel_best = trm.groupby("target")["target_mean_expr"].transform("max")
    trm["panel_coverage_ratio"] = np.where(
        trm["global_best_mean_expr"] > 0,
        panel_best / trm["global_best_mean_expr"],
        1.0
    )
    trm["panel_is_adequate"] = trm["panel_coverage_ratio"] >= PANEL_COVERAGE_THRESHOLD
    trm["coverage_flag"] = np.where(trm["panel_is_adequate"], "OK", "LOW_COVERAGE")
    return trm

# === BUILD TRMs ===
print("\nBUILDING TARGET-RECEIVER MAPS:")
target_receiver_map_high = build_trm_for_group("TIL_high", targets_high)
target_receiver_map_low = build_trm_for_group("TIL_low", targets_low)

# === COMPUTE GLOBAL BEST ===
print(f"\nCOMPUTING GLOBAL BEST (min {MIN_CELLS_FOR_GLOBAL_BEST} cells, excluding Doublets/Other):")
global_best_high = compute_global_best("TIL_high", targets_high)
global_best_low = compute_global_best("TIL_low", targets_low)

# Verify no garbage in global_best
for name, gb in [("high", global_best_high), ("low", global_best_low)]:
    if not gb.empty:
        garbage = [ct for ct in gb["global_best_celltype"].dropna().unique() if is_garbage_ct(ct)]
        if garbage:
            raise RuntimeError(f"BUG: Garbage cell types in global_best_{name}: {garbage}")
        print(f"  global_best_{name}: {len(gb)} targets, no garbage cell types ✓")

# === ADD COVERAGE ===
target_receiver_map_high = add_coverage(target_receiver_map_high, global_best_high)
target_receiver_map_low = add_coverage(target_receiver_map_low, global_best_low)

# === SUMMARY ===
print("\nTARGET-RECEIVER MAP SUMMARY:")
for name, trm in [("TIL_high", target_receiver_map_high), ("TIL_low", target_receiver_map_low)]:
    print(f"\n  {name}:")
    print(f"    rows: {len(trm)}")
    print(f"    targets: {trm['target'].nunique()}")
    print(f"    receivers: {trm['receiver_celltype'].nunique()}")
    if "coverage_flag" in trm.columns:
        n_ok = (trm["coverage_flag"] == "OK").sum()
        n_low = (trm["coverage_flag"] == "LOW_COVERAGE").sum()
        print(f"    coverage: OK={n_ok}, LOW_COVERAGE={n_low}")

# Show LOW_COVERAGE targets
print("\nLOW_COVERAGE TARGETS:")
for name, trm in [("TIL_high", target_receiver_map_high), ("TIL_low", target_receiver_map_low)]:
    low = trm[trm["coverage_flag"] == "LOW_COVERAGE"][["target", "global_best_celltype", "global_best_n_cells", "panel_coverage_ratio"]].drop_duplicates()
    if low.empty:
        print(f"  {name}: None")
    else:
        print(f"  {name}:")
        for _, r in low.iterrows():
            print(f"    {r['target']}: global_best={r['global_best_celltype']} (n={r['global_best_n_cells']}), coverage={r['panel_coverage_ratio']:.1%}")

if target_receiver_map_high.empty or target_receiver_map_low.empty:
    raise ValueError("One or both target_receiver_maps are empty!")

# Store for downstream
target_panel_coverage = pd.concat([
    global_best_high.assign(direction="TIL_high"),
    global_best_low.assign(direction="TIL_low")
], ignore_index=True)

print("\n✓ Created: target_receiver_map_high, target_receiver_map_low, target_panel_coverage")

### Build SCE (python)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmwrite

if "adata_sc" not in globals():
    raise NameError("adata_sc not found")
if "SENDERS" not in globals() or "RECEIVERS" not in globals():
    raise NameError("SENDERS and/or RECEIVERS not found")

COUNTS_LAYER = "unnormalized_counts"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"
BATCH_KEY = "orig.ident"

if COUNTS_LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{COUNTS_LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

for k in [CT_KEY, GROUP_KEY, SAMPLE_KEY, BATCH_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found. Available obs cols (first 50): {list(adata_sc.obs.columns)[:50]}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

def drop_bad_ct(ct_list):
    out = []
    for ct in ct_list:
        s = str(ct)
        if s.startswith("Doublet"):
            continue
        if s.startswith("Other"):
            continue
        out.append(s)
    return sorted(set(out))

senders_py = drop_bad_ct(list(SENDERS))
receivers_py = drop_bad_ct(list(RECEIVERS))

senders_std = [std_ct(x) for x in senders_py]
receivers_std = [std_ct(x) for x in receivers_py]
panel_ct_std = sorted(set(senders_std) | set(receivers_std))

ct_std = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

m = grp.isin(["TIL_low", "TIL_high"]) & ct_std.isin(panel_ct_std)
n_keep = int(m.sum())
if n_keep == 0:
    raise ValueError("No cells left after filtering to (TIL_low/TIL_high) and sender/receiver panel celltypes.")

adata_sub = adata_sc[m].copy()

X = adata_sub.layers[COUNTS_LAYER]
if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()

EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
zone_dir = ZONE.replace("+", "plus").replace("-", "minus")  # CHANGED: derive from ZONE
out_dir = EFS_ROOT / "results" / "visium" / "til_zones" / "nichenet_filtering" / f"mnn_{zone_dir}_input"  # CHANGED: use zone_dir
out_dir.mkdir(parents=True, exist_ok=True)

for fname in ["counts.mtx", "features.tsv", "barcodes.tsv", "metadata.csv", "senders.txt", "receivers.txt"]:
    fpath = out_dir / fname
    if fpath.exists():
        fpath.unlink()

counts_gene_cell = X.T.tocsc()
mmwrite(str(out_dir / "counts.mtx"), counts_gene_cell)

genes = pd.Series(adata_sub.var_names.astype(str))
genes.to_csv(out_dir / "features.tsv", sep="\t", index=False, header=False)

cells = pd.Series(adata_sub.obs_names.astype(str))
cells.to_csv(out_dir / "barcodes.tsv", sep="\t", index=False, header=False)

ct_sub_std = adata_sub.obs[CT_KEY].astype(str).map(std_ct).values

meta = pd.DataFrame(
    {
        "cell": cells.values,
        "celltype": ct_sub_std,
        "sample": adata_sub.obs[SAMPLE_KEY].astype(str).values,
        "group": adata_sub.obs[GROUP_KEY].astype(str).values,
        "batch": adata_sub.obs[BATCH_KEY].astype(str).values,
    }
)
meta.to_csv(out_dir / "metadata.csv", index=False)

pd.Series(senders_std).to_csv(out_dir / "senders.txt", index=False, header=False)
pd.Series(receivers_std).to_csv(out_dir / "receivers.txt", index=False, header=False)

# ADDED: Export for R cell
SCE_INPUT_DIR = f"mnn_{zone_dir}_input"

print("Exported for SCE:")
print("  out_dir:", str(out_dir))
print("  genes x cells:", counts_gene_cell.shape[0], "x", counts_gene_cell.shape[1])
print("  nnz:", int(counts_gene_cell.nnz))
print("  groups:", dict(pd.Series(meta["group"]).value_counts()))
print("  celltypes in panel (senders, receivers):", len(senders_std), ",", len(receivers_std))

### Build SCE (R)

In [ ]:
%%R -i SCE_INPUT_DIR
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(S4Vectors)
  library(dplyr)
  library(readr)
  library(multinichenetr)
})

EFS_ROOT  <- "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd"
in_dir <- file.path(EFS_ROOT, "results", "visium", "til_zones", "nichenet_filtering", SCE_INPUT_DIR)  # CHANGED: use variable
stopifnot(dir.exists(in_dir))

counts <- readMM(file.path(in_dir, "counts.mtx"))
counts <- as(counts, "dgCMatrix")

feat <- read_tsv(file.path(in_dir, "features.tsv"), col_names = FALSE, show_col_types = FALSE)
genes <- if (ncol(feat) == 1) feat[[1]] else feat[[2]]

cells <- read_tsv(file.path(in_dir, "barcodes.tsv"), col_names = FALSE, show_col_types = FALSE)[[1]]
meta  <- read_csv(file.path(in_dir, "metadata.csv"), show_col_types = FALSE)

stopifnot(nrow(counts) == length(genes))
stopifnot(ncol(counts) == length(cells))

rownames(counts) <- genes
colnames(counts) <- cells

stopifnot("cell" %in% colnames(meta))
meta <- meta %>% tibble::column_to_rownames("cell")
meta <- meta[colnames(counts), , drop = FALSE]

stopifnot(all(c("celltype", "sample", "group", "batch") %in% colnames(meta)))

meta$celltype_mnn <- make.names(meta$celltype)

sce <- SingleCellExperiment(
  assays = list(counts = counts),
  colData = S4Vectors::DataFrame(meta),
  rowData = S4Vectors::DataFrame(gene = rownames(counts))
)

stopifnot("sample" %in% colnames(colData(sce)))
stopifnot("celltype_mnn" %in% colnames(colData(sce)))
stopifnot("group" %in% colnames(colData(sce)))

colData(sce)$sample_id <- as.character(colData(sce)$sample)
colData(sce)$celltype  <- as.character(colData(sce)$celltype_mnn)

if (anyNA(colData(sce)$sample_id)) stop("sample_id has NA values (check sample in metadata.csv).")
if (anyNA(colData(sce)$celltype))  stop("celltype has NA values (check celltype in metadata.csv).")

senders_oi   <- make.names(readLines(file.path(in_dir, "senders.txt")))
receivers_oi <- make.names(readLines(file.path(in_dir, "receivers.txt")))

ct_levels <- unique(colData(sce)$celltype)
missing_senders   <- setdiff(senders_oi, ct_levels)
missing_receivers <- setdiff(receivers_oi, ct_levels)
if (length(missing_senders) > 0) {
  stop("Senders not found in sce$celltype: ", paste(missing_senders, collapse = ", "))
}
if (length(missing_receivers) > 0) {
  stop("Receivers not found in sce$celltype: ", paste(missing_receivers, collapse = ", "))
}

cat("SCE:", dim(sce)[1], "genes x", dim(sce)[2], "cells\n")
cat("Groups (group):\n")
print(table(colData(sce)$group, useNA = "ifany"))
cat("\nCelltypes (celltype) top 15:\n")
print(sort(table(colData(sce)$celltype), decreasing = TRUE)[1:15])

cat("\nSenders (n=", length(senders_oi), "):\n", sep = "")
print(senders_oi)
cat("\nReceivers (n=", length(receivers_oi), "):\n", sep = "")
print(receivers_oi)

stopifnot(exists("lr_network"), exists("ligand_target_matrix"), exists("weighted_networks"))
cat("\nPriors OK:\n")
cat("  lr_network dim:", paste(dim(lr_network), collapse = " x "), "\n")
cat("  ligand_target_matrix dim:", paste(dim(ligand_target_matrix), collapse = " x "), "\n")
cat("  weighted_networks class:", paste(class(weighted_networks), collapse = ","), "\n")

### Compute ligand activity per direction

In [ ]:
# Make R-safe copies (rpy2 cannot convert Python sets)
targets_high_r = sorted(list(targets_high))
targets_low_r  = sorted(list(targets_low))

background_high_r = list(background_high)
background_low_r  = list(background_low)
ligands_high_r    = list(ligands_high)
ligands_low_r     = list(ligands_low)

print("R-ready objects:")
print("  targets_high_r:", type(targets_high_r), len(targets_high_r))
print("  targets_low_r :", type(targets_low_r),  len(targets_low_r))
print("  background_high_r:", type(background_high_r), len(background_high_r))
print("  background_low_r :", type(background_low_r),  len(background_low_r))
print("  ligands_high_r:", type(ligands_high_r), len(ligands_high_r))
print("  ligands_low_r :", type(ligands_low_r),  len(ligands_low_r))

In [ ]:
%%R -i targets_high_r -i targets_low_r -i background_high_r -i background_low_r -i ligands_high_r -i ligands_low_r
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
  library(nichenetr)
})

stopifnot(exists("ligand_target_matrix"))

to_chr_unique <- function(x) {
  unique(as.character(unlist(x, use.names = FALSE)))
}

targets_high_chr    <- to_chr_unique(targets_high_r)
targets_low_chr     <- to_chr_unique(targets_low_r)
background_high_chr <- to_chr_unique(background_high_r)
background_low_chr  <- to_chr_unique(background_low_r)
ligands_high_chr    <- to_chr_unique(ligands_high_r)
ligands_low_chr     <- to_chr_unique(ligands_low_r)

ltm <- ligand_target_matrix
ltm_ligands <- rownames(ltm)
ltm_targets <- colnames(ltm)

targets_high_use <- intersect(targets_high_chr, ltm_targets)
targets_low_use  <- intersect(targets_low_chr,  ltm_targets)

bg_high_use <- intersect(background_high_chr, ltm_targets)
bg_low_use  <- intersect(background_low_chr,  ltm_targets)

lig_high_use <- intersect(ligands_high_chr, ltm_ligands)
lig_low_use  <- intersect(ligands_low_chr,  ltm_ligands)

get_top_ltm_ligands <- function(targets_use, ltm, k_per_target) {
  out <- character()
  if (length(targets_use) == 0) return(out)

  for (t in targets_use) {
    w <- ltm[, t, drop = TRUE]
    w <- w[w > 0]
    if (length(w) == 0) next
    top <- sort(w, decreasing = TRUE)
    if (length(top) > k_per_target) top <- top[seq_len(k_per_target)]
    out <- c(out, names(top))
  }
  unique(out)
}

LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY <- 10

top_ltm_high <- get_top_ltm_ligands(targets_high_use, ltm, LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY)
top_ltm_low  <- get_top_ltm_ligands(targets_low_use,  ltm, LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY)

potential_high <- union(lig_high_use, top_ltm_high)
potential_low  <- union(lig_low_use,  top_ltm_low)

cat("Ligand activity inputs after LTM intersection + LTM augmentation:\n")
cat("  LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY=", LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY, "\n", sep = "")
cat("  TIL_high: targets=", length(targets_high_use),
    " background=", length(bg_high_use),
    " sender_panel_ligs=", length(lig_high_use),
    " ltm_top_ligs=", length(top_ltm_high),
    " potential_ligs=", length(potential_high), "\n", sep = "")
cat("  TIL_low : targets=", length(targets_low_use),
    " background=", length(bg_low_use),
    " sender_panel_ligs=", length(lig_low_use),
    " ltm_top_ligs=", length(top_ltm_low),
    " potential_ligs=", length(potential_low), "\n\n", sep = "")

if (length(targets_high_use) < 5) stop("Too few TIL_high targets after intersecting with ligand_target_matrix columns.")
if (length(targets_low_use)  < 5) stop("Too few TIL_low targets after intersecting with ligand_target_matrix columns.")
if (length(bg_high_use) < 50) stop("Too few TIL_high background genes after intersecting with ligand_target_matrix columns.")
if (length(bg_low_use)  < 50) stop("Too few TIL_low background genes after intersecting with ligand_target_matrix columns.")
if (length(potential_high) < 20) stop("Too few TIL_high potential ligands after augmentation.")
if (length(potential_low)  < 20) stop("Too few TIL_low potential ligands after augmentation.")

ligand_activities_high <- predict_ligand_activities(
  geneset = targets_high_use,
  background_expressed_genes = bg_high_use,
  potential_ligands = potential_high,
  ligand_target_matrix = ltm
) %>%
  as_tibble() %>%
  mutate(
    direction = "TIL_high",
    test_ligand = as.character(test_ligand),
    aupr_corrected = as.numeric(aupr_corrected),
    in_sender_panel = test_ligand %in% lig_high_use,
    in_ltm_top = test_ligand %in% top_ltm_high
  ) %>%
  arrange(desc(aupr_corrected), test_ligand) %>%
  mutate(rank = dplyr::row_number()) %>%
  select(test_ligand, direction, aupr_corrected, rank, in_sender_panel, in_ltm_top, everything())

ligand_activities_low <- predict_ligand_activities(
  geneset = targets_low_use,
  background_expressed_genes = bg_low_use,
  potential_ligands = potential_low,
  ligand_target_matrix = ltm
) %>%
  as_tibble() %>%
  mutate(
    direction = "TIL_low",
    test_ligand = as.character(test_ligand),
    aupr_corrected = as.numeric(aupr_corrected),
    in_sender_panel = test_ligand %in% lig_low_use,
    in_ltm_top = test_ligand %in% top_ltm_low
  ) %>%
  arrange(desc(aupr_corrected), test_ligand) %>%
  mutate(rank = dplyr::row_number()) %>%
  select(test_ligand, direction, aupr_corrected, rank, in_sender_panel, in_ltm_top, everything())

ligand_activities <- bind_rows(ligand_activities_high, ligand_activities_low)

cat("Ligand activities computed:\n")
cat("  rows:", nrow(ligand_activities), "\n")
cat("  TIL_high:", nrow(ligand_activities_high), "\n")
cat("  TIL_low :", nrow(ligand_activities_low), "\n\n")

cat("Top 10 (TIL_high):\n")
print(ligand_activities_high %>% slice_head(n = 10) %>% select(test_ligand, aupr_corrected, rank, in_sender_panel, in_ltm_top))
cat("\nTop 10 (TIL_low):\n")
print(ligand_activities_low %>% slice_head(n = 10) %>% select(test_ligand, aupr_corrected, rank, in_sender_panel, in_ltm_top))

### Ligand-receptor DE per cell type

In [ ]:
# Pass zone prefix to R for file naming
zone_prefix = ZONE.replace("+", "plus").replace("-", "minus")
print(f"Zone prefix for R: {zone_prefix}")

In [ ]:
%%R -i zone_prefix
# LR-DE: edgeR pseudobulk per celltype, TIL_high vs TIL_low
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(SummarizedExperiment)
  library(S4Vectors)
  library(dplyr)
  library(tibble)
  library(readr)
  library(edgeR)
})

stopifnot(exists("sce"), exists("senders_oi"), exists("receivers_oi"))
stopifnot(exists("lr_network"), exists("ligand_target_matrix"))
stopifnot(exists("ligand_activities"))
stopifnot("counts" %in% SummarizedExperiment::assayNames(sce))

EFS_ROOT  <- "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd"
FILTER_DIR <- file.path(EFS_ROOT, "results", "visium", "til_zones", "nichenet_filtering")
dir.create(FILTER_DIR, recursive = TRUE, showWarnings = FALSE)

GROUPS <- c("TIL_low","TIL_high")

MIN_CELLS_PER_SAMPLE  <- 5
MIN_SAMPLES_PER_GROUP <- 3

CT_COL     <- "celltype"
GROUP_COL  <- "group"
SAMPLE_COL <- "sample_id"

cd0 <- as.data.frame(colData(sce))
stopifnot(CT_COL %in% colnames(cd0), GROUP_COL %in% colnames(cd0), SAMPLE_COL %in% colnames(cd0))

cd0[[CT_COL]]     <- as.character(cd0[[CT_COL]])
cd0[[GROUP_COL]]  <- as.character(cd0[[GROUP_COL]])
cd0[[SAMPLE_COL]] <- as.character(cd0[[SAMPLE_COL]])

bad_sample <- is.na(cd0[[SAMPLE_COL]]) | cd0[[SAMPLE_COL]] == ""
if (any(bad_sample)) {
  ex <- unique(cd0[[SAMPLE_COL]][bad_sample])[1:min(10, sum(bad_sample))]
  stop(paste0("sample_id contains NA/empty values (n=", sum(bad_sample), "). Examples: ", paste(ex, collapse=", ")))
}

bad_group <- is.na(cd0[[GROUP_COL]]) | !(cd0[[GROUP_COL]] %in% GROUPS)
if (any(bad_group)) {
  ex <- unique(cd0[[GROUP_COL]][bad_group])[1:min(10, sum(bad_group))]
  stop(paste0("group has NA or unexpected labels (n=", sum(bad_group), "). Examples: ", paste(ex, collapse=", ")))
}

cat("Using columns:\n")
cat("  celltype:", CT_COL, "\n")
cat("  group   :", GROUP_COL, "\n")
cat("  sample  :", SAMPLE_COL, "\n")
cat("Cells:", nrow(cd0), "\n\n")

cat("Group counts in sce:\n")
print(table(cd0[[GROUP_COL]], useNA = "ifany"))
cat("\n")

assays_avail <- assayNames(sce)
if (!("counts" %in% assays_avail)) {
  stop(paste0("sce is missing assay 'counts'. Available assays: ", paste(assays_avail, collapse=", ")))
}
counts_all <- assay(sce, "counts")
if (!is(counts_all, "dgCMatrix")) counts_all <- as(counts_all, "CsparseMatrix")
counts_all <- as(counts_all, "dgCMatrix")

cat("Counts matrix:\n")
cat("  class:", class(counts_all)[1], "\n")
cat("  dim  :", nrow(counts_all), "x", ncol(counts_all), "\n\n")

cat("Ligand activities directions:\n")
print(table(as.character(ligand_activities$direction), useNA="ifany"))
cat("\n")

# Harmonize lr_network columns
lr_cols <- colnames(lr_network)
lig_col <- if ("from" %in% lr_cols) "from" else if ("ligand" %in% lr_cols) "ligand" else stop("lr_network missing ligand column (from/ligand).")
rec_col <- if ("to" %in% lr_cols) "to" else if ("receptor" %in% lr_cols) "receptor" else stop("lr_network missing receptor column (to/receptor).")

lr_tbl <- lr_network %>%
  dplyr::distinct(.data[[lig_col]], .data[[rec_col]])
colnames(lr_tbl) <- c("from","to")

# Feasible ligands: use NicheNet activity list, constrained to LTM rows + sce genes
ltm <- ligand_target_matrix
ltm_ligands <- rownames(ltm)
ltm_targets <- colnames(ltm)

feasible_ligands <- intersect(
  unique(as.character(ligand_activities$test_ligand)),
  intersect(ltm_ligands, rownames(counts_all))
)

# Feasible receptors: receptors paired to feasible ligands, constrained to sce genes
feasible_receptors <- intersect(
  unique(lr_tbl$to[lr_tbl$from %in% feasible_ligands]),
  rownames(counts_all)
)

cat("DE universes:\n")
cat("  feasible ligands   :", length(feasible_ligands), "\n")
cat("  feasible receptors :", length(feasible_receptors), "\n\n")

if (length(feasible_ligands) < 20) stop("Too few feasible ligands after intersecting activity ligands with LTM rows and sce genes.")
if (length(feasible_receptors) < 20) stop("Too few feasible receptors after intersecting LR receptors with sce genes.")

# Fast pseudobulk using sparse model matrix
pseudobulk_by_sample <- function(counts_gene_cell, sample_vec) {
  sample_f <- factor(sample_vec)
  M <- Matrix::sparse.model.matrix(~0 + sample_f)
  colnames(M) <- levels(sample_f)
  pb <- counts_gene_cell %*% M
  as(pb, "dgCMatrix")
}

run_edger_for_celltype <- function(celltype_name, genes_keep, mode_label) {
  tryCatch({

    idx_ct <- which(cd0[[CT_COL]] == celltype_name & cd0[[GROUP_COL]] %in% GROUPS)
    if (length(idx_ct) == 0) {
      return(list(status="skip", reason="no cells after filtering", de=NULL, meta=NULL))
    }

    meta_ct <- cd0[idx_ct, , drop=FALSE]

    cells_per_sample <- meta_ct %>%
      dplyr::count(.data[[SAMPLE_COL]], name="n_cells")

    keep_samples <- cells_per_sample %>%
      dplyr::filter(n_cells >= MIN_CELLS_PER_SAMPLE) %>%
      dplyr::pull(.data[[SAMPLE_COL]])

    if (length(keep_samples) == 0) {
      return(list(status="skip", reason=paste0("no samples with >= ", MIN_CELLS_PER_SAMPLE, " cells"), de=NULL, meta=cells_per_sample))
    }

    keep_mask <- meta_ct[[SAMPLE_COL]] %in% keep_samples
    meta_ct <- meta_ct[keep_mask, , drop=FALSE]
    idx_ct  <- idx_ct[keep_mask]

    sample_group_tbl <- meta_ct %>%
      dplyr::group_by(.data[[SAMPLE_COL]]) %>%
      dplyr::summarise(
        n_cells = dplyr::n(),
        n_groups = dplyr::n_distinct(.data[[GROUP_COL]]),
        group = dplyr::first(.data[[GROUP_COL]]),
        .groups="drop"
      )

    bad <- sample_group_tbl %>% dplyr::filter(n_groups > 1)
    if (nrow(bad) > 0) {
      return(list(status="skip", reason="sample maps to >1 group within this celltype", de=NULL, meta=sample_group_tbl))
    }

    n_by_group <- sample_group_tbl %>% dplyr::count(group, name="n_samples")
    n_low  <- n_by_group %>% dplyr::filter(group=="TIL_low")  %>% dplyr::pull(n_samples)
    n_high <- n_by_group %>% dplyr::filter(group=="TIL_high") %>% dplyr::pull(n_samples)
    n_low  <- ifelse(length(n_low)==0, 0, n_low)
    n_high <- ifelse(length(n_high)==0, 0, n_high)

    if (n_low < MIN_SAMPLES_PER_GROUP || n_high < MIN_SAMPLES_PER_GROUP) {
      return(list(
        status="skip",
        reason=paste0("too few samples per group (low=", n_low, ", high=", n_high, ")"),
        de=NULL,
        meta=sample_group_tbl
      ))
    }

    genes_keep <- intersect(as.character(genes_keep), rownames(counts_all))
    if (length(genes_keep) < 10) {
      return(list(status="skip", reason="too few genes in universe after intersecting with sce genes", de=NULL, meta=sample_group_tbl))
    }

    counts_sub <- counts_all[genes_keep, idx_ct, drop=FALSE]
    pb <- pseudobulk_by_sample(counts_sub, meta_ct[[SAMPLE_COL]])

    group_vec <- sample_group_tbl$group
    names(group_vec) <- sample_group_tbl[[SAMPLE_COL]]
    group_vec <- group_vec[colnames(pb)]

    if (any(is.na(group_vec))) {
      na_samples <- colnames(pb)[is.na(group_vec)]
      return(list(
        status="error",
        reason=paste0("group_vec has NA after aligning to pseudobulk columns. Examples: ", paste(head(na_samples, 10), collapse=", ")),
        de=NULL,
        meta=sample_group_tbl
      ))
    }

    group_vec <- factor(group_vec, levels=GROUPS)

    y <- DGEList(counts = as.matrix(pb))
    design <- model.matrix(~ group_vec)

    keep <- filterByExpr(y, design=design)
    y <- y[keep, , keep.lib.sizes=FALSE]
    y <- calcNormFactors(y)

    y <- estimateDisp(y, design, robust=TRUE)
    fit <- glmQLFit(y, design, robust=TRUE)
    qlf <- glmQLFTest(fit, coef=2)

    tt <- topTags(qlf, n=Inf)$table %>%
      tibble::rownames_to_column("gene") %>%
      dplyr::as_tibble() %>%
      dplyr::transmute(
        gene,
        logFC,
        logCPM,
        F,
        PValue,
        FDR,
        celltype = celltype_name,
        mode = mode_label,
        n_samples_total = ncol(pb),
        n_TIL_low = sum(group_vec=="TIL_low"),
        n_TIL_high = sum(group_vec=="TIL_high")
      )

    meta_out <- sample_group_tbl %>%
      dplyr::rename(sample = .data[[SAMPLE_COL]]) %>%
      dplyr::mutate(celltype = celltype_name, mode = mode_label)

    list(status="ok", reason="", de=tt, meta=meta_out)

  }, error = function(e) {
    list(status="error", reason=paste0("edgeR failed: ", conditionMessage(e)), de=NULL, meta=NULL)
  })
}

run_panel <- function(celltypes_vec, genes_keep, mode_label) {
  de_list <- list()
  meta_list <- list()
  status_rows <- list()

  for (ct in celltypes_vec) {
    res <- run_edger_for_celltype(ct, genes_keep, mode_label)
    status_rows[[ct]] <- tibble(
      mode = mode_label,
      celltype = ct,
      status = res$status,
      reason = res$reason
    )
    if (identical(res$status, "ok")) {
      de_list[[ct]] <- res$de
      meta_list[[ct]] <- res$meta
    } else {
      if (!is.null(res$meta)) meta_list[[ct]] <- res$meta
    }
  }

  list(
    status = dplyr::bind_rows(status_rows),
    de = if (length(de_list)) dplyr::bind_rows(de_list) else tibble(),
    meta = if (length(meta_list)) dplyr::bind_rows(meta_list) else tibble()
  )
}

# Run
sender_res   <- run_panel(senders_oi,   feasible_ligands,   "sender_ligand")
receiver_res <- run_panel(receivers_oi, feasible_receptors, "receiver_receptor")

# CHANGED: Use zone_prefix for output file names
out_sender_de     <- file.path(FILTER_DIR, paste0(zone_prefix, "_sender_ligand_DE_edger_by_celltype_TILhigh_vs_TILlow_activityLigs_LTMrows.csv"))
out_receiver_de   <- file.path(FILTER_DIR, paste0(zone_prefix, "_receiver_receptor_DE_edger_by_celltype_TILhigh_vs_TILlow_LRreceptorsOfActivityLigs.csv"))
out_sender_meta   <- file.path(FILTER_DIR, paste0(zone_prefix, "_sender_pseudobulk_samples_by_celltype.csv"))
out_receiver_meta <- file.path(FILTER_DIR, paste0(zone_prefix, "_receiver_pseudobulk_samples_by_celltype.csv"))
out_status        <- file.path(FILTER_DIR, paste0(zone_prefix, "_LR_DE_status_by_celltype.csv"))

readr::write_csv(sender_res$de, out_sender_de)
readr::write_csv(receiver_res$de, out_receiver_de)
readr::write_csv(sender_res$meta, out_sender_meta)
readr::write_csv(receiver_res$meta, out_receiver_meta)
readr::write_csv(dplyr::bind_rows(sender_res$status, receiver_res$status), out_status)

cat("Wrote:\n")
cat(" ", out_sender_de, "\n")
cat(" ", out_receiver_de, "\n")
cat(" ", out_sender_meta, "\n")
cat(" ", out_receiver_meta, "\n")
cat(" ", out_status, "\n\n")

cat("Celltypes with OK DE (senders):", sum(sender_res$status$status=="ok"), "/", length(senders_oi), "\n")
cat("Celltypes with OK DE (receivers):", sum(receiver_res$status$status=="ok"), "/", length(receivers_oi), "\n\n")

if (any(sender_res$status$status=="error") || any(receiver_res$status$status=="error")) {
  cat("Errors detected in some celltypes:\n")
  print(dplyr::bind_rows(sender_res$status, receiver_res$status) %>% dplyr::filter(status=="error"))
  cat("\n")
}

if (nrow(sender_res$de) > 0) {
  cat("Top 10 ligand DE rows overall (by FDR):\n")
  print(sender_res$de %>% dplyr::arrange(FDR) %>% dplyr::slice_head(n=10))
  cat("\n")
}
if (nrow(receiver_res$de) > 0) {
  cat("Top 10 receptor DE rows overall (by FDR):\n")
  print(receiver_res$de %>% dplyr::arrange(FDR) %>% dplyr::slice_head(n=10))
  cat("\n")
}

### Export results to python

In [ ]:
%%R -o ligand_activities -o sender_de -o receiver_de
# Export R results to Python (rpy2)

stopifnot(exists("ligand_activities"))
stopifnot(exists("sender_res"), exists("receiver_res"))

# Pull DE tables out of the result bundles
sender_de   <- sender_res$de
receiver_de <- receiver_res$de

# Ensure base data.frame for rpy2 conversion stability
ligand_activities <- as.data.frame(ligand_activities)
sender_de   <- as.data.frame(sender_de)
receiver_de <- as.data.frame(receiver_de)

# Minimal sanity checks for downstream Cell 5 expectations
req_act <- c("test_ligand", "direction", "aupr_corrected", "rank")
req_de  <- c("gene", "celltype", "logFC", "FDR")

missing_act <- setdiff(req_act, colnames(ligand_activities))
missing_sde <- setdiff(req_de,  colnames(sender_de))
missing_rde <- setdiff(req_de,  colnames(receiver_de))

if (length(missing_act) > 0) stop(paste0("ligand_activities missing: ", paste(missing_act, collapse=", ")))
if (length(missing_sde) > 0) stop(paste0("sender_de missing: ", paste(missing_sde, collapse=", ")))
if (length(missing_rde) > 0) stop(paste0("receiver_de missing: ", paste(missing_rde, collapse=", ")))

cat("Export objects prepared:\n")
cat("  ligand_activities rows:", nrow(ligand_activities), "\n")
cat("  sender_de rows       :", nrow(sender_de), "\n")
cat("  receiver_de rows     :", nrow(receiver_de), "\n")

### Construct axis scores

In [ ]:
# %% [markdown]
# ### Construct axis scores

# %%
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path

# === CLEAR STALE VARIABLES ===
for _v in ["all_axes", "top_axes", "sender_candidates", "ligand_sender_gating", 
           "base", "axes", "trm_high", "trm_low", "ltm_df", "act_by_dir",
           "allowed_ligands_by_dir", "lr_use_by_dir", "receptors_by_dir"]:
    if _v in globals():
        del globals()[_v]

# === VERIFY INPUTS ===
required = ["target_receiver_map_high", "target_receiver_map_low", "SENDERS", "RECEIVERS",
            "ligand_activities", "sender_de", "receiver_de", "lr_network", 
            "ligand_target_matrix", "de_df", "OUT_DIR",
            "TOP_LIGANDS_PER_DIRECTION", "LTM_TOPK_PER_TARGET", "TOP_AXES_PER_TARGET",
            "TOP_LIGANDS_PER_TARGET", "TOP_AXES_PER_LIGAND_PER_TARGET", "SENDER_TOPK_PER_LIGAND"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing: {missing}")

print("INPUT CHECK:")
print(f"  target_receiver_map_high: {len(target_receiver_map_high)} rows, {target_receiver_map_high['target'].nunique()} targets")
print(f"  target_receiver_map_low: {len(target_receiver_map_low)} rows, {target_receiver_map_low['target'].nunique()} targets")

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"

MIN_PCT_EXPRESSED_LIGAND = 0.10
MIN_SAMPLES_EXPRESSING_LIGAND = 3

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = sorted({std_ct(x) for x in SENDERS})
receivers_std = sorted({std_ct(x) for x in RECEIVERS})

TOP_LIGANDS_PER_DIRECTION = int(TOP_LIGANDS_PER_DIRECTION)
LTM_TOPK_PER_TARGET = int(LTM_TOPK_PER_TARGET)
TOP_AXES_PER_TARGET = int(TOP_AXES_PER_TARGET)
TOP_LIGANDS_PER_TARGET = int(TOP_LIGANDS_PER_TARGET)
TOP_AXES_PER_LIGAND_PER_TARGET = int(TOP_AXES_PER_LIGAND_PER_TARGET)
SENDER_TOPK_PER_LIGAND = int(SENDER_TOPK_PER_LIGAND) if SENDER_TOPK_PER_LIGAND is not None else None

LIGANDS_FROM_LTM_PER_TARGET = 10
KEEP_LIGANDS = []

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nSELECTION CONTROLS:")
print(f"  TOP_LIGANDS_PER_DIRECTION: {TOP_LIGANDS_PER_DIRECTION}")
print(f"  LTM_TOPK_PER_TARGET: {LTM_TOPK_PER_TARGET}")
print(f"  TOP_AXES_PER_TARGET: {TOP_AXES_PER_TARGET}")
print(f"  TOP_LIGANDS_PER_TARGET: {TOP_LIGANDS_PER_TARGET}")
print(f"  TOP_AXES_PER_LIGAND_PER_TARGET: {TOP_AXES_PER_LIGAND_PER_TARGET}")
print(f"  SENDER_TOPK_PER_LIGAND: {SENDER_TOPK_PER_LIGAND}")

# === COPY INPUTS ===
ligand_activities = pd.DataFrame(ligand_activities).copy()
sender_de = pd.DataFrame(sender_de).copy()
receiver_de = pd.DataFrame(receiver_de).copy()

ligand_activities["test_ligand"] = ligand_activities["test_ligand"].astype(str)
ligand_activities["direction"] = ligand_activities["direction"].astype(str)
ligand_activities["rank"] = pd.to_numeric(ligand_activities["rank"], errors="coerce")
ligand_activities["aupr_corrected"] = pd.to_numeric(ligand_activities["aupr_corrected"], errors="coerce")

for df_ in (sender_de, receiver_de):
    df_["gene"] = df_["gene"].astype(str)
    df_["celltype"] = df_["celltype"].astype(str)
    df_["logFC"] = pd.to_numeric(df_["logFC"], errors="coerce")
    df_["FDR"] = pd.to_numeric(df_["FDR"], errors="coerce")

lr_df0 = pd.DataFrame(lr_network).copy()
cols = set(lr_df0.columns.astype(str))
if {"from", "to"}.issubset(cols):
    lr_df = lr_df0.rename(columns={"from": "ligand", "to": "receptor"})[["ligand", "receptor"]].copy()
elif {"ligand", "receptor"}.issubset(cols):
    lr_df = lr_df0[["ligand", "receptor"]].copy()
else:
    raise KeyError(f"lr_network columns not recognized: {list(lr_df0.columns)}")
lr_df["ligand"] = lr_df["ligand"].astype(str)
lr_df["receptor"] = lr_df["receptor"].astype(str)
lr_df = lr_df.drop_duplicates()

# Fresh copy of TRMs
trm_high = pd.DataFrame(target_receiver_map_high).copy()
trm_low = pd.DataFrame(target_receiver_map_low).copy()

for trm in (trm_high, trm_low):
    trm["target"] = trm["target"].astype(str)
    trm["receiver_celltype"] = trm["receiver_celltype"].astype(str)
    for col in ["target_pct_expr", "target_mean_expr", "receiver_rank_score", "panel_coverage_ratio"]:
        if col in trm.columns:
            trm[col] = pd.to_numeric(trm[col], errors="coerce")

ltm_df_full = pd.DataFrame(
    ligand_target_matrix.values,
    index=pd.Index(ligand_target_matrix.index.astype(str), name="ligand"),
    columns=pd.Index(ligand_target_matrix.columns.astype(str), name="target"),
)

def get_ltm_ligands_for_targets(targets, ltm_df, k):
    targets = [t for t in targets if t in ltm_df.columns]
    ligs = set()
    for t in targets:
        w = ltm_df[t]
        w = w[w > 0].nlargest(k)
        ligs.update(w.index.astype(str))
    return ligs

# === ACTIVITY MAPS ===
act_by_dir = {}
for direction in ["TIL_high", "TIL_low"]:
    act = ligand_activities[ligand_activities["direction"] == direction].dropna(subset=["rank"])
    act = act.sort_values("rank")
    act_by_dir[direction] = {
        "top_ligs": act.nsmallest(TOP_LIGANDS_PER_DIRECTION, "rank")["test_ligand"].tolist(),
        "activity_map": act.set_index("test_ligand")["aupr_corrected"].to_dict(),
        "rank_map": act.set_index("test_ligand")["rank"].to_dict(),
    }

# === LIGAND UNIVERSE ===
allowed_ligands_by_dir = {}
for direction, trm in [("TIL_high", trm_high), ("TIL_low", trm_low)]:
    top_act = set(act_by_dir[direction]["top_ligs"])
    dir_targets = sorted(set(trm["target"]) & set(ltm_df_full.columns))
    top_ltm = get_ltm_ligands_for_targets(dir_targets, ltm_df_full, LIGANDS_FROM_LTM_PER_TARGET)
    allowed_ligands_by_dir[direction] = top_act | top_ltm | set(KEEP_LIGANDS)
    print(f"{direction} ligand universe: {len(allowed_ligands_by_dir[direction])}")

need_ligs = sorted(allowed_ligands_by_dir["TIL_high"] | allowed_ligands_by_dir["TIL_low"])
need_tgts = sorted(set(trm_high["target"]) | set(trm_low["target"]))
use_ligs = sorted(set(ltm_df_full.index) & set(need_ligs))
use_tgts = sorted(set(ltm_df_full.columns) & set(need_tgts))
ltm_df = ltm_df_full.loc[use_ligs, use_tgts].copy()

print(f"LTM subset: {len(use_ligs)} ligands x {len(use_tgts)} targets")

# === SENDER GATING ===
ct_std = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)
sample_ids = adata_sc.obs[SAMPLE_KEY].astype(str)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)
ligs_in_var = sorted(set(use_ligs) & var_set)

sender_rows = []
for direction in ["TIL_high", "TIL_low"]:
    dir_ligs = sorted(allowed_ligands_by_dir[direction] & set(ligs_in_var))
    if not dir_ligs:
        continue
    lig_idx = var_names.get_indexer(dir_ligs)

    for sct in senders_std:
        m = (grp == direction).to_numpy() & (ct_std == sct).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue

        sample_sub = sample_ids.iloc[idx].values
        n_samples = len(np.unique(sample_sub))
        Xsub = X[idx, :][:, lig_idx]
        pct = np.asarray(Xsub.getnnz(axis=0)).ravel() / idx.size
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        Xsub_dense = Xsub.toarray()

        for i, lig in enumerate(dir_ligs):
            n_samp_expr = len(set(sample_sub[Xsub_dense[:, i] > 0]))
            if pct[i] >= MIN_PCT_EXPRESSED_LIGAND and n_samp_expr >= MIN_SAMPLES_EXPRESSING_LIGAND:
                sender_rows.append({
                    "direction": direction, "ligand": lig, "sender_celltype": sct,
                    "pct_expr": float(pct[i]), "mean_expr": float(mean_expr[i]),
                    "n_samples_expressing": n_samp_expr,
                    "sample_frac": n_samp_expr / n_samples if n_samples > 0 else 0,
                })

sender_gating = pd.DataFrame(sender_rows)

# Sender specificity and rank score
for direction in ["TIL_high", "TIL_low"]:
    mask = sender_gating["direction"] == direction
    if mask.sum() == 0:
        continue
    lig_mean = sender_gating.loc[mask].groupby("ligand")["mean_expr"].transform("mean").replace(0, np.nan)
    sender_gating.loc[mask, "specificity"] = (sender_gating.loc[mask, "mean_expr"] / lig_mean).fillna(0)

sender_gating["_mr"] = sender_gating.groupby(["direction", "ligand"])["mean_expr"].rank(ascending=False)
sender_gating["_pr"] = sender_gating.groupby(["direction", "ligand"])["pct_expr"].rank(ascending=False)
sender_gating["_sfr"] = sender_gating.groupby(["direction", "ligand"])["sample_frac"].rank(ascending=False)
sender_gating["_spr"] = sender_gating.groupby(["direction", "ligand"])["specificity"].rank(ascending=False)
sender_gating["sender_rank_score"] = 0.35/sender_gating["_mr"] + 0.25/sender_gating["_pr"] + 0.20/sender_gating["_sfr"] + 0.20/sender_gating["_spr"]
sender_gating = sender_gating.drop(columns=["_mr", "_pr", "_sfr", "_spr"])

# Merge sender DE
sender_de_use = sender_de.rename(columns={"gene": "ligand", "celltype": "sender_celltype", "logFC": "ligand_logFC", "FDR": "ligand_FDR"})
sender_candidates = sender_gating.merge(sender_de_use[["ligand", "sender_celltype", "ligand_logFC", "ligand_FDR"]], on=["ligand", "sender_celltype"], how="left")

# Concordance bonus
exp_sign = np.where(sender_candidates["direction"] == "TIL_high", 1, -1)
conc = sender_candidates["ligand_logFC"].notna() & ((exp_sign * sender_candidates["ligand_logFC"]) > 0)
sender_candidates.loc[conc, "sender_rank_score"] += 0.05

sender_candidates = sender_candidates.sort_values(["direction", "ligand", "sender_rank_score"], ascending=[True, True, False])
if SENDER_TOPK_PER_LIGAND is not None:
    sender_candidates = sender_candidates.groupby(["direction", "ligand"], as_index=False).head(SENDER_TOPK_PER_LIGAND)

print(f"Sender candidates: {len(sender_candidates)} rows")

# === LR BY DIRECTION ===
lr_by_dir = {}
for d in ["TIL_high", "TIL_low"]:
    lr_by_dir[d] = lr_df[lr_df["ligand"].isin(allowed_ligands_by_dir[d])].copy()
    lr_by_dir[d]["direction"] = d

# === LTM TOP-K ===
def ltm_topk(direction, trm):
    tgts = sorted(set(trm["target"]) & set(ltm_df.columns))
    elig_ligs = sorted(set(lr_by_dir[direction]["ligand"]) & set(ltm_df.index))
    if not tgts or not elig_ligs:
        return pd.DataFrame(columns=["direction", "target", "ligand", "ltm_weight"])
    sub = ltm_df.loc[elig_ligs, tgts]
    rows = []
    for t in tgts:
        w = sub[t]
        w = w[w > 0].nlargest(LTM_TOPK_PER_TARGET)
        for lig, wt in w.items():
            rows.append({"direction": direction, "target": t, "ligand": str(lig), "ltm_weight": float(wt)})
    return pd.DataFrame(rows)

topk_high = ltm_topk("TIL_high", trm_high)
topk_low = ltm_topk("TIL_low", trm_low)

# === MERGE TRM ===
trm_cols = ["target", "receiver_celltype", "target_pct_expr", "target_mean_expr", "receiver_rank_score",
            "global_best_celltype", "panel_coverage_ratio", "coverage_flag"]
trm_cols_high = [c for c in trm_cols if c in trm_high.columns]
trm_cols_low = [c for c in trm_cols if c in trm_low.columns]

base_high = topk_high.merge(trm_high[trm_cols_high], on="target", how="inner")
base_low = topk_low.merge(trm_low[trm_cols_low], on="target", how="inner")
base = pd.concat([base_high, base_low], ignore_index=True)

print(f"Base: {len(base)} rows (high={len(base_high)}, low={len(base_low)})")

# Attach activity
base["ligand_activity"] = np.nan
base["ligand_activity_rank"] = np.nan
for d in ["TIL_high", "TIL_low"]:
    m = base["direction"] == d
    base.loc[m, "ligand_activity"] = base.loc[m, "ligand"].map(act_by_dir[d]["activity_map"])
    base.loc[m, "ligand_activity_rank"] = base.loc[m, "ligand"].map(act_by_dir[d]["rank_map"])

# Attach sender
base = base.merge(sender_candidates, on=["direction", "ligand"], how="inner")
print(f"After sender merge: {len(base)} rows")

# Attach receptor
lr_dir = pd.concat([lr_by_dir["TIL_high"], lr_by_dir["TIL_low"]], ignore_index=True)
axes = base.merge(lr_dir, on=["direction", "ligand"], how="inner")

receiver_tbl = receiver_de.rename(columns={"celltype": "receiver_celltype", "gene": "receptor", "logFC": "receptor_logFC", "FDR": "receptor_FDR"})
axes = axes.merge(receiver_tbl[["receiver_celltype", "receptor", "receptor_logFC", "receptor_FDR"]], on=["receiver_celltype", "receptor"], how="left")

# Attach target DE
de_use = de_df[["gene", "logFC", "FDR"]].copy()
de_use.columns = ["target", "target_logFC", "target_FDR"]
axes = axes.merge(de_use, on="target", how="left")

axes = axes.dropna(subset=["receptor_logFC"])
print(f"After receptor filter: {len(axes)} axes")

# === SCORE ===
def score_axes(df):
    df = df.copy()
    for d in ["TIL_high", "TIL_low"]:
        m = df["direction"] == d
        if m.sum() == 0:
            continue
        ranks = df.loc[m, "ligand_activity_rank"]
        max_r = ranks.max()
        df.loc[m, "activity_pctl"] = 1 - (ranks - 1) / max_r if max_r > 1 else 1.0
        df.loc[m, "ltm_pctl"] = df.loc[m, "ltm_weight"].rank(pct=True)
        if "receiver_rank_score" in df.columns:
            df.loc[m, "receiver_rank_pctl"] = df.loc[m, "receiver_rank_score"].rank(pct=True)
        else:
            df.loc[m, "receiver_rank_pctl"] = 0.5

    exp_sign = np.where(df["direction"] == "TIL_high", 1, -1)
    df["ligand_concordant"] = (exp_sign * df["ligand_logFC"]) > 0
    df["receptor_concordant"] = (exp_sign * df["receptor_logFC"]) > 0

    df["axis_score"] = (
        0.35 * df["activity_pctl"].fillna(0) +
        0.25 * df["ltm_pctl"].fillna(0) +
        0.15 * df["receiver_rank_pctl"].fillna(0) +
        0.10 * df["ligand_concordant"].astype(float) +
        0.15 * df["receptor_concordant"].astype(float)
    )
    return df

all_axes = score_axes(axes)
all_axes = all_axes.sort_values(["direction", "target", "axis_score"], ascending=[True, True, False])
all_axes["axis_rank"] = all_axes.groupby(["direction", "target"]).cumcount() + 1
all_axes = all_axes.drop_duplicates(subset=["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"])

# === TOP AXES SELECTION ===
lig_best = all_axes.groupby(["direction", "target", "ligand"]).agg(
    best_score=("axis_score", "max"), best_ltm=("ltm_weight", "max"), best_act=("ligand_activity", "max"), n=("axis_score", "size")
).reset_index()
lig_best = lig_best.sort_values(["direction", "target", "best_score", "best_ltm", "best_act"], ascending=[True, True, False, False, False])
lig_best["lig_rank"] = lig_best.groupby(["direction", "target"]).cumcount() + 1
keep_ligs = lig_best[lig_best["lig_rank"] <= TOP_LIGANDS_PER_TARGET][["direction", "target", "ligand"]]

sel = all_axes.merge(keep_ligs, on=["direction", "target", "ligand"], how="inner")
sel = sel.sort_values(["direction", "target", "ligand", "axis_score"], ascending=[True, True, True, False])
sel["ax_in_lig"] = sel.groupby(["direction", "target", "ligand"]).cumcount() + 1
sel = sel[sel["ax_in_lig"] <= TOP_AXES_PER_LIGAND_PER_TARGET]
sel = sel.sort_values(["direction", "target", "axis_score"], ascending=[True, True, False])
sel["ax_final"] = sel.groupby(["direction", "target"]).cumcount() + 1
top_axes = sel[sel["ax_final"] <= TOP_AXES_PER_TARGET].copy()
top_axes["axis_rank"] = top_axes.groupby(["direction", "target"]).cumcount() + 1

# === SUMMARY ===
print(f"\nAXES BUILT:")
print(f"  all_axes: {len(all_axes)} rows, {all_axes['target'].nunique()} targets")
print(f"  top_axes: {len(top_axes)} rows, {top_axes['target'].nunique()} targets")

for d in ["TIL_high", "TIL_low"]:
    sub = all_axes[all_axes["direction"] == d]
    print(f"\n  {d}: {len(sub)} axes, {sub['target'].nunique()} targets, {sub['ligand'].nunique()} ligands")

# === SAVE ===
all_axes.to_csv(OUT_DIR / "all_axes.csv", index=False)
top_axes.to_csv(OUT_DIR / "top_axes.csv", index=False)
print(f"\n✓ Saved to {OUT_DIR}")

### Summarize axis scores for plot selection

In [ ]:
# %% [markdown]
# ### Summarize axis scores for plot selection

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib

# === CLEAR STALE VARIABLES ===
for _v in ["df", "df_all", "df_top", "axes_catalog", "axes_shortlist", "short", "short2"]:
    if _v in globals():
        del globals()[_v]

# === VERIFY INPUTS ===
if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR not found.")
if "top_axes" not in globals():
    raise NameError("top_axes not found. Run Cell 5 first.")

OUT_DIR = Path(OUT_DIR)

print("INPUT CHECK:")
print(f"  top_axes: {len(top_axes)} rows, {top_axes['target'].nunique()} targets")
if "all_axes" in globals():
    print(f"  all_axes: {len(all_axes)} rows (for reference, not used in catalog)")

# Use top_axes (filtered) for catalog, NOT all_axes
df = pd.DataFrame(top_axes).copy()

req_cols = {"direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
            "target_logFC", "target_FDR", "ligand_activity", "ltm_weight",
            "ligand_concordant", "receptor_concordant", "axis_score", "axis_rank"}
missing = req_cols - set(df.columns)
if missing:
    raise KeyError(f"top_axes missing columns: {sorted(missing)}")

# Normalize dtypes
for c in ["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"]:
    df[c] = df[c].astype(str)
for c in ["axis_score", "axis_rank", "target_logFC", "target_FDR", "ligand_activity", "ltm_weight"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["ligand_concordant"] = df["ligand_concordant"].astype(bool)
df["receptor_concordant"] = df["receptor_concordant"].astype(bool)

if "receiver_rank_score" in df.columns:
    df["receiver_rank_score"] = pd.to_numeric(df["receiver_rank_score"], errors="coerce")
if "panel_coverage_ratio" in df.columns:
    df["panel_coverage_ratio"] = pd.to_numeric(df["panel_coverage_ratio"], errors="coerce")

# === FILTER 1: positive activity ===
n_before = len(df)
df = df[df["ligand_activity"] > 0].copy()
print(f"\nFilter: ligand_activity > 0: {n_before} -> {len(df)} axes")

# === ABUNDANCE ANNOTATIONS ===
def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

abundance_available = False
if "EFS_ROOT" in globals() and "ZONE" in globals():
    ab_path = Path(EFS_ROOT) / "results/visium/til_zones/celltype_zone_specificity_curated/celltype_zone_specificity_level4_curated.csv"

    if ab_path.exists():
        ab = pd.read_csv(ab_path)
        ab["group"] = ab["group"].astype(str)
        ab["zone"] = ab["zone"].astype(str)
        ab["celltype_std"] = ab["celltype_level4_curated"].astype(str).map(std_ct)
        ab_zone = ab[ab["zone"] == str(ZONE)].copy()

        delta_tbl = ab_zone.pivot_table(index="celltype_std", columns="group", values="mean_zone", aggfunc="first").reset_index()
        if "TIL_high" in delta_tbl.columns and "TIL_low" in delta_tbl.columns:
            delta_tbl["mean_zone_high_minus_low"] = pd.to_numeric(delta_tbl["TIL_high"], errors="coerce") - pd.to_numeric(delta_tbl["TIL_low"], errors="coerce")
            delta_tbl = delta_tbl[["celltype_std", "mean_zone_high_minus_low"]]
        else:
            delta_tbl = pd.DataFrame({"celltype_std": ab_zone["celltype_std"].unique(), "mean_zone_high_minus_low": np.nan})

        df["sender_celltype_std"] = df["sender_celltype"].map(std_ct)
        df["receiver_celltype_std"] = df["receiver_celltype"].map(std_ct)

        # Sender abundance
        sender_ab = ab_zone.rename(columns={"celltype_std": "sender_celltype_std", "mean_zone": "sender_mean_zone"})
        sender_ab = sender_ab[["group", "sender_celltype_std", "sender_mean_zone"]].copy()
        df = df.merge(sender_ab, left_on=["direction", "sender_celltype_std"], right_on=["group", "sender_celltype_std"], how="left")
        df = df.drop(columns=["group"], errors="ignore")

        # Receiver abundance
        receiver_ab = ab_zone.rename(columns={"celltype_std": "receiver_celltype_std", "mean_zone": "receiver_mean_zone"})
        receiver_ab = receiver_ab[["group", "receiver_celltype_std", "receiver_mean_zone"]].copy()
        df = df.merge(receiver_ab, left_on=["direction", "receiver_celltype_std"], right_on=["group", "receiver_celltype_std"], how="left")
        df = df.drop(columns=["group"], errors="ignore")

        # Delta
        delta_sender = delta_tbl.rename(columns={"celltype_std": "sender_celltype_std", "mean_zone_high_minus_low": "sender_mean_zone_high_minus_low"})
        delta_receiver = delta_tbl.rename(columns={"celltype_std": "receiver_celltype_std", "mean_zone_high_minus_low": "receiver_mean_zone_high_minus_low"})
        df = df.merge(delta_sender, on="sender_celltype_std", how="left")
        df = df.merge(delta_receiver, on="receiver_celltype_std", how="left")

        print(f"Abundance annotations loaded from: {ab_path}")
        abundance_available = True
    else:
        print(f"Abundance CSV not found: {ab_path}")
else:
    print("EFS_ROOT or ZONE not set, skipping abundance annotations")

# === FILTER 2: concordance/abundance ===
ABUNDANCE_THRESHOLD = 0.05
n_before = len(df)

if abundance_available:
    high_mask = df["direction"] == "TIL_high"
    low_mask = df["direction"] == "TIL_low"
    high_keep = df["ligand_concordant"] | (df["sender_mean_zone_high_minus_low"] > ABUNDANCE_THRESHOLD)
    low_keep = df["ligand_concordant"] | (df["sender_mean_zone_high_minus_low"] < -ABUNDANCE_THRESHOLD)
    df = df[(high_mask & high_keep) | (low_mask & low_keep)].copy()
else:
    df = df[df["ligand_concordant"]].copy()

print(f"Filter: concordance/abundance: {n_before} -> {len(df)} axes")

# === AXIS ID ===
key_cols = ["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"]
df = df.drop_duplicates(subset=key_cols).copy()

def axis_hash_id(row):
    s = "|".join(str(row[c]) for c in key_cols)
    h = hashlib.md5(s.encode()).hexdigest()[:6].upper()
    return ("H" if row["direction"] == "TIL_high" else "L") + h

df["axis_id"] = df.apply(axis_hash_id, axis=1)

# === RANK SCORE ===
df["de_strength"] = -np.log10(df["target_FDR"].clip(lower=1e-300))
df["rank_score"] = df["axis_score"] * df["de_strength"] * df["target_logFC"].abs()

# === VERIFY AXES PER TARGET ===
axes_per_target = df.groupby(["direction", "target"]).size()
print(f"\nAxes per target: min={axes_per_target.min()}, max={axes_per_target.max()}, mean={axes_per_target.mean():.1f}")

# === CATALOG COLUMNS ===
catalog_cols = ["axis_id", "direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
                "axis_score", "axis_rank", "rank_score", "target_logFC", "target_FDR",
                "ligand_activity", "ltm_weight", "ligand_concordant", "receptor_concordant"]

# Add optional columns if present
for c in ["receiver_rank_score", "global_best_celltype", "global_best_n_cells", "panel_coverage_ratio", "coverage_flag",
          "sender_mean_zone", "receiver_mean_zone", "sender_mean_zone_high_minus_low", "receiver_mean_zone_high_minus_low"]:
    if c in df.columns:
        catalog_cols.append(c)

axes_catalog = df[[c for c in catalog_cols if c in df.columns]].copy()
axes_catalog.to_csv(OUT_DIR / "axes_catalog.csv", index=False)

# === SHORTLIST (further filtered subset) ===
AXIS_SCORE_MIN = 0.5
MAX_AXIS_RANK_PER_TARGET = 3
TARGET_FDR_MAX = 0.05
TARGET_LOGFC_MIN = 0.25
AXES_PER_TARGET_DE_BAL = 2

short = df[(df["axis_score"] > AXIS_SCORE_MIN) & (df["axis_rank"] <= MAX_AXIS_RANK_PER_TARGET)].copy()
short = short.dropna(subset=["target_FDR", "target_logFC", "rank_score"])
short = short[short["target_FDR"] <= TARGET_FDR_MAX]

high = short[(short["direction"] == "TIL_high") & (short["target_logFC"] >= TARGET_LOGFC_MIN)]
low = short[(short["direction"] == "TIL_low") & (short["target_logFC"] <= -TARGET_LOGFC_MIN)]
short = pd.concat([high, low])
short = short.sort_values(["direction", "target", "rank_score"], ascending=[True, True, False])
axes_shortlist = short.groupby(["direction", "target"], as_index=False).head(AXES_PER_TARGET_DE_BAL)
axes_shortlist = axes_shortlist[[c for c in catalog_cols if c in axes_shortlist.columns]]
axes_shortlist.to_csv(OUT_DIR / "axes_shortlist.csv", index=False)

# === SUMMARY ===
print(f"\nCATALOG SUMMARY:")
for d in ["TIL_high", "TIL_low"]:
    sub = df[df["direction"] == d]
    print(f"  {d}: {len(sub)} axes, {sub['target'].nunique()} targets, {sub['ligand'].nunique()} ligands")

if "coverage_flag" in df.columns:
    print(f"\nPANEL COVERAGE:")
    for d in ["TIL_high", "TIL_low"]:
        sub = df[df["direction"] == d]
        if sub.empty:
            continue
        n_ok = (sub["coverage_flag"] == "OK").sum()
        n_low = (sub["coverage_flag"] == "LOW_COVERAGE").sum()
        print(f"  {d}: OK={n_ok}, LOW_COVERAGE={n_low}")

print(f"\nSHORTLIST: {len(axes_shortlist)} axes, {axes_shortlist['target'].nunique()} targets")

print(f"\nWROTE:")
print(f"  {OUT_DIR / 'axes_catalog.csv'}")
print(f"  {OUT_DIR / 'axes_shortlist.csv'}")

# === DISPLAY ===
display_cols = ["axis_id", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
                "axis_score", "rank_score", "target_logFC"]
if "coverage_flag" in axes_catalog.columns:
    display_cols.append("coverage_flag")
display_cols = [c for c in display_cols if c in axes_catalog.columns]

print(f"\nTOP 15 TIL_high:")
display(axes_catalog[axes_catalog["direction"] == "TIL_high"].nlargest(15, "rank_score")[display_cols])

print(f"\nTOP 15 TIL_low:")
display(axes_catalog[axes_catalog["direction"] == "TIL_low"].nlargest(15, "rank_score")[display_cols])

# === WARNINGS ===
if "coverage_flag" in axes_shortlist.columns:
    low_cov = axes_shortlist[axes_shortlist["coverage_flag"] == "LOW_COVERAGE"]
    if len(low_cov) > 0:
        print(f"\n⚠️ WARNING: {len(low_cov)} shortlist axes have LOW_COVERAGE:")
        for _, r in low_cov.iterrows():
            gb_ct = r.get('global_best_celltype', 'N/A')
            gb_n = r.get('global_best_n_cells', 'N/A')
            print(f"  {r['axis_id']}: {r['target']} (global_best={gb_ct}, n={gb_n})")

print("\n✓ Done")

### Axes for TIL+ Circos plot

In [ ]:
# Cell 7: Enter selected axis IDs (uses current run's catalog on disk)

import pandas as pd
from pathlib import Path

if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR not found.")

OUT_DIR = Path(OUT_DIR)
catalog_path = OUT_DIR / "axes_catalog.csv"
if not catalog_path.exists():
    raise FileNotFoundError(f"Missing catalog: {catalog_path}. Run Cell 6 first.")

axes_catalog = pd.read_csv(catalog_path)

if "axis_id" not in axes_catalog.columns:
    raise KeyError(f"axes_catalog.csv missing axis_id. Columns: {list(axes_catalog.columns)}")

# Paste your chosen IDs here (from this run's axes_catalog.csv)
SELECT_TIL_HIGH = [
    "H9EA1CD",
"HA67944",
"H3198DC",
"H522103",
"HD6FF39",
"HB6D080",
"H284E2B",
"H82D1AE",
"HB28C48",
"HA4B3CA",
"H3FDA84",
"H4B4B64",
"HEC305E",
"H9E0E8A",
"H260FE0",
"H6B6D82",
"HD94B30",
"H48199E",
"HF8FE2A",
"H996E52",
"H56FE9B"
]

SELECT_TIL_LOW = [
    "L8CFF7D",
"L80BFB0",
"L541B52",
"L62A08E",
"LA902FE",
"L84A624",
"L56288E",
"LDAFCBF",
"L6BF2B0",
"L185E64",
"L08D48E",
"L545826",
"LB8C606",
"L22A8D9",
"L61489E",
"L47B3FA",
"LF2BC52",
"LDEB697",
"L139E93",
"L174C55",
"LCE8586",
"LD39CD2",
"L83008E",
"LCD6359",
"L570219"
]

# Normalize and dedupe while preserving input order
raw_ids = [str(x).strip() for x in (SELECT_TIL_HIGH + SELECT_TIL_LOW) if str(x).strip()]
if len(raw_ids) == 0:
    raise ValueError("No axis IDs provided. Fill SELECT_TIL_HIGH and/or SELECT_TIL_LOW.")

seen = set()
selected_ids = []
for x in raw_ids:
    if x not in seen:
        selected_ids.append(x)
        seen.add(x)

catalog_ids = axes_catalog["axis_id"].astype(str)
missing = sorted(set(selected_ids) - set(catalog_ids))
if missing:
    raise ValueError(
        "Some axis IDs were not found in the current run's catalog.\n"
        "This usually means you copied IDs from a different run.\n\n"
        "Missing:\n" + "\n".join(missing)
    )

sel = axes_catalog[catalog_ids.isin(selected_ids)].copy()

# Preserve the order you typed them
order = {aid: i for i, aid in enumerate(selected_ids)}
sel["input_order"] = sel["axis_id"].astype(str).map(order)
sel = sel.sort_values(["input_order"]).drop(columns=["input_order"])

# Sanity: ensure selected HIGH/LOW match your lists
if "direction" in sel.columns:
    bad_high = sel[(sel["axis_id"].isin(SELECT_TIL_HIGH)) & (sel["direction"] != "TIL_high")]
    bad_low  = sel[(sel["axis_id"].isin(SELECT_TIL_LOW))  & (sel["direction"] != "TIL_low")]
    if len(bad_high) > 0 or len(bad_low) > 0:
        msg = []
        if len(bad_high) > 0:
            msg.append("Some IDs in SELECT_TIL_HIGH are not TIL_high:\n" + "\n".join(bad_high["axis_id"].astype(str).tolist()))
        if len(bad_low) > 0:
            msg.append("Some IDs in SELECT_TIL_LOW are not TIL_low:\n" + "\n".join(bad_low["axis_id"].astype(str).tolist()))
        raise ValueError("\n\n".join(msg))

out_path = OUT_DIR / "axes_selected.csv"
sel.to_csv(out_path, index=False)

print("Selected axes:", len(sel))
if "direction" in sel.columns:
    print("  HIGH:", int((sel["direction"] == "TIL_high").sum()))
    print("  LOW :", int((sel["direction"] == "TIL_low").sum()))
print("Wrote:", str(out_path))

show_cols = [c for c in [
    "axis_id","direction","target","ligand","receptor",
    "sender_celltype","receiver_celltype",
    "axis_score","rank_score","target_logFC","target_FDR",
    "ligand_activity","ltm_weight",
    "ligand_concordant","receptor_concordant",
] if c in sel.columns]

display(sel[show_cols])

# Border Zone 2 (BZ2) NicheNet

### Define BZ2 senders and receivers

In [ ]:
def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

TILPLUS_GROUP = "TIL_high"

SENDERS = [
    "Malignant_MES-hypoxia",
    "Malignant_AC-like_IFN_response",
    "Malignant_AC1",
    "Malignant_OPC2",
    "Malignant_NPC2",
    "Astrocyte_inflammatory",
    "Astrocyte_reactive",
    "Astrocyte_intermediate_reactive",
    "Macrophage_LAM",
    "Microglia_TREM2_pos",
    "Microglia_homeostatic"
]

RECEIVERS = [
    "Perivascular_fibroblast_type-I",
    "Perivascular_fibroblast_type-II",
    "Perivascular_fibroblast_type-III",
    "Pericyte_fibro-like",
    "Endothelial_capillary",
    "Endothelial",
    "Macrophage_LAM",
    "Microglia_TREM2_pos",
    "Microglia_homeostatic"
]

SENDERS_STD = [std_ct(x) for x in SENDERS]
RECEIVERS_STD = [std_ct(x) for x in RECEIVERS]

tilplus_panel = {
    "group": TILPLUS_GROUP,
    "senders": SENDERS_STD,
    "receivers": RECEIVERS_STD,
}

print("TIL+ panel defined:")
print("  group:", tilplus_panel["group"])
print("  senders:", len(tilplus_panel["senders"]))
print("  receivers:", len(tilplus_panel["receivers"]))

### GSEA of HIGH and LOW BZ2 programs

In [ ]:
# %% [markdown]
# ### ULM pathway enrichment scoring

# %%
import numpy as np
import pandas as pd
from pathlib import Path
import requests
import decoupler as dc

# User config
ZONE = "BZ2"  # Options: "TIL+", "BZ1", "BZ2"

# Paths
cache_dir = Path.home() / ".cache" / "msigdb"
cache_dir.mkdir(parents=True, exist_ok=True)

EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
DE_DIR = EFS_ROOT / "results/visium/til_zones/deg_pseudobulk_edger"
OUT_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_filtering"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Zone-specific DE file
visium_de_file = DE_DIR / f"DE_edger_zone_{ZONE}_TIL_high_vs_TIL_low.csv"

print(f"ZONE: {ZONE}")
print(f"DE file: {visium_de_file.name}")

if not visium_de_file.exists():
    raise FileNotFoundError(f"Missing: {visium_de_file}")

print("Input file found")

# MSigDB GMT download and parse
urls = {
    "hallmark": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.symbols.gmt",
    "kegg":     "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.kegg_medicus.v2023.2.Hs.symbols.gmt",
    "biocarta": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/c2.cp.biocarta.v2023.2.Hs.symbols.gmt",
}
paths = {k: cache_dir / Path(v).name for k, v in urls.items()}

def download_text(url: str, out_path: Path, timeout: int = 120) -> None:
    if out_path.exists() and out_path.stat().st_size > 0:
        return
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    out_path.write_text(r.text)

def parse_gmt(gmt_path: Path) -> dict:
    gene_sets = {}
    with gmt_path.open("r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                gene_sets[parts[0]] = [g for g in parts[2:] if g]
    return gene_sets

for k, url in urls.items():
    download_text(url, paths[k])

hallmark_sets = parse_gmt(paths["hallmark"])
kegg_sets = parse_gmt(paths["kegg"])
biocarta_sets = parse_gmt(paths["biocarta"])

def gmt_to_net(gene_sets: dict) -> pd.DataFrame:
    rows = [(gs, g, 1.0) for gs, genes in gene_sets.items() for g in genes]
    return pd.DataFrame(rows, columns=["source", "target", "weight"])

net = pd.concat(
    [gmt_to_net(hallmark_sets), gmt_to_net(kegg_sets), gmt_to_net(biocarta_sets)],
    ignore_index=True
)

# Load full DE results (all genes)
visium_de = pd.read_csv(visium_de_file)
for c in ["gene", "logFC"]:
    if c not in visium_de.columns:
        raise ValueError(f"Missing column '{c}' in {visium_de_file.name}")

visium_de["gene"] = visium_de["gene"].astype(str)
visium_de["logFC"] = pd.to_numeric(visium_de["logFC"], errors="coerce")
visium_de = visium_de.dropna(subset=["logFC"])
visium_de = visium_de.drop_duplicates(subset=["gene"], keep="first")

print(f"Genes in DE results: {len(visium_de)}")

# BH FDR correction
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    p0 = p[ok]
    n = p0.size
    order = np.argsort(p0)
    ranked = p0[order]
    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty_like(p0)
    out[order] = np.clip(adj, 0, 1)
    q[ok] = out
    return q

# Run ULM on all genes
# Positive logFC = upregulated in TIL_high
# Negative logFC = upregulated in TIL_low
def run_ulm_all_genes(de_df: pd.DataFrame, label: str) -> pd.DataFrame:
    w = de_df.set_index("gene")["logFC"].astype(float)
    mat = w.to_frame().T
    mat.index = [label]

    est, pvs = dc.run_ulm(
        mat=mat,
        net=net,
        source="source",
        target="target",
        weight="weight",
        min_n=5,
        verbose=False,
    )

    res = pd.DataFrame({
        "gene_set": est.columns.astype(str),
        "score": est.iloc[0].to_numpy(dtype=float),
        "pval": pvs.iloc[0].to_numpy(dtype=float),
    })
    res["fdr"] = bh_fdr(res["pval"].to_numpy())
    res["collection"] = np.select(
        [
            res["gene_set"].str.startswith("HALLMARK_"),
            res["gene_set"].str.startswith("KEGG_"),
            res["gene_set"].str.startswith("BIOCARTA_"),
        ],
        ["Hallmark", "KEGG", "BioCarta"],
        default="Other",
    )
    return res

ulm_results = run_ulm_all_genes(visium_de, label=f"{ZONE}_TIL_high_vs_low")

# Positive score = enriched in TIL_high direction
# Negative score = enriched in TIL_low direction
ulm_high = ulm_results[ulm_results["score"] > 0].sort_values("score", ascending=False).copy()
ulm_low = ulm_results[ulm_results["score"] < 0].sort_values("score", ascending=True).copy()

# Save outputs
zone_file_prefix = ZONE.replace("+", "plus").replace("-", "minus")
out_all = OUT_DIR / f"{zone_file_prefix}_ULM_all_pathways.csv"
out_high = OUT_DIR / f"{zone_file_prefix}_ULM_TIL_high_enriched.csv"
out_low = OUT_DIR / f"{zone_file_prefix}_ULM_TIL_low_enriched.csv"

ulm_results.sort_values("score", ascending=False).to_csv(out_all, index=False)
ulm_high.to_csv(out_high, index=False)
ulm_low.to_csv(out_low, index=False)

print(f"\nGene sets: Hallmark={len(hallmark_sets)}, KEGG={len(kegg_sets)}, BioCarta={len(biocarta_sets)}")
print(f"Network: {net['source'].nunique()} sets, {net['target'].nunique()} genes")
print(f"\nWrote:")
print(f"  {out_all}")
print(f"  {out_high}")
print(f"  {out_low}")

# Display top results
def show_top(res: pd.DataFrame, title: str, n: int = 20) -> None:
    if res.empty:
        print(f"\n{title}: no results")
        return
    top = res.head(n).copy()
    top["score"] = top["score"].round(3)
    top["pval"] = top["pval"].map(lambda x: f"{x:.2e}" if np.isfinite(x) else "nan")
    top["fdr"] = top["fdr"].map(lambda x: f"{x:.2e}" if np.isfinite(x) else "nan")
    print(f"\n{title}")
    print(top[["gene_set", "collection", "score", "pval", "fdr"]].to_string(index=False))

show_top(ulm_high, f"Top pathways enriched in TIL_high ({ZONE})")
show_top(ulm_low, f"Top pathways enriched in TIL_low ({ZONE})")

### Define pathway filters

In [ ]:
selected_pathways = [
    "HALLMARK_ANGIOGENESIS",
    "HALLMARK_COMPLEMENT",
    "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
    "HALLMARK_HYPOXIA",
    "HALLMARK_APOPTOSIS",
    "HALLMARK_UV_RESPONSE_DN",
    "HALLMARK_IL6_JAK_STAT3_SIGNALING",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE",
    "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "HALLMARK_TGF_BETA_SIGNALING",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "HALLMARK_IL2_STAT5_SIGNALING",
    "HALLMARK_GLYCOLYSIS",
    "HALLMARK_MTORC1_SIGNALING",
    "HALLMARK_MYOGENESIS",
    "HALLMARK_PROTEIN_SECRETION",
    "KEGG_MEDICUS_REFERENCE_WNT_SIGNALING_MODULATION_WNT_ACYLATION",
    "HALLMARK_COAGULATION"
]

ecm_genes_brain = [
    "COL1A1","COL1A2","COL3A1","COL4A1","COL4A2","COL5A1","COL6A1","COL6A2","COL6A3","COL7A1",
    "COL8A1","COL10A1","COL11A1","COL12A1","COL14A1","COL15A1","COL16A1","FN1","FBN1","FBN2",
    "TNC","SPARC","POSTN","LAMA1","LAMA2","LAMA3","LAMB1","LAMB2","LAMC1","THBS1","THBS2","THBS4",
    "VTN","NID1","NID2","VCAN","DCN","BGN","LUM","HSPG2","AGRN","GPC1","GPC3","MMP1","MMP2","MMP9",
    "MMP11","MMP14","ADAMTS1","ADAMTS4","ADAMTS5","TIMP1","TIMP2","LOX","LOXL1","SERPINE1","PLAU",
    "PLAT","CSPG4","BCAN","NCAN","HAPLN1","HAPLN2","PRELP","FMOD","COL18A1","LOXL2","LOXL3","HPSE",
    "SDC1","GPC2","CSPG5","PTPRZ1","COL5A2","COL5A3","LAMA4","LAMA5","ACAN","TNR","HAPLN3","HAPLN4",
    "FBLN1","FBLN2","FBLN5","SDC4","SPP1","IGFBP2","IGFBP5","SRGN","MMP3","MMP7","MMP12","ADAMTS8",
    "ADAM8","TIMP3","LOXL4",
]

# Combine all pathway sets into one lookup
all_pathway_sets = {}
if "hallmark_sets" in globals():
    all_pathway_sets.update(hallmark_sets)
if "kegg_sets" in globals():
    all_pathway_sets.update(kegg_sets)
if "biocarta_sets" in globals():
    all_pathway_sets.update(biocarta_sets)

if not all_pathway_sets:
    raise RuntimeError("No pathway sets found. Run ULM scoring cell first.")

print(f"Available pathway sets: {len(all_pathway_sets)}")

# Validate selected pathways exist
missing = [p for p in selected_pathways if p not in all_pathway_sets]
if missing:
    print("\nMissing pathways:")
    for m in missing:
        print(f"  {m}")
    raise RuntimeError(f"{len(missing)} selected pathways not found in available sets")

# Build gene filter
pathway_genes = set(ecm_genes_brain)
for pathway in selected_pathways:
    pathway_genes.update(all_pathway_sets[pathway])

PATHWAY_FILTER = sorted(pathway_genes)

print(f"\nSelected pathways: {len(selected_pathways)}")
print(f"Custom gene lists: 1 (ecm_genes_brain: {len(ecm_genes_brain)} genes)")
print(f"PATHWAY_FILTER: {len(PATHWAY_FILTER)} unique genes")

### Config

In [ ]:
# Cell 1: Configuration

from pathlib import Path

ZONE = "BZ2"

# Set this to False for unbiased targets
USE_PATHWAY_FILTER = True

# If USE_PATHWAY_FILTER is True, PATHWAY_FILTER must already exist as an iterable of gene symbols
if USE_PATHWAY_FILTER:
    if "PATHWAY_FILTER" not in globals():
        raise NameError("USE_PATHWAY_FILTER=True but PATHWAY_FILTER is not defined in the notebook.")
    PATHWAY_FILTER = set(map(str, PATHWAY_FILTER))
else:
    PATHWAY_FILTER = None

FDR_CUTOFF = 0.05
LOGFC_CUTOFF = 0.25

# Axis construction and selection controls
TOP_LIGANDS_PER_DIRECTION = 50
LTM_TOPK_PER_TARGET = 7
TOP_AXES_PER_TARGET = 7
SENDER_TOPK_PER_LIGAND = 7  # set to None to keep all passing senders

# Ligand diversity controls within a target (used in Cell 5)
TOP_LIGANDS_PER_TARGET = 7
TOP_AXES_PER_LIGAND_PER_TARGET = 7

# Catalog truncation control (used in Cell 6)
CATALOG_MAX_AXES_PER_TARGET = 7  # set to None for no truncation, or increase to browse more

# Expression threshold for target->receiver mapping
FRACTION_CUTOFF = 0.05

# IMPORTANT: keep this consistent with other cells (they use /mnt/...)
EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
if not EFS_ROOT.exists():
    raise FileNotFoundError(f"EFS_ROOT does not exist: {EFS_ROOT}")

GATING_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_gating"
OUT_DIR = EFS_ROOT / "results/visium/til_zones/nichenet_analysis" / ZONE.replace("+", "plus").replace("-", "minus")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Zone: {ZONE}")
print(f"Pathway filter: {'None (unbiased)' if PATHWAY_FILTER is None else f'{len(PATHWAY_FILTER)} genes'}")
print(f"FDR_CUTOFF: {FDR_CUTOFF}")
print(f"LOGFC_CUTOFF: {LOGFC_CUTOFF}")
print(f"FRACTION_CUTOFF: {FRACTION_CUTOFF}")
print(f"Output: {OUT_DIR}")

print(f"TOP_LIGANDS_PER_DIRECTION: {TOP_LIGANDS_PER_DIRECTION}")
print(f"LTM_TOPK_PER_TARGET: {LTM_TOPK_PER_TARGET}")
print(f"TOP_AXES_PER_TARGET: {TOP_AXES_PER_TARGET}")
print(f"SENDER_TOPK_PER_LIGAND: {SENDER_TOPK_PER_LIGAND}")

print(f"TOP_LIGANDS_PER_TARGET: {TOP_LIGANDS_PER_TARGET}")
print(f"TOP_AXES_PER_LIGAND_PER_TARGET: {TOP_AXES_PER_LIGAND_PER_TARGET}")
print(f"CATALOG_MAX_AXES_PER_TARGET: {CATALOG_MAX_AXES_PER_TARGET}")

### Zone background gene universe

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Defines these as sorted Python lists of gene symbols for downstream %%R cells:
#   background_high, background_low
#   ligands_high, ligands_low
#   receptors_high, receptors_low
#
# Also keeps a structured dict for inspection:
#   universes_panel

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
MIN_PCT_EXPRESSED = 0.10

required = ["adata_sc", "nichenet_priors", "SENDERS", "RECEIVERS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required variables: {missing}")

if LAYER not in adata_sc.layers:
    raise KeyError(
        f"adata_sc.layers['{LAYER}'] not found. Available layers: {list(adata_sc.layers.keys())}"
    )
for k in [CT_KEY, GROUP_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(
            f"adata_sc.obs['{k}'] not found. Available obs columns (first 60): {list(adata_sc.obs.columns)[:60]}"
        )

for k in ["ltm_targets", "ltm_ligands", "lr_ligands", "lr_receptors"]:
    if k not in nichenet_priors:
        raise KeyError(f"nichenet_priors['{k}'] not found. Available keys: {list(nichenet_priors.keys())}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = set(std_ct(x) for x in SENDERS)
receivers_std = set(std_ct(x) for x in RECEIVERS)

ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

expected_groups = {"TIL_high", "TIL_low"}
groups_present = set(grp.unique())
if not expected_groups.issubset(groups_present):
    raise ValueError(f"Expected {expected_groups} in {GROUP_KEY}, found: {sorted(groups_present)}")

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

ltm_targets = set(map(str, nichenet_priors["ltm_targets"]))
ltm_ligands = set(map(str, nichenet_priors["ltm_ligands"]))
lr_ligands = set(map(str, nichenet_priors["lr_ligands"]))
lr_receptors = set(map(str, nichenet_priors["lr_receptors"]))

candidate_background = ltm_targets
candidate_ligands = ltm_ligands.intersection(lr_ligands)
candidate_receptors = lr_receptors

def expressed_genes(mask_cells: np.ndarray, candidate_genes: set, min_pct: float) -> set:
    idx = np.flatnonzero(mask_cells)
    if idx.size == 0:
        return set()

    genes_present = sorted(candidate_genes.intersection(var_set))
    if not genes_present:
        return set()

    gene_idx = var_names.get_indexer(genes_present)
    Xsub = X[idx, :][:, gene_idx]

    nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
    pct = nnz / float(idx.size)

    keep = pct >= float(min_pct)
    return set(np.asarray(genes_present, dtype=object)[keep])

universes_panel = {}

for g in ["TIL_high", "TIL_low"]:
    sender_mask = (grp == g).to_numpy() & ct.isin(senders_std).to_numpy()
    receiver_mask = (grp == g).to_numpy() & ct.isin(receivers_std).to_numpy()

    bg = expressed_genes(receiver_mask, candidate_background, MIN_PCT_EXPRESSED)
    lig = expressed_genes(sender_mask, candidate_ligands, MIN_PCT_EXPRESSED)
    rec = expressed_genes(receiver_mask, candidate_receptors, MIN_PCT_EXPRESSED)

    universes_panel[g] = {
        "background": bg,
        "ligands": lig,
        "receptors": rec,
        "n_sender_cells": int(sender_mask.sum()),
        "n_receiver_cells": int(receiver_mask.sum()),
    }

background_high = sorted(map(str, universes_panel["TIL_high"]["background"]))
background_low = sorted(map(str, universes_panel["TIL_low"]["background"]))
ligands_high = sorted(map(str, universes_panel["TIL_high"]["ligands"]))
ligands_low = sorted(map(str, universes_panel["TIL_low"]["ligands"]))
receptors_high = sorted(map(str, universes_panel["TIL_high"]["receptors"]))
receptors_low = sorted(map(str, universes_panel["TIL_low"]["receptors"]))

summary_df = pd.DataFrame(
    [
        {
            "group": g,
            "n_sender_cells": universes_panel[g]["n_sender_cells"],
            "n_receiver_cells": universes_panel[g]["n_receiver_cells"],
            "n_background": len(universes_panel[g]["background"]),
            "n_ligands": len(universes_panel[g]["ligands"]),
            "n_receptors": len(universes_panel[g]["receptors"]),
        }
        for g in ["TIL_high", "TIL_low"]
    ]
)

print(summary_df.to_string(index=False))

print("\nUniverse integrity checks:")
print("  candidate_background (ltm_targets):", len(candidate_background))
print("  candidate_ligands (ltm_ligands ∩ lr_ligands):", len(candidate_ligands))
print("  candidate_receptors (lr_receptors):", len(candidate_receptors))
print("  var genes:", len(var_names))
print("  overlap candidate_ligands with var:", len(candidate_ligands.intersection(var_set)))
print("  overlap candidate_receptors with var:", len(candidate_receptors.intersection(var_set)))

if len(background_high) == 0 or len(background_low) == 0:
    raise ValueError("Background is empty for at least one group. Check receiver panel, group labels, and MIN_PCT_EXPRESSED.")
if len(ligands_high) == 0 or len(ligands_low) == 0:
    raise ValueError("Ligands are empty for at least one group. Check sender panel, group labels, and MIN_PCT_EXPRESSED.")
if len(receptors_high) == 0 or len(receptors_low) == 0:
    raise ValueError("Receptors are empty for at least one group. Check receiver panel, group labels, and MIN_PCT_EXPRESSED.")

print("\nExported lists ready for R:")
print("  background_high:", len(background_high), "background_low:", len(background_low))
print("  ligands_high   :", len(ligands_high), "ligands_low   :", len(ligands_low))
print("  receptors_high :", len(receptors_high), "receptors_low :", len(receptors_low))


### Target gene universe

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Builds canonical objects for this session:
#   targets_high, targets_low
#   universes (dict with targets/ligands/background/receptors per direction)
# Keeps de_df for downstream axis construction (Cell 5 expects de_df).

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
ZONE_KEY = ZONE  

MIN_PCT_EXPRESSED = 0.10

required = [
    "adata_sc", "nichenet_priors", "SENDERS", "RECEIVERS",
    "de_tables", "PATHWAY_FILTER", "FDR_CUTOFF", "LOGFC_CUTOFF"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects/params (run upstream cells): {missing}")

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

for k in [CT_KEY, GROUP_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found. Available (first 60): {list(adata_sc.obs.columns)[:60]}")

if ZONE_KEY not in de_tables:
    raise KeyError(f"de_tables['{ZONE_KEY}'] not found. Available keys: {list(de_tables.keys())}")

for k in ["ltm_targets", "ltm_ligands", "lr_ligands", "lr_receptors"]:
    if k not in nichenet_priors:
        raise KeyError(f"nichenet_priors['{k}'] not found. Available keys: {list(nichenet_priors.keys())}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = set(std_ct(x) for x in SENDERS)
receivers_std = set(std_ct(x) for x in RECEIVERS)

ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

expected_groups = {"TIL_high", "TIL_low"}
groups_present = set(grp.unique())
if not expected_groups.issubset(groups_present):
    raise ValueError(f"Expected {expected_groups} in {GROUP_KEY}, found: {sorted(groups_present)}")

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

ltm_targets = set(map(str, nichenet_priors["ltm_targets"]))
ltm_ligands = set(map(str, nichenet_priors["ltm_ligands"]))
lr_ligands = set(map(str, nichenet_priors["lr_ligands"]))
lr_receptors = set(map(str, nichenet_priors["lr_receptors"]))

candidate_background = ltm_targets
candidate_ligands = ltm_ligands.intersection(lr_ligands)
candidate_receptors = lr_receptors

def expressed_genes(mask_cells: np.ndarray, candidate_genes: set, min_pct: float) -> set:
    idx = np.flatnonzero(mask_cells)
    if idx.size == 0:
        return set()

    genes_present = sorted(candidate_genes.intersection(var_set))
    if not genes_present:
        return set()

    gene_idx = var_names.get_indexer(genes_present)
    Xsub = X[idx, :][:, gene_idx]

    nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
    pct = nnz / float(idx.size)

    keep = pct >= float(min_pct)
    return set(np.asarray(genes_present, dtype=object)[keep])

panel = {}
for g in ["TIL_high", "TIL_low"]:
    sender_mask = (grp == g).to_numpy() & ct.isin(senders_std).to_numpy()
    receiver_mask = (grp == g).to_numpy() & ct.isin(receivers_std).to_numpy()

    panel[g] = {
        "n_sender_cells": int(sender_mask.sum()),
        "n_receiver_cells": int(receiver_mask.sum()),
        "background": expressed_genes(receiver_mask, candidate_background, MIN_PCT_EXPRESSED),
        "ligands": expressed_genes(sender_mask, candidate_ligands, MIN_PCT_EXPRESSED),
        "receptors": expressed_genes(receiver_mask, candidate_receptors, MIN_PCT_EXPRESSED),
    }

de_df = de_tables[ZONE_KEY].copy()

req = {"gene", "FDR", "logFC"}
if not req.issubset(set(de_df.columns)):
    raise KeyError(f"de_tables['{ZONE_KEY}'] must contain {sorted(req)}. Found: {list(de_df.columns)}")

de_df["gene"] = de_df["gene"].astype(str)
de_df["FDR"] = pd.to_numeric(de_df["FDR"], errors="coerce")
de_df["logFC"] = pd.to_numeric(de_df["logFC"], errors="coerce")

# DE-defined targets by direction (sign anchored to TIL_high_vs_low)
de_high = set(
    de_df.loc[
        (de_df["FDR"] <= float(FDR_CUTOFF)) & (de_df["logFC"] >= float(LOGFC_CUTOFF)),
        "gene"
    ].dropna()
)
de_low = set(
    de_df.loc[
        (de_df["FDR"] <= float(FDR_CUTOFF)) & (de_df["logFC"] <= -float(LOGFC_CUTOFF)),
        "gene"
    ].dropna()
)

# Gate to receiver-expressed background per group
targets_high = de_high.intersection(panel["TIL_high"]["background"])
targets_low = de_low.intersection(panel["TIL_low"]["background"])

targets_high_pre = set(targets_high)
targets_low_pre = set(targets_low)

# Optional pathway filter (apply after background gating)
if PATHWAY_FILTER is not None:
    pf = set(map(str, PATHWAY_FILTER))
    targets_high = targets_high.intersection(pf)
    targets_low = targets_low.intersection(pf)

universes = {
    "targets_high": sorted(map(str, targets_high)),
    "targets_low": sorted(map(str, targets_low)),
    "ligands_high": sorted(map(str, panel["TIL_high"]["ligands"])),
    "ligands_low": sorted(map(str, panel["TIL_low"]["ligands"])),
    "background_high": sorted(map(str, panel["TIL_high"]["background"])),
    "background_low": sorted(map(str, panel["TIL_low"]["background"])),
    "receptors_high": sorted(map(str, panel["TIL_high"]["receptors"])),
    "receptors_low": sorted(map(str, panel["TIL_low"]["receptors"])),
}

summary_df = pd.DataFrame(
    [
        {
            "group": g,
            "n_sender_cells": panel[g]["n_sender_cells"],
            "n_receiver_cells": panel[g]["n_receiver_cells"],
            "n_background": len(panel[g]["background"]),
            "n_ligands": len(panel[g]["ligands"]),
            "n_receptors": len(panel[g]["receptors"]),
        }
        for g in ["TIL_high", "TIL_low"]
    ]
)

print(summary_df.to_string(index=False))

print("\nTargets:")
print("  targets_high (DE∩background) pre pathway:", len(targets_high_pre), "post:", len(targets_high))
print("  targets_low  (DE∩background) pre pathway:", len(targets_low_pre),  "post:", len(targets_low))
print("  pathway filter enabled:", PATHWAY_FILTER is not None)

if PATHWAY_FILTER is not None:
    pf_set = set(map(str, PATHWAY_FILTER))
    print("  targets_high removed by PATHWAY_FILTER:", len(targets_high_pre - pf_set))
    print("  targets_low  removed by PATHWAY_FILTER:", len(targets_low_pre - pf_set))

if len(universes["targets_high"]) == 0 or len(universes["targets_low"]) == 0:
    raise ValueError(
        "Targets empty after DE/background gating and (optional) pathway filter. "
        "Check thresholds, DE orientation, and PATHWAY_FILTER."
    )

### Map targets to receiver cell types by expression fraction

In [ ]:
# %% [markdown]
# ### Map targets to receiver cell types by expression fraction

# %%
import numpy as np
import pandas as pd
import scipy.sparse as sp

# === CLEAR STALE VARIABLES ===
for _v in ["target_receiver_map_high", "target_receiver_map_low", "target_panel_coverage",
           "global_best_high", "global_best_low", "all_celltypes", "valid_celltypes", 
           "excluded_celltypes", "ct", "grp", "X", "var_names", "var_set"]:
    if _v in globals():
        del globals()[_v]

required = ["adata_sc", "targets_high", "targets_low", "RECEIVERS", "FRACTION_CUTOFF"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects/params: {missing}")

# === VERIFY INPUTS ===
print(f"INPUT CHECK:")
print(f"  targets_high: {len(targets_high)} genes")
print(f"  targets_low: {len(targets_low)} genes")
print(f"  RECEIVERS: {len(RECEIVERS)} cell types")
print(f"  FRACTION_CUTOFF: {FRACTION_CUTOFF}")

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"

MIN_SAMPLES_EXPRESSING = 3
PANEL_COVERAGE_THRESHOLD = 0.50

# Minimum cells required for a cell type to be considered in global best computation
# This prevents rare cell types (e.g., 1 cell) from dominating due to noise
MIN_CELLS_FOR_GLOBAL_BEST = 50

if LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{LAYER}'] not found.")
for k in [CT_KEY, GROUP_KEY, SAMPLE_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found.")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

def is_garbage_ct(ct_name: str) -> bool:
    """Exclude Doublet, Other, Unknown, Unassigned cell types."""
    s = str(ct_name)
    for bad in ["Doublet", "doublet", "Other", "other", "Unknown", "Unassigned"]:
        if bad in s:
            return True
    return False

receivers_std = sorted(set(std_ct(x) for x in RECEIVERS))

# Build fresh from adata_sc
ct = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)
sample_ids = adata_sc.obs[SAMPLE_KEY].astype(str)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)

# Get valid cell types (excluding garbage)
all_celltypes_raw = sorted(ct.unique())
valid_celltypes = [c for c in all_celltypes_raw if not is_garbage_ct(c)]
excluded_celltypes = [c for c in all_celltypes_raw if is_garbage_ct(c)]

print(f"\nCELL TYPE FILTERING:")
print(f"  Raw cell types: {len(all_celltypes_raw)}")
print(f"  Valid cell types (no Doublet/Other): {len(valid_celltypes)}")
print(f"  Excluded: {len(excluded_celltypes)}")

# Compute cell counts per cell type per group
celltype_counts = {}
for group_label in ["TIL_high", "TIL_low"]:
    celltype_counts[group_label] = {}
    for celltype in valid_celltypes:
        m = (grp == group_label).to_numpy() & (ct == celltype).to_numpy()
        n_cells = m.sum()
        celltype_counts[group_label][celltype] = n_cells

# Filter valid cell types for global best by minimum cell count
valid_celltypes_for_global = {}
for group_label in ["TIL_high", "TIL_low"]:
    valid_celltypes_for_global[group_label] = [
        c for c in valid_celltypes 
        if celltype_counts[group_label][c] >= MIN_CELLS_FOR_GLOBAL_BEST
    ]
    n_excluded_by_count = len(valid_celltypes) - len(valid_celltypes_for_global[group_label])
    print(f"  {group_label}: {len(valid_celltypes_for_global[group_label])} cell types with >= {MIN_CELLS_FOR_GLOBAL_BEST} cells ({n_excluded_by_count} excluded)")

def build_trm_for_group(group_label: str, targets_panel) -> pd.DataFrame:
    """Build target-receiver map for panel receivers only."""
    targets_panel = [str(x) for x in list(targets_panel)]
    targets_present = [t for t in targets_panel if t in var_set]
    if len(targets_present) == 0:
        raise ValueError(f"No targets found in var_names for {group_label}.")

    print(f"  {group_label}: {len(targets_present)} targets in var_names")
    
    gene_idx = var_names.get_indexer(targets_present)
    rows = []

    for rct in receivers_std:
        m = (grp == group_label).to_numpy() & (ct == rct).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue

        sample_sub = sample_ids.iloc[idx].values
        n_samples = len(np.unique(sample_sub))

        Xsub = X[idx, :][:, gene_idx]
        nnz = np.asarray(Xsub.getnnz(axis=0)).ravel()
        pct = nnz / float(idx.size)
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        Xsub_dense = Xsub.toarray()

        for i, target in enumerate(targets_present):
            gene_expr = Xsub_dense[:, i]
            n_samples_expressing = len(set(sample_sub[gene_expr > 0]))
            sample_frac = n_samples_expressing / n_samples if n_samples > 0 else 0.0

            if pct[i] >= float(FRACTION_CUTOFF) and n_samples_expressing >= MIN_SAMPLES_EXPRESSING:
                rows.append({
                    "target": target,
                    "receiver_celltype": rct,
                    "target_pct_expr": float(pct[i]),
                    "target_mean_expr": float(mean_expr[i]),
                    "n_samples": n_samples,
                    "n_samples_expressing": n_samples_expressing,
                    "sample_frac": float(sample_frac),
                })

    if not rows:
        return pd.DataFrame(columns=[
            "target", "receiver_celltype", "target_pct_expr", "target_mean_expr",
            "n_samples", "n_samples_expressing", "sample_frac", "specificity", "receiver_rank_score"
        ])

    out = pd.DataFrame(rows)

    # Specificity within panel
    target_mean_all = out.groupby("target")["target_mean_expr"].transform("mean").replace(0, np.nan)
    out["specificity"] = (out["target_mean_expr"] / target_mean_all).fillna(0)

    # Receiver rank score
    out["_mr"] = out.groupby("target")["target_mean_expr"].rank(ascending=False)
    out["_pr"] = out.groupby("target")["target_pct_expr"].rank(ascending=False)
    out["_sfr"] = out.groupby("target")["sample_frac"].rank(ascending=False)
    out["_spr"] = out.groupby("target")["specificity"].rank(ascending=False)
    out["receiver_rank_score"] = 0.35/out["_mr"] + 0.25/out["_pr"] + 0.20/out["_sfr"] + 0.20/out["_spr"]
    out = out.drop(columns=["_mr", "_pr", "_sfr", "_spr"])

    return out

def compute_global_best(group_label: str, targets_list) -> pd.DataFrame:
    """Find global best cell type per target.
    
    Only considers cell types with >= MIN_CELLS_FOR_GLOBAL_BEST cells
    to avoid noise from rare populations.
    """
    targets_list = [str(x) for x in targets_list if str(x) in var_set]
    if not targets_list:
        return pd.DataFrame(columns=["target", "global_best_celltype", "global_best_mean_expr", "global_best_n_cells"])

    gene_idx = var_names.get_indexer(targets_list)
    best = {t: (None, 0.0, 0) for t in targets_list}  # (celltype, mean_expr, n_cells)

    # Only use cell types with sufficient cells
    celltypes_to_check = valid_celltypes_for_global[group_label]
    
    for celltype in celltypes_to_check:
        m = (grp == group_label).to_numpy() & (ct == celltype).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue
        n_cells = idx.size
        Xsub = X[idx, :][:, gene_idx]
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        for i, target in enumerate(targets_list):
            if mean_expr[i] > best[target][1]:
                best[target] = (celltype, float(mean_expr[i]), n_cells)

    results = [{"target": t, "global_best_celltype": v[0], "global_best_mean_expr": v[1], "global_best_n_cells": v[2]} 
               for t, v in best.items() if v[0] is not None]
    return pd.DataFrame(results)

def add_coverage(trm: pd.DataFrame, global_best: pd.DataFrame) -> pd.DataFrame:
    if trm.empty:
        return trm
    trm = trm.merge(global_best, on="target", how="left")
    panel_best = trm.groupby("target")["target_mean_expr"].transform("max")
    trm["panel_coverage_ratio"] = np.where(
        trm["global_best_mean_expr"] > 0,
        panel_best / trm["global_best_mean_expr"],
        1.0
    )
    trm["panel_is_adequate"] = trm["panel_coverage_ratio"] >= PANEL_COVERAGE_THRESHOLD
    trm["coverage_flag"] = np.where(trm["panel_is_adequate"], "OK", "LOW_COVERAGE")
    return trm

# === BUILD TRMs ===
print("\nBUILDING TARGET-RECEIVER MAPS:")
target_receiver_map_high = build_trm_for_group("TIL_high", targets_high)
target_receiver_map_low = build_trm_for_group("TIL_low", targets_low)

# === COMPUTE GLOBAL BEST ===
print(f"\nCOMPUTING GLOBAL BEST (min {MIN_CELLS_FOR_GLOBAL_BEST} cells, excluding Doublets/Other):")
global_best_high = compute_global_best("TIL_high", targets_high)
global_best_low = compute_global_best("TIL_low", targets_low)

# Verify no garbage in global_best
for name, gb in [("high", global_best_high), ("low", global_best_low)]:
    if not gb.empty:
        garbage = [ct for ct in gb["global_best_celltype"].dropna().unique() if is_garbage_ct(ct)]
        if garbage:
            raise RuntimeError(f"BUG: Garbage cell types in global_best_{name}: {garbage}")
        print(f"  global_best_{name}: {len(gb)} targets, no garbage cell types ✓")

# === ADD COVERAGE ===
target_receiver_map_high = add_coverage(target_receiver_map_high, global_best_high)
target_receiver_map_low = add_coverage(target_receiver_map_low, global_best_low)

# === SUMMARY ===
print("\nTARGET-RECEIVER MAP SUMMARY:")
for name, trm in [("TIL_high", target_receiver_map_high), ("TIL_low", target_receiver_map_low)]:
    print(f"\n  {name}:")
    print(f"    rows: {len(trm)}")
    print(f"    targets: {trm['target'].nunique()}")
    print(f"    receivers: {trm['receiver_celltype'].nunique()}")
    if "coverage_flag" in trm.columns:
        n_ok = (trm["coverage_flag"] == "OK").sum()
        n_low = (trm["coverage_flag"] == "LOW_COVERAGE").sum()
        print(f"    coverage: OK={n_ok}, LOW_COVERAGE={n_low}")

# Show LOW_COVERAGE targets
print("\nLOW_COVERAGE TARGETS:")
for name, trm in [("TIL_high", target_receiver_map_high), ("TIL_low", target_receiver_map_low)]:
    low = trm[trm["coverage_flag"] == "LOW_COVERAGE"][["target", "global_best_celltype", "global_best_n_cells", "panel_coverage_ratio"]].drop_duplicates()
    if low.empty:
        print(f"  {name}: None")
    else:
        print(f"  {name}:")
        for _, r in low.iterrows():
            print(f"    {r['target']}: global_best={r['global_best_celltype']} (n={r['global_best_n_cells']}), coverage={r['panel_coverage_ratio']:.1%}")

if target_receiver_map_high.empty or target_receiver_map_low.empty:
    raise ValueError("One or both target_receiver_maps are empty!")

# Store for downstream
target_panel_coverage = pd.concat([
    global_best_high.assign(direction="TIL_high"),
    global_best_low.assign(direction="TIL_low")
], ignore_index=True)

print("\n✓ Created: target_receiver_map_high, target_receiver_map_low, target_panel_coverage")

### Build SCE (python)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmwrite

if "adata_sc" not in globals():
    raise NameError("adata_sc not found")
if "SENDERS" not in globals() or "RECEIVERS" not in globals():
    raise NameError("SENDERS and/or RECEIVERS not found")

COUNTS_LAYER = "unnormalized_counts"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"
BATCH_KEY = "orig.ident"

if COUNTS_LAYER not in adata_sc.layers:
    raise KeyError(f"adata_sc.layers['{COUNTS_LAYER}'] not found. Available: {list(adata_sc.layers.keys())}")

for k in [CT_KEY, GROUP_KEY, SAMPLE_KEY, BATCH_KEY]:
    if k not in adata_sc.obs.columns:
        raise KeyError(f"adata_sc.obs['{k}'] not found. Available obs cols (first 50): {list(adata_sc.obs.columns)[:50]}")

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

def drop_bad_ct(ct_list):
    out = []
    for ct in ct_list:
        s = str(ct)
        if s.startswith("Doublet"):
            continue
        if s.startswith("Other"):
            continue
        out.append(s)
    return sorted(set(out))

senders_py = drop_bad_ct(list(SENDERS))
receivers_py = drop_bad_ct(list(RECEIVERS))

senders_std = [std_ct(x) for x in senders_py]
receivers_std = [std_ct(x) for x in receivers_py]
panel_ct_std = sorted(set(senders_std) | set(receivers_std))

ct_std = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)

m = grp.isin(["TIL_low", "TIL_high"]) & ct_std.isin(panel_ct_std)
n_keep = int(m.sum())
if n_keep == 0:
    raise ValueError("No cells left after filtering to (TIL_low/TIL_high) and sender/receiver panel celltypes.")

adata_sub = adata_sc[m].copy()

X = adata_sub.layers[COUNTS_LAYER]
if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()

EFS_ROOT = Path("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd")
zone_dir = ZONE.replace("+", "plus").replace("-", "minus")  # CHANGED: derive from ZONE
out_dir = EFS_ROOT / "results" / "visium" / "til_zones" / "nichenet_filtering" / f"mnn_{zone_dir}_input"  # CHANGED: use zone_dir
out_dir.mkdir(parents=True, exist_ok=True)

for fname in ["counts.mtx", "features.tsv", "barcodes.tsv", "metadata.csv", "senders.txt", "receivers.txt"]:
    fpath = out_dir / fname
    if fpath.exists():
        fpath.unlink()

counts_gene_cell = X.T.tocsc()
mmwrite(str(out_dir / "counts.mtx"), counts_gene_cell)

genes = pd.Series(adata_sub.var_names.astype(str))
genes.to_csv(out_dir / "features.tsv", sep="\t", index=False, header=False)

cells = pd.Series(adata_sub.obs_names.astype(str))
cells.to_csv(out_dir / "barcodes.tsv", sep="\t", index=False, header=False)

ct_sub_std = adata_sub.obs[CT_KEY].astype(str).map(std_ct).values

meta = pd.DataFrame(
    {
        "cell": cells.values,
        "celltype": ct_sub_std,
        "sample": adata_sub.obs[SAMPLE_KEY].astype(str).values,
        "group": adata_sub.obs[GROUP_KEY].astype(str).values,
        "batch": adata_sub.obs[BATCH_KEY].astype(str).values,
    }
)
meta.to_csv(out_dir / "metadata.csv", index=False)

pd.Series(senders_std).to_csv(out_dir / "senders.txt", index=False, header=False)
pd.Series(receivers_std).to_csv(out_dir / "receivers.txt", index=False, header=False)

# ADDED: Export for R cell
SCE_INPUT_DIR = f"mnn_{zone_dir}_input"

print("Exported for SCE:")
print("  out_dir:", str(out_dir))
print("  genes x cells:", counts_gene_cell.shape[0], "x", counts_gene_cell.shape[1])
print("  nnz:", int(counts_gene_cell.nnz))
print("  groups:", dict(pd.Series(meta["group"]).value_counts()))
print("  celltypes in panel (senders, receivers):", len(senders_std), ",", len(receivers_std))

### Build SCE (R)

In [ ]:
%%R -i SCE_INPUT_DIR
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(S4Vectors)
  library(dplyr)
  library(readr)
  library(multinichenetr)
})

EFS_ROOT  <- "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd"
in_dir <- file.path(EFS_ROOT, "results", "visium", "til_zones", "nichenet_filtering", SCE_INPUT_DIR)  # CHANGED: use variable
stopifnot(dir.exists(in_dir))

counts <- readMM(file.path(in_dir, "counts.mtx"))
counts <- as(counts, "dgCMatrix")

feat <- read_tsv(file.path(in_dir, "features.tsv"), col_names = FALSE, show_col_types = FALSE)
genes <- if (ncol(feat) == 1) feat[[1]] else feat[[2]]

cells <- read_tsv(file.path(in_dir, "barcodes.tsv"), col_names = FALSE, show_col_types = FALSE)[[1]]
meta  <- read_csv(file.path(in_dir, "metadata.csv"), show_col_types = FALSE)

stopifnot(nrow(counts) == length(genes))
stopifnot(ncol(counts) == length(cells))

rownames(counts) <- genes
colnames(counts) <- cells

stopifnot("cell" %in% colnames(meta))
meta <- meta %>% tibble::column_to_rownames("cell")
meta <- meta[colnames(counts), , drop = FALSE]

stopifnot(all(c("celltype", "sample", "group", "batch") %in% colnames(meta)))

meta$celltype_mnn <- make.names(meta$celltype)

sce <- SingleCellExperiment(
  assays = list(counts = counts),
  colData = S4Vectors::DataFrame(meta),
  rowData = S4Vectors::DataFrame(gene = rownames(counts))
)

stopifnot("sample" %in% colnames(colData(sce)))
stopifnot("celltype_mnn" %in% colnames(colData(sce)))
stopifnot("group" %in% colnames(colData(sce)))

colData(sce)$sample_id <- as.character(colData(sce)$sample)
colData(sce)$celltype  <- as.character(colData(sce)$celltype_mnn)

if (anyNA(colData(sce)$sample_id)) stop("sample_id has NA values (check sample in metadata.csv).")
if (anyNA(colData(sce)$celltype))  stop("celltype has NA values (check celltype in metadata.csv).")

senders_oi   <- make.names(readLines(file.path(in_dir, "senders.txt")))
receivers_oi <- make.names(readLines(file.path(in_dir, "receivers.txt")))

ct_levels <- unique(colData(sce)$celltype)
missing_senders   <- setdiff(senders_oi, ct_levels)
missing_receivers <- setdiff(receivers_oi, ct_levels)
if (length(missing_senders) > 0) {
  stop("Senders not found in sce$celltype: ", paste(missing_senders, collapse = ", "))
}
if (length(missing_receivers) > 0) {
  stop("Receivers not found in sce$celltype: ", paste(missing_receivers, collapse = ", "))
}

cat("SCE:", dim(sce)[1], "genes x", dim(sce)[2], "cells\n")
cat("Groups (group):\n")
print(table(colData(sce)$group, useNA = "ifany"))
cat("\nCelltypes (celltype) top 15:\n")
print(sort(table(colData(sce)$celltype), decreasing = TRUE)[1:15])

cat("\nSenders (n=", length(senders_oi), "):\n", sep = "")
print(senders_oi)
cat("\nReceivers (n=", length(receivers_oi), "):\n", sep = "")
print(receivers_oi)

stopifnot(exists("lr_network"), exists("ligand_target_matrix"), exists("weighted_networks"))
cat("\nPriors OK:\n")
cat("  lr_network dim:", paste(dim(lr_network), collapse = " x "), "\n")
cat("  ligand_target_matrix dim:", paste(dim(ligand_target_matrix), collapse = " x "), "\n")
cat("  weighted_networks class:", paste(class(weighted_networks), collapse = ","), "\n")

### Compute ligand activity per direction

In [ ]:
# Make R-safe copies (rpy2 cannot convert Python sets)
targets_high_r = sorted(list(targets_high))
targets_low_r  = sorted(list(targets_low))

background_high_r = list(background_high)
background_low_r  = list(background_low)
ligands_high_r    = list(ligands_high)
ligands_low_r     = list(ligands_low)

print("R-ready objects:")
print("  targets_high_r:", type(targets_high_r), len(targets_high_r))
print("  targets_low_r :", type(targets_low_r),  len(targets_low_r))
print("  background_high_r:", type(background_high_r), len(background_high_r))
print("  background_low_r :", type(background_low_r),  len(background_low_r))
print("  ligands_high_r:", type(ligands_high_r), len(ligands_high_r))
print("  ligands_low_r :", type(ligands_low_r),  len(ligands_low_r))

In [ ]:
%%R -i targets_high_r -i targets_low_r -i background_high_r -i background_low_r -i ligands_high_r -i ligands_low_r
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
  library(nichenetr)
})

stopifnot(exists("ligand_target_matrix"))

to_chr_unique <- function(x) {
  unique(as.character(unlist(x, use.names = FALSE)))
}

targets_high_chr    <- to_chr_unique(targets_high_r)
targets_low_chr     <- to_chr_unique(targets_low_r)
background_high_chr <- to_chr_unique(background_high_r)
background_low_chr  <- to_chr_unique(background_low_r)
ligands_high_chr    <- to_chr_unique(ligands_high_r)
ligands_low_chr     <- to_chr_unique(ligands_low_r)

ltm <- ligand_target_matrix
ltm_ligands <- rownames(ltm)
ltm_targets <- colnames(ltm)

targets_high_use <- intersect(targets_high_chr, ltm_targets)
targets_low_use  <- intersect(targets_low_chr,  ltm_targets)

bg_high_use <- intersect(background_high_chr, ltm_targets)
bg_low_use  <- intersect(background_low_chr,  ltm_targets)

lig_high_use <- intersect(ligands_high_chr, ltm_ligands)
lig_low_use  <- intersect(ligands_low_chr,  ltm_ligands)

get_top_ltm_ligands <- function(targets_use, ltm, k_per_target) {
  out <- character()
  if (length(targets_use) == 0) return(out)

  for (t in targets_use) {
    w <- ltm[, t, drop = TRUE]
    w <- w[w > 0]
    if (length(w) == 0) next
    top <- sort(w, decreasing = TRUE)
    if (length(top) > k_per_target) top <- top[seq_len(k_per_target)]
    out <- c(out, names(top))
  }
  unique(out)
}

LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY <- 10

top_ltm_high <- get_top_ltm_ligands(targets_high_use, ltm, LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY)
top_ltm_low  <- get_top_ltm_ligands(targets_low_use,  ltm, LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY)

potential_high <- union(lig_high_use, top_ltm_high)
potential_low  <- union(lig_low_use,  top_ltm_low)

cat("Ligand activity inputs after LTM intersection + LTM augmentation:\n")
cat("  LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY=", LIGANDS_FROM_LTM_PER_TARGET_ACTIVITY, "\n", sep = "")
cat("  TIL_high: targets=", length(targets_high_use),
    " background=", length(bg_high_use),
    " sender_panel_ligs=", length(lig_high_use),
    " ltm_top_ligs=", length(top_ltm_high),
    " potential_ligs=", length(potential_high), "\n", sep = "")
cat("  TIL_low : targets=", length(targets_low_use),
    " background=", length(bg_low_use),
    " sender_panel_ligs=", length(lig_low_use),
    " ltm_top_ligs=", length(top_ltm_low),
    " potential_ligs=", length(potential_low), "\n\n", sep = "")

if (length(targets_high_use) < 5) stop("Too few TIL_high targets after intersecting with ligand_target_matrix columns.")
if (length(targets_low_use)  < 5) stop("Too few TIL_low targets after intersecting with ligand_target_matrix columns.")
if (length(bg_high_use) < 50) stop("Too few TIL_high background genes after intersecting with ligand_target_matrix columns.")
if (length(bg_low_use)  < 50) stop("Too few TIL_low background genes after intersecting with ligand_target_matrix columns.")
if (length(potential_high) < 20) stop("Too few TIL_high potential ligands after augmentation.")
if (length(potential_low)  < 20) stop("Too few TIL_low potential ligands after augmentation.")

ligand_activities_high <- predict_ligand_activities(
  geneset = targets_high_use,
  background_expressed_genes = bg_high_use,
  potential_ligands = potential_high,
  ligand_target_matrix = ltm
) %>%
  as_tibble() %>%
  mutate(
    direction = "TIL_high",
    test_ligand = as.character(test_ligand),
    aupr_corrected = as.numeric(aupr_corrected),
    in_sender_panel = test_ligand %in% lig_high_use,
    in_ltm_top = test_ligand %in% top_ltm_high
  ) %>%
  arrange(desc(aupr_corrected), test_ligand) %>%
  mutate(rank = dplyr::row_number()) %>%
  select(test_ligand, direction, aupr_corrected, rank, in_sender_panel, in_ltm_top, everything())

ligand_activities_low <- predict_ligand_activities(
  geneset = targets_low_use,
  background_expressed_genes = bg_low_use,
  potential_ligands = potential_low,
  ligand_target_matrix = ltm
) %>%
  as_tibble() %>%
  mutate(
    direction = "TIL_low",
    test_ligand = as.character(test_ligand),
    aupr_corrected = as.numeric(aupr_corrected),
    in_sender_panel = test_ligand %in% lig_low_use,
    in_ltm_top = test_ligand %in% top_ltm_low
  ) %>%
  arrange(desc(aupr_corrected), test_ligand) %>%
  mutate(rank = dplyr::row_number()) %>%
  select(test_ligand, direction, aupr_corrected, rank, in_sender_panel, in_ltm_top, everything())

ligand_activities <- bind_rows(ligand_activities_high, ligand_activities_low)

cat("Ligand activities computed:\n")
cat("  rows:", nrow(ligand_activities), "\n")
cat("  TIL_high:", nrow(ligand_activities_high), "\n")
cat("  TIL_low :", nrow(ligand_activities_low), "\n\n")

cat("Top 10 (TIL_high):\n")
print(ligand_activities_high %>% slice_head(n = 10) %>% select(test_ligand, aupr_corrected, rank, in_sender_panel, in_ltm_top))
cat("\nTop 10 (TIL_low):\n")
print(ligand_activities_low %>% slice_head(n = 10) %>% select(test_ligand, aupr_corrected, rank, in_sender_panel, in_ltm_top))

### Ligand-receptor DE per cell type

In [ ]:
# Pass zone prefix to R for file naming
zone_prefix = ZONE.replace("+", "plus").replace("-", "minus")
print(f"Zone prefix for R: {zone_prefix}")

In [ ]:
%%R -i zone_prefix
# LR-DE: edgeR pseudobulk per celltype, TIL_high vs TIL_low
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(SummarizedExperiment)
  library(S4Vectors)
  library(dplyr)
  library(tibble)
  library(readr)
  library(edgeR)
})

stopifnot(exists("sce"), exists("senders_oi"), exists("receivers_oi"))
stopifnot(exists("lr_network"), exists("ligand_target_matrix"))
stopifnot(exists("ligand_activities"))
stopifnot("counts" %in% SummarizedExperiment::assayNames(sce))

EFS_ROOT  <- "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd"
FILTER_DIR <- file.path(EFS_ROOT, "results", "visium", "til_zones", "nichenet_filtering")
dir.create(FILTER_DIR, recursive = TRUE, showWarnings = FALSE)

GROUPS <- c("TIL_low","TIL_high")

MIN_CELLS_PER_SAMPLE  <- 5
MIN_SAMPLES_PER_GROUP <- 3

CT_COL     <- "celltype"
GROUP_COL  <- "group"
SAMPLE_COL <- "sample_id"

cd0 <- as.data.frame(colData(sce))
stopifnot(CT_COL %in% colnames(cd0), GROUP_COL %in% colnames(cd0), SAMPLE_COL %in% colnames(cd0))

cd0[[CT_COL]]     <- as.character(cd0[[CT_COL]])
cd0[[GROUP_COL]]  <- as.character(cd0[[GROUP_COL]])
cd0[[SAMPLE_COL]] <- as.character(cd0[[SAMPLE_COL]])

bad_sample <- is.na(cd0[[SAMPLE_COL]]) | cd0[[SAMPLE_COL]] == ""
if (any(bad_sample)) {
  ex <- unique(cd0[[SAMPLE_COL]][bad_sample])[1:min(10, sum(bad_sample))]
  stop(paste0("sample_id contains NA/empty values (n=", sum(bad_sample), "). Examples: ", paste(ex, collapse=", ")))
}

bad_group <- is.na(cd0[[GROUP_COL]]) | !(cd0[[GROUP_COL]] %in% GROUPS)
if (any(bad_group)) {
  ex <- unique(cd0[[GROUP_COL]][bad_group])[1:min(10, sum(bad_group))]
  stop(paste0("group has NA or unexpected labels (n=", sum(bad_group), "). Examples: ", paste(ex, collapse=", ")))
}

cat("Using columns:\n")
cat("  celltype:", CT_COL, "\n")
cat("  group   :", GROUP_COL, "\n")
cat("  sample  :", SAMPLE_COL, "\n")
cat("Cells:", nrow(cd0), "\n\n")

cat("Group counts in sce:\n")
print(table(cd0[[GROUP_COL]], useNA = "ifany"))
cat("\n")

assays_avail <- assayNames(sce)
if (!("counts" %in% assays_avail)) {
  stop(paste0("sce is missing assay 'counts'. Available assays: ", paste(assays_avail, collapse=", ")))
}
counts_all <- assay(sce, "counts")
if (!is(counts_all, "dgCMatrix")) counts_all <- as(counts_all, "CsparseMatrix")
counts_all <- as(counts_all, "dgCMatrix")

cat("Counts matrix:\n")
cat("  class:", class(counts_all)[1], "\n")
cat("  dim  :", nrow(counts_all), "x", ncol(counts_all), "\n\n")

cat("Ligand activities directions:\n")
print(table(as.character(ligand_activities$direction), useNA="ifany"))
cat("\n")

# Harmonize lr_network columns
lr_cols <- colnames(lr_network)
lig_col <- if ("from" %in% lr_cols) "from" else if ("ligand" %in% lr_cols) "ligand" else stop("lr_network missing ligand column (from/ligand).")
rec_col <- if ("to" %in% lr_cols) "to" else if ("receptor" %in% lr_cols) "receptor" else stop("lr_network missing receptor column (to/receptor).")

lr_tbl <- lr_network %>%
  dplyr::distinct(.data[[lig_col]], .data[[rec_col]])
colnames(lr_tbl) <- c("from","to")

# Feasible ligands: use NicheNet activity list, constrained to LTM rows + sce genes
ltm <- ligand_target_matrix
ltm_ligands <- rownames(ltm)
ltm_targets <- colnames(ltm)

feasible_ligands <- intersect(
  unique(as.character(ligand_activities$test_ligand)),
  intersect(ltm_ligands, rownames(counts_all))
)

# Feasible receptors: receptors paired to feasible ligands, constrained to sce genes
feasible_receptors <- intersect(
  unique(lr_tbl$to[lr_tbl$from %in% feasible_ligands]),
  rownames(counts_all)
)

cat("DE universes:\n")
cat("  feasible ligands   :", length(feasible_ligands), "\n")
cat("  feasible receptors :", length(feasible_receptors), "\n\n")

if (length(feasible_ligands) < 20) stop("Too few feasible ligands after intersecting activity ligands with LTM rows and sce genes.")
if (length(feasible_receptors) < 20) stop("Too few feasible receptors after intersecting LR receptors with sce genes.")

# Fast pseudobulk using sparse model matrix
pseudobulk_by_sample <- function(counts_gene_cell, sample_vec) {
  sample_f <- factor(sample_vec)
  M <- Matrix::sparse.model.matrix(~0 + sample_f)
  colnames(M) <- levels(sample_f)
  pb <- counts_gene_cell %*% M
  as(pb, "dgCMatrix")
}

run_edger_for_celltype <- function(celltype_name, genes_keep, mode_label) {
  tryCatch({

    idx_ct <- which(cd0[[CT_COL]] == celltype_name & cd0[[GROUP_COL]] %in% GROUPS)
    if (length(idx_ct) == 0) {
      return(list(status="skip", reason="no cells after filtering", de=NULL, meta=NULL))
    }

    meta_ct <- cd0[idx_ct, , drop=FALSE]

    cells_per_sample <- meta_ct %>%
      dplyr::count(.data[[SAMPLE_COL]], name="n_cells")

    keep_samples <- cells_per_sample %>%
      dplyr::filter(n_cells >= MIN_CELLS_PER_SAMPLE) %>%
      dplyr::pull(.data[[SAMPLE_COL]])

    if (length(keep_samples) == 0) {
      return(list(status="skip", reason=paste0("no samples with >= ", MIN_CELLS_PER_SAMPLE, " cells"), de=NULL, meta=cells_per_sample))
    }

    keep_mask <- meta_ct[[SAMPLE_COL]] %in% keep_samples
    meta_ct <- meta_ct[keep_mask, , drop=FALSE]
    idx_ct  <- idx_ct[keep_mask]

    sample_group_tbl <- meta_ct %>%
      dplyr::group_by(.data[[SAMPLE_COL]]) %>%
      dplyr::summarise(
        n_cells = dplyr::n(),
        n_groups = dplyr::n_distinct(.data[[GROUP_COL]]),
        group = dplyr::first(.data[[GROUP_COL]]),
        .groups="drop"
      )

    bad <- sample_group_tbl %>% dplyr::filter(n_groups > 1)
    if (nrow(bad) > 0) {
      return(list(status="skip", reason="sample maps to >1 group within this celltype", de=NULL, meta=sample_group_tbl))
    }

    n_by_group <- sample_group_tbl %>% dplyr::count(group, name="n_samples")
    n_low  <- n_by_group %>% dplyr::filter(group=="TIL_low")  %>% dplyr::pull(n_samples)
    n_high <- n_by_group %>% dplyr::filter(group=="TIL_high") %>% dplyr::pull(n_samples)
    n_low  <- ifelse(length(n_low)==0, 0, n_low)
    n_high <- ifelse(length(n_high)==0, 0, n_high)

    if (n_low < MIN_SAMPLES_PER_GROUP || n_high < MIN_SAMPLES_PER_GROUP) {
      return(list(
        status="skip",
        reason=paste0("too few samples per group (low=", n_low, ", high=", n_high, ")"),
        de=NULL,
        meta=sample_group_tbl
      ))
    }

    genes_keep <- intersect(as.character(genes_keep), rownames(counts_all))
    if (length(genes_keep) < 10) {
      return(list(status="skip", reason="too few genes in universe after intersecting with sce genes", de=NULL, meta=sample_group_tbl))
    }

    counts_sub <- counts_all[genes_keep, idx_ct, drop=FALSE]
    pb <- pseudobulk_by_sample(counts_sub, meta_ct[[SAMPLE_COL]])

    group_vec <- sample_group_tbl$group
    names(group_vec) <- sample_group_tbl[[SAMPLE_COL]]
    group_vec <- group_vec[colnames(pb)]

    if (any(is.na(group_vec))) {
      na_samples <- colnames(pb)[is.na(group_vec)]
      return(list(
        status="error",
        reason=paste0("group_vec has NA after aligning to pseudobulk columns. Examples: ", paste(head(na_samples, 10), collapse=", ")),
        de=NULL,
        meta=sample_group_tbl
      ))
    }

    group_vec <- factor(group_vec, levels=GROUPS)

    y <- DGEList(counts = as.matrix(pb))
    design <- model.matrix(~ group_vec)

    keep <- filterByExpr(y, design=design)
    y <- y[keep, , keep.lib.sizes=FALSE]
    y <- calcNormFactors(y)

    y <- estimateDisp(y, design, robust=TRUE)
    fit <- glmQLFit(y, design, robust=TRUE)
    qlf <- glmQLFTest(fit, coef=2)

    tt <- topTags(qlf, n=Inf)$table %>%
      tibble::rownames_to_column("gene") %>%
      dplyr::as_tibble() %>%
      dplyr::transmute(
        gene,
        logFC,
        logCPM,
        F,
        PValue,
        FDR,
        celltype = celltype_name,
        mode = mode_label,
        n_samples_total = ncol(pb),
        n_TIL_low = sum(group_vec=="TIL_low"),
        n_TIL_high = sum(group_vec=="TIL_high")
      )

    meta_out <- sample_group_tbl %>%
      dplyr::rename(sample = .data[[SAMPLE_COL]]) %>%
      dplyr::mutate(celltype = celltype_name, mode = mode_label)

    list(status="ok", reason="", de=tt, meta=meta_out)

  }, error = function(e) {
    list(status="error", reason=paste0("edgeR failed: ", conditionMessage(e)), de=NULL, meta=NULL)
  })
}

run_panel <- function(celltypes_vec, genes_keep, mode_label) {
  de_list <- list()
  meta_list <- list()
  status_rows <- list()

  for (ct in celltypes_vec) {
    res <- run_edger_for_celltype(ct, genes_keep, mode_label)
    status_rows[[ct]] <- tibble(
      mode = mode_label,
      celltype = ct,
      status = res$status,
      reason = res$reason
    )
    if (identical(res$status, "ok")) {
      de_list[[ct]] <- res$de
      meta_list[[ct]] <- res$meta
    } else {
      if (!is.null(res$meta)) meta_list[[ct]] <- res$meta
    }
  }

  list(
    status = dplyr::bind_rows(status_rows),
    de = if (length(de_list)) dplyr::bind_rows(de_list) else tibble(),
    meta = if (length(meta_list)) dplyr::bind_rows(meta_list) else tibble()
  )
}

# Run
sender_res   <- run_panel(senders_oi,   feasible_ligands,   "sender_ligand")
receiver_res <- run_panel(receivers_oi, feasible_receptors, "receiver_receptor")

# CHANGED: Use zone_prefix for output file names
out_sender_de     <- file.path(FILTER_DIR, paste0(zone_prefix, "_sender_ligand_DE_edger_by_celltype_TILhigh_vs_TILlow_activityLigs_LTMrows.csv"))
out_receiver_de   <- file.path(FILTER_DIR, paste0(zone_prefix, "_receiver_receptor_DE_edger_by_celltype_TILhigh_vs_TILlow_LRreceptorsOfActivityLigs.csv"))
out_sender_meta   <- file.path(FILTER_DIR, paste0(zone_prefix, "_sender_pseudobulk_samples_by_celltype.csv"))
out_receiver_meta <- file.path(FILTER_DIR, paste0(zone_prefix, "_receiver_pseudobulk_samples_by_celltype.csv"))
out_status        <- file.path(FILTER_DIR, paste0(zone_prefix, "_LR_DE_status_by_celltype.csv"))

readr::write_csv(sender_res$de, out_sender_de)
readr::write_csv(receiver_res$de, out_receiver_de)
readr::write_csv(sender_res$meta, out_sender_meta)
readr::write_csv(receiver_res$meta, out_receiver_meta)
readr::write_csv(dplyr::bind_rows(sender_res$status, receiver_res$status), out_status)

cat("Wrote:\n")
cat(" ", out_sender_de, "\n")
cat(" ", out_receiver_de, "\n")
cat(" ", out_sender_meta, "\n")
cat(" ", out_receiver_meta, "\n")
cat(" ", out_status, "\n\n")

cat("Celltypes with OK DE (senders):", sum(sender_res$status$status=="ok"), "/", length(senders_oi), "\n")
cat("Celltypes with OK DE (receivers):", sum(receiver_res$status$status=="ok"), "/", length(receivers_oi), "\n\n")

if (any(sender_res$status$status=="error") || any(receiver_res$status$status=="error")) {
  cat("Errors detected in some celltypes:\n")
  print(dplyr::bind_rows(sender_res$status, receiver_res$status) %>% dplyr::filter(status=="error"))
  cat("\n")
}

if (nrow(sender_res$de) > 0) {
  cat("Top 10 ligand DE rows overall (by FDR):\n")
  print(sender_res$de %>% dplyr::arrange(FDR) %>% dplyr::slice_head(n=10))
  cat("\n")
}
if (nrow(receiver_res$de) > 0) {
  cat("Top 10 receptor DE rows overall (by FDR):\n")
  print(receiver_res$de %>% dplyr::arrange(FDR) %>% dplyr::slice_head(n=10))
  cat("\n")
}

### Export results to python

In [ ]:
%%R -o ligand_activities -o sender_de -o receiver_de
# Export R results to Python (rpy2)

stopifnot(exists("ligand_activities"))
stopifnot(exists("sender_res"), exists("receiver_res"))

# Pull DE tables out of the result bundles
sender_de   <- sender_res$de
receiver_de <- receiver_res$de

# Ensure base data.frame for rpy2 conversion stability
ligand_activities <- as.data.frame(ligand_activities)
sender_de   <- as.data.frame(sender_de)
receiver_de <- as.data.frame(receiver_de)

# Minimal sanity checks for downstream Cell 5 expectations
req_act <- c("test_ligand", "direction", "aupr_corrected", "rank")
req_de  <- c("gene", "celltype", "logFC", "FDR")

missing_act <- setdiff(req_act, colnames(ligand_activities))
missing_sde <- setdiff(req_de,  colnames(sender_de))
missing_rde <- setdiff(req_de,  colnames(receiver_de))

if (length(missing_act) > 0) stop(paste0("ligand_activities missing: ", paste(missing_act, collapse=", ")))
if (length(missing_sde) > 0) stop(paste0("sender_de missing: ", paste(missing_sde, collapse=", ")))
if (length(missing_rde) > 0) stop(paste0("receiver_de missing: ", paste(missing_rde, collapse=", ")))

cat("Export objects prepared:\n")
cat("  ligand_activities rows:", nrow(ligand_activities), "\n")
cat("  sender_de rows       :", nrow(sender_de), "\n")
cat("  receiver_de rows     :", nrow(receiver_de), "\n")

### Construct axis scores

In [ ]:
# %% [markdown]
# ### Construct axis scores

# %%
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path

# === CLEAR STALE VARIABLES ===
for _v in ["all_axes", "top_axes", "sender_candidates", "ligand_sender_gating", 
           "base", "axes", "trm_high", "trm_low", "ltm_df", "act_by_dir",
           "allowed_ligands_by_dir", "lr_use_by_dir", "receptors_by_dir"]:
    if _v in globals():
        del globals()[_v]

# === VERIFY INPUTS ===
required = ["target_receiver_map_high", "target_receiver_map_low", "SENDERS", "RECEIVERS",
            "ligand_activities", "sender_de", "receiver_de", "lr_network", 
            "ligand_target_matrix", "de_df", "OUT_DIR",
            "TOP_LIGANDS_PER_DIRECTION", "LTM_TOPK_PER_TARGET", "TOP_AXES_PER_TARGET",
            "TOP_LIGANDS_PER_TARGET", "TOP_AXES_PER_LIGAND_PER_TARGET", "SENDER_TOPK_PER_LIGAND"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing: {missing}")

print("INPUT CHECK:")
print(f"  target_receiver_map_high: {len(target_receiver_map_high)} rows, {target_receiver_map_high['target'].nunique()} targets")
print(f"  target_receiver_map_low: {len(target_receiver_map_low)} rows, {target_receiver_map_low['target'].nunique()} targets")

LAYER = "LogNormalize"
CT_KEY = "celltype_level4_curated"
GROUP_KEY = "TIL_group"
SAMPLE_KEY = "sample_id"

MIN_PCT_EXPRESSED_LIGAND = 0.10
MIN_SAMPLES_EXPRESSING_LIGAND = 3

def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

senders_std = sorted({std_ct(x) for x in SENDERS})
receivers_std = sorted({std_ct(x) for x in RECEIVERS})

TOP_LIGANDS_PER_DIRECTION = int(TOP_LIGANDS_PER_DIRECTION)
LTM_TOPK_PER_TARGET = int(LTM_TOPK_PER_TARGET)
TOP_AXES_PER_TARGET = int(TOP_AXES_PER_TARGET)
TOP_LIGANDS_PER_TARGET = int(TOP_LIGANDS_PER_TARGET)
TOP_AXES_PER_LIGAND_PER_TARGET = int(TOP_AXES_PER_LIGAND_PER_TARGET)
SENDER_TOPK_PER_LIGAND = int(SENDER_TOPK_PER_LIGAND) if SENDER_TOPK_PER_LIGAND is not None else None

LIGANDS_FROM_LTM_PER_TARGET = 10
KEEP_LIGANDS = []

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nSELECTION CONTROLS:")
print(f"  TOP_LIGANDS_PER_DIRECTION: {TOP_LIGANDS_PER_DIRECTION}")
print(f"  LTM_TOPK_PER_TARGET: {LTM_TOPK_PER_TARGET}")
print(f"  TOP_AXES_PER_TARGET: {TOP_AXES_PER_TARGET}")
print(f"  TOP_LIGANDS_PER_TARGET: {TOP_LIGANDS_PER_TARGET}")
print(f"  TOP_AXES_PER_LIGAND_PER_TARGET: {TOP_AXES_PER_LIGAND_PER_TARGET}")
print(f"  SENDER_TOPK_PER_LIGAND: {SENDER_TOPK_PER_LIGAND}")

# === COPY INPUTS ===
ligand_activities = pd.DataFrame(ligand_activities).copy()
sender_de = pd.DataFrame(sender_de).copy()
receiver_de = pd.DataFrame(receiver_de).copy()

ligand_activities["test_ligand"] = ligand_activities["test_ligand"].astype(str)
ligand_activities["direction"] = ligand_activities["direction"].astype(str)
ligand_activities["rank"] = pd.to_numeric(ligand_activities["rank"], errors="coerce")
ligand_activities["aupr_corrected"] = pd.to_numeric(ligand_activities["aupr_corrected"], errors="coerce")

for df_ in (sender_de, receiver_de):
    df_["gene"] = df_["gene"].astype(str)
    df_["celltype"] = df_["celltype"].astype(str)
    df_["logFC"] = pd.to_numeric(df_["logFC"], errors="coerce")
    df_["FDR"] = pd.to_numeric(df_["FDR"], errors="coerce")

lr_df0 = pd.DataFrame(lr_network).copy()
cols = set(lr_df0.columns.astype(str))
if {"from", "to"}.issubset(cols):
    lr_df = lr_df0.rename(columns={"from": "ligand", "to": "receptor"})[["ligand", "receptor"]].copy()
elif {"ligand", "receptor"}.issubset(cols):
    lr_df = lr_df0[["ligand", "receptor"]].copy()
else:
    raise KeyError(f"lr_network columns not recognized: {list(lr_df0.columns)}")
lr_df["ligand"] = lr_df["ligand"].astype(str)
lr_df["receptor"] = lr_df["receptor"].astype(str)
lr_df = lr_df.drop_duplicates()

# Fresh copy of TRMs
trm_high = pd.DataFrame(target_receiver_map_high).copy()
trm_low = pd.DataFrame(target_receiver_map_low).copy()

for trm in (trm_high, trm_low):
    trm["target"] = trm["target"].astype(str)
    trm["receiver_celltype"] = trm["receiver_celltype"].astype(str)
    for col in ["target_pct_expr", "target_mean_expr", "receiver_rank_score", "panel_coverage_ratio"]:
        if col in trm.columns:
            trm[col] = pd.to_numeric(trm[col], errors="coerce")

ltm_df_full = pd.DataFrame(
    ligand_target_matrix.values,
    index=pd.Index(ligand_target_matrix.index.astype(str), name="ligand"),
    columns=pd.Index(ligand_target_matrix.columns.astype(str), name="target"),
)

def get_ltm_ligands_for_targets(targets, ltm_df, k):
    targets = [t for t in targets if t in ltm_df.columns]
    ligs = set()
    for t in targets:
        w = ltm_df[t]
        w = w[w > 0].nlargest(k)
        ligs.update(w.index.astype(str))
    return ligs

# === ACTIVITY MAPS ===
act_by_dir = {}
for direction in ["TIL_high", "TIL_low"]:
    act = ligand_activities[ligand_activities["direction"] == direction].dropna(subset=["rank"])
    act = act.sort_values("rank")
    act_by_dir[direction] = {
        "top_ligs": act.nsmallest(TOP_LIGANDS_PER_DIRECTION, "rank")["test_ligand"].tolist(),
        "activity_map": act.set_index("test_ligand")["aupr_corrected"].to_dict(),
        "rank_map": act.set_index("test_ligand")["rank"].to_dict(),
    }

# === LIGAND UNIVERSE ===
allowed_ligands_by_dir = {}
for direction, trm in [("TIL_high", trm_high), ("TIL_low", trm_low)]:
    top_act = set(act_by_dir[direction]["top_ligs"])
    dir_targets = sorted(set(trm["target"]) & set(ltm_df_full.columns))
    top_ltm = get_ltm_ligands_for_targets(dir_targets, ltm_df_full, LIGANDS_FROM_LTM_PER_TARGET)
    allowed_ligands_by_dir[direction] = top_act | top_ltm | set(KEEP_LIGANDS)
    print(f"{direction} ligand universe: {len(allowed_ligands_by_dir[direction])}")

need_ligs = sorted(allowed_ligands_by_dir["TIL_high"] | allowed_ligands_by_dir["TIL_low"])
need_tgts = sorted(set(trm_high["target"]) | set(trm_low["target"]))
use_ligs = sorted(set(ltm_df_full.index) & set(need_ligs))
use_tgts = sorted(set(ltm_df_full.columns) & set(need_tgts))
ltm_df = ltm_df_full.loc[use_ligs, use_tgts].copy()

print(f"LTM subset: {len(use_ligs)} ligands x {len(use_tgts)} targets")

# === SENDER GATING ===
ct_std = adata_sc.obs[CT_KEY].astype(str).map(std_ct)
grp = adata_sc.obs[GROUP_KEY].astype(str)
sample_ids = adata_sc.obs[SAMPLE_KEY].astype(str)

X = adata_sc.layers[LAYER]
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

var_names = pd.Index(adata_sc.var_names.astype(str))
var_set = set(var_names)
ligs_in_var = sorted(set(use_ligs) & var_set)

sender_rows = []
for direction in ["TIL_high", "TIL_low"]:
    dir_ligs = sorted(allowed_ligands_by_dir[direction] & set(ligs_in_var))
    if not dir_ligs:
        continue
    lig_idx = var_names.get_indexer(dir_ligs)

    for sct in senders_std:
        m = (grp == direction).to_numpy() & (ct_std == sct).to_numpy()
        idx = np.flatnonzero(m)
        if idx.size == 0:
            continue

        sample_sub = sample_ids.iloc[idx].values
        n_samples = len(np.unique(sample_sub))
        Xsub = X[idx, :][:, lig_idx]
        pct = np.asarray(Xsub.getnnz(axis=0)).ravel() / idx.size
        mean_expr = np.asarray(Xsub.mean(axis=0)).ravel()
        Xsub_dense = Xsub.toarray()

        for i, lig in enumerate(dir_ligs):
            n_samp_expr = len(set(sample_sub[Xsub_dense[:, i] > 0]))
            if pct[i] >= MIN_PCT_EXPRESSED_LIGAND and n_samp_expr >= MIN_SAMPLES_EXPRESSING_LIGAND:
                sender_rows.append({
                    "direction": direction, "ligand": lig, "sender_celltype": sct,
                    "pct_expr": float(pct[i]), "mean_expr": float(mean_expr[i]),
                    "n_samples_expressing": n_samp_expr,
                    "sample_frac": n_samp_expr / n_samples if n_samples > 0 else 0,
                })

sender_gating = pd.DataFrame(sender_rows)

# Sender specificity and rank score
for direction in ["TIL_high", "TIL_low"]:
    mask = sender_gating["direction"] == direction
    if mask.sum() == 0:
        continue
    lig_mean = sender_gating.loc[mask].groupby("ligand")["mean_expr"].transform("mean").replace(0, np.nan)
    sender_gating.loc[mask, "specificity"] = (sender_gating.loc[mask, "mean_expr"] / lig_mean).fillna(0)

sender_gating["_mr"] = sender_gating.groupby(["direction", "ligand"])["mean_expr"].rank(ascending=False)
sender_gating["_pr"] = sender_gating.groupby(["direction", "ligand"])["pct_expr"].rank(ascending=False)
sender_gating["_sfr"] = sender_gating.groupby(["direction", "ligand"])["sample_frac"].rank(ascending=False)
sender_gating["_spr"] = sender_gating.groupby(["direction", "ligand"])["specificity"].rank(ascending=False)
sender_gating["sender_rank_score"] = 0.35/sender_gating["_mr"] + 0.25/sender_gating["_pr"] + 0.20/sender_gating["_sfr"] + 0.20/sender_gating["_spr"]
sender_gating = sender_gating.drop(columns=["_mr", "_pr", "_sfr", "_spr"])

# Merge sender DE
sender_de_use = sender_de.rename(columns={"gene": "ligand", "celltype": "sender_celltype", "logFC": "ligand_logFC", "FDR": "ligand_FDR"})
sender_candidates = sender_gating.merge(sender_de_use[["ligand", "sender_celltype", "ligand_logFC", "ligand_FDR"]], on=["ligand", "sender_celltype"], how="left")

# Concordance bonus
exp_sign = np.where(sender_candidates["direction"] == "TIL_high", 1, -1)
conc = sender_candidates["ligand_logFC"].notna() & ((exp_sign * sender_candidates["ligand_logFC"]) > 0)
sender_candidates.loc[conc, "sender_rank_score"] += 0.05

sender_candidates = sender_candidates.sort_values(["direction", "ligand", "sender_rank_score"], ascending=[True, True, False])
if SENDER_TOPK_PER_LIGAND is not None:
    sender_candidates = sender_candidates.groupby(["direction", "ligand"], as_index=False).head(SENDER_TOPK_PER_LIGAND)

print(f"Sender candidates: {len(sender_candidates)} rows")

# === LR BY DIRECTION ===
lr_by_dir = {}
for d in ["TIL_high", "TIL_low"]:
    lr_by_dir[d] = lr_df[lr_df["ligand"].isin(allowed_ligands_by_dir[d])].copy()
    lr_by_dir[d]["direction"] = d

# === LTM TOP-K ===
def ltm_topk(direction, trm):
    tgts = sorted(set(trm["target"]) & set(ltm_df.columns))
    elig_ligs = sorted(set(lr_by_dir[direction]["ligand"]) & set(ltm_df.index))
    if not tgts or not elig_ligs:
        return pd.DataFrame(columns=["direction", "target", "ligand", "ltm_weight"])
    sub = ltm_df.loc[elig_ligs, tgts]
    rows = []
    for t in tgts:
        w = sub[t]
        w = w[w > 0].nlargest(LTM_TOPK_PER_TARGET)
        for lig, wt in w.items():
            rows.append({"direction": direction, "target": t, "ligand": str(lig), "ltm_weight": float(wt)})
    return pd.DataFrame(rows)

topk_high = ltm_topk("TIL_high", trm_high)
topk_low = ltm_topk("TIL_low", trm_low)

# === MERGE TRM ===
trm_cols = ["target", "receiver_celltype", "target_pct_expr", "target_mean_expr", "receiver_rank_score",
            "global_best_celltype", "panel_coverage_ratio", "coverage_flag"]
trm_cols_high = [c for c in trm_cols if c in trm_high.columns]
trm_cols_low = [c for c in trm_cols if c in trm_low.columns]

base_high = topk_high.merge(trm_high[trm_cols_high], on="target", how="inner")
base_low = topk_low.merge(trm_low[trm_cols_low], on="target", how="inner")
base = pd.concat([base_high, base_low], ignore_index=True)

print(f"Base: {len(base)} rows (high={len(base_high)}, low={len(base_low)})")

# Attach activity
base["ligand_activity"] = np.nan
base["ligand_activity_rank"] = np.nan
for d in ["TIL_high", "TIL_low"]:
    m = base["direction"] == d
    base.loc[m, "ligand_activity"] = base.loc[m, "ligand"].map(act_by_dir[d]["activity_map"])
    base.loc[m, "ligand_activity_rank"] = base.loc[m, "ligand"].map(act_by_dir[d]["rank_map"])

# Attach sender
base = base.merge(sender_candidates, on=["direction", "ligand"], how="inner")
print(f"After sender merge: {len(base)} rows")

# Attach receptor
lr_dir = pd.concat([lr_by_dir["TIL_high"], lr_by_dir["TIL_low"]], ignore_index=True)
axes = base.merge(lr_dir, on=["direction", "ligand"], how="inner")

receiver_tbl = receiver_de.rename(columns={"celltype": "receiver_celltype", "gene": "receptor", "logFC": "receptor_logFC", "FDR": "receptor_FDR"})
axes = axes.merge(receiver_tbl[["receiver_celltype", "receptor", "receptor_logFC", "receptor_FDR"]], on=["receiver_celltype", "receptor"], how="left")

# Attach target DE
de_use = de_df[["gene", "logFC", "FDR"]].copy()
de_use.columns = ["target", "target_logFC", "target_FDR"]
axes = axes.merge(de_use, on="target", how="left")

axes = axes.dropna(subset=["receptor_logFC"])
print(f"After receptor filter: {len(axes)} axes")

# === SCORE ===
def score_axes(df):
    df = df.copy()
    for d in ["TIL_high", "TIL_low"]:
        m = df["direction"] == d
        if m.sum() == 0:
            continue
        ranks = df.loc[m, "ligand_activity_rank"]
        max_r = ranks.max()
        df.loc[m, "activity_pctl"] = 1 - (ranks - 1) / max_r if max_r > 1 else 1.0
        df.loc[m, "ltm_pctl"] = df.loc[m, "ltm_weight"].rank(pct=True)
        if "receiver_rank_score" in df.columns:
            df.loc[m, "receiver_rank_pctl"] = df.loc[m, "receiver_rank_score"].rank(pct=True)
        else:
            df.loc[m, "receiver_rank_pctl"] = 0.5

    exp_sign = np.where(df["direction"] == "TIL_high", 1, -1)
    df["ligand_concordant"] = (exp_sign * df["ligand_logFC"]) > 0
    df["receptor_concordant"] = (exp_sign * df["receptor_logFC"]) > 0

    df["axis_score"] = (
        0.35 * df["activity_pctl"].fillna(0) +
        0.25 * df["ltm_pctl"].fillna(0) +
        0.15 * df["receiver_rank_pctl"].fillna(0) +
        0.10 * df["ligand_concordant"].astype(float) +
        0.15 * df["receptor_concordant"].astype(float)
    )
    return df

all_axes = score_axes(axes)
all_axes = all_axes.sort_values(["direction", "target", "axis_score"], ascending=[True, True, False])
all_axes["axis_rank"] = all_axes.groupby(["direction", "target"]).cumcount() + 1
all_axes = all_axes.drop_duplicates(subset=["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"])

# === TOP AXES SELECTION ===
lig_best = all_axes.groupby(["direction", "target", "ligand"]).agg(
    best_score=("axis_score", "max"), best_ltm=("ltm_weight", "max"), best_act=("ligand_activity", "max"), n=("axis_score", "size")
).reset_index()
lig_best = lig_best.sort_values(["direction", "target", "best_score", "best_ltm", "best_act"], ascending=[True, True, False, False, False])
lig_best["lig_rank"] = lig_best.groupby(["direction", "target"]).cumcount() + 1
keep_ligs = lig_best[lig_best["lig_rank"] <= TOP_LIGANDS_PER_TARGET][["direction", "target", "ligand"]]

sel = all_axes.merge(keep_ligs, on=["direction", "target", "ligand"], how="inner")
sel = sel.sort_values(["direction", "target", "ligand", "axis_score"], ascending=[True, True, True, False])
sel["ax_in_lig"] = sel.groupby(["direction", "target", "ligand"]).cumcount() + 1
sel = sel[sel["ax_in_lig"] <= TOP_AXES_PER_LIGAND_PER_TARGET]
sel = sel.sort_values(["direction", "target", "axis_score"], ascending=[True, True, False])
sel["ax_final"] = sel.groupby(["direction", "target"]).cumcount() + 1
top_axes = sel[sel["ax_final"] <= TOP_AXES_PER_TARGET].copy()
top_axes["axis_rank"] = top_axes.groupby(["direction", "target"]).cumcount() + 1

# === SUMMARY ===
print(f"\nAXES BUILT:")
print(f"  all_axes: {len(all_axes)} rows, {all_axes['target'].nunique()} targets")
print(f"  top_axes: {len(top_axes)} rows, {top_axes['target'].nunique()} targets")

for d in ["TIL_high", "TIL_low"]:
    sub = all_axes[all_axes["direction"] == d]
    print(f"\n  {d}: {len(sub)} axes, {sub['target'].nunique()} targets, {sub['ligand'].nunique()} ligands")

# === SAVE ===
all_axes.to_csv(OUT_DIR / "all_axes.csv", index=False)
top_axes.to_csv(OUT_DIR / "top_axes.csv", index=False)
print(f"\n✓ Saved to {OUT_DIR}")

### Summarize axis scores for plot selection

In [ ]:
# %% [markdown]
# ### Summarize axis scores for plot selection

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib

# === CLEAR STALE VARIABLES ===
for _v in ["df", "df_all", "df_top", "axes_catalog", "axes_shortlist", "short", "short2"]:
    if _v in globals():
        del globals()[_v]

# === VERIFY INPUTS ===
if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR not found.")
if "top_axes" not in globals():
    raise NameError("top_axes not found. Run Cell 5 first.")

OUT_DIR = Path(OUT_DIR)

print("INPUT CHECK:")
print(f"  top_axes: {len(top_axes)} rows, {top_axes['target'].nunique()} targets")
if "all_axes" in globals():
    print(f"  all_axes: {len(all_axes)} rows (for reference, not used in catalog)")

# Use top_axes (filtered) for catalog, NOT all_axes
df = pd.DataFrame(top_axes).copy()

req_cols = {"direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
            "target_logFC", "target_FDR", "ligand_activity", "ltm_weight",
            "ligand_concordant", "receptor_concordant", "axis_score", "axis_rank"}
missing = req_cols - set(df.columns)
if missing:
    raise KeyError(f"top_axes missing columns: {sorted(missing)}")

# Normalize dtypes
for c in ["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"]:
    df[c] = df[c].astype(str)
for c in ["axis_score", "axis_rank", "target_logFC", "target_FDR", "ligand_activity", "ltm_weight"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["ligand_concordant"] = df["ligand_concordant"].astype(bool)
df["receptor_concordant"] = df["receptor_concordant"].astype(bool)

if "receiver_rank_score" in df.columns:
    df["receiver_rank_score"] = pd.to_numeric(df["receiver_rank_score"], errors="coerce")
if "panel_coverage_ratio" in df.columns:
    df["panel_coverage_ratio"] = pd.to_numeric(df["panel_coverage_ratio"], errors="coerce")

# === FILTER 1: positive activity ===
n_before = len(df)
df = df[df["ligand_activity"] > 0].copy()
print(f"\nFilter: ligand_activity > 0: {n_before} -> {len(df)} axes")

# === ABUNDANCE ANNOTATIONS ===
def std_ct(x: str) -> str:
    return str(x).replace("-", ".")

abundance_available = False
if "EFS_ROOT" in globals() and "ZONE" in globals():
    ab_path = Path(EFS_ROOT) / "results/visium/til_zones/celltype_zone_specificity_curated/celltype_zone_specificity_level4_curated.csv"

    if ab_path.exists():
        ab = pd.read_csv(ab_path)
        ab["group"] = ab["group"].astype(str)
        ab["zone"] = ab["zone"].astype(str)
        ab["celltype_std"] = ab["celltype_level4_curated"].astype(str).map(std_ct)
        ab_zone = ab[ab["zone"] == str(ZONE)].copy()

        delta_tbl = ab_zone.pivot_table(index="celltype_std", columns="group", values="mean_zone", aggfunc="first").reset_index()
        if "TIL_high" in delta_tbl.columns and "TIL_low" in delta_tbl.columns:
            delta_tbl["mean_zone_high_minus_low"] = pd.to_numeric(delta_tbl["TIL_high"], errors="coerce") - pd.to_numeric(delta_tbl["TIL_low"], errors="coerce")
            delta_tbl = delta_tbl[["celltype_std", "mean_zone_high_minus_low"]]
        else:
            delta_tbl = pd.DataFrame({"celltype_std": ab_zone["celltype_std"].unique(), "mean_zone_high_minus_low": np.nan})

        df["sender_celltype_std"] = df["sender_celltype"].map(std_ct)
        df["receiver_celltype_std"] = df["receiver_celltype"].map(std_ct)

        # Sender abundance
        sender_ab = ab_zone.rename(columns={"celltype_std": "sender_celltype_std", "mean_zone": "sender_mean_zone"})
        sender_ab = sender_ab[["group", "sender_celltype_std", "sender_mean_zone"]].copy()
        df = df.merge(sender_ab, left_on=["direction", "sender_celltype_std"], right_on=["group", "sender_celltype_std"], how="left")
        df = df.drop(columns=["group"], errors="ignore")

        # Receiver abundance
        receiver_ab = ab_zone.rename(columns={"celltype_std": "receiver_celltype_std", "mean_zone": "receiver_mean_zone"})
        receiver_ab = receiver_ab[["group", "receiver_celltype_std", "receiver_mean_zone"]].copy()
        df = df.merge(receiver_ab, left_on=["direction", "receiver_celltype_std"], right_on=["group", "receiver_celltype_std"], how="left")
        df = df.drop(columns=["group"], errors="ignore")

        # Delta
        delta_sender = delta_tbl.rename(columns={"celltype_std": "sender_celltype_std", "mean_zone_high_minus_low": "sender_mean_zone_high_minus_low"})
        delta_receiver = delta_tbl.rename(columns={"celltype_std": "receiver_celltype_std", "mean_zone_high_minus_low": "receiver_mean_zone_high_minus_low"})
        df = df.merge(delta_sender, on="sender_celltype_std", how="left")
        df = df.merge(delta_receiver, on="receiver_celltype_std", how="left")

        print(f"Abundance annotations loaded from: {ab_path}")
        abundance_available = True
    else:
        print(f"Abundance CSV not found: {ab_path}")
else:
    print("EFS_ROOT or ZONE not set, skipping abundance annotations")

# === FILTER 2: concordance/abundance ===
ABUNDANCE_THRESHOLD = 0.05
n_before = len(df)

if abundance_available:
    high_mask = df["direction"] == "TIL_high"
    low_mask = df["direction"] == "TIL_low"
    high_keep = df["ligand_concordant"] | (df["sender_mean_zone_high_minus_low"] > ABUNDANCE_THRESHOLD)
    low_keep = df["ligand_concordant"] | (df["sender_mean_zone_high_minus_low"] < -ABUNDANCE_THRESHOLD)
    df = df[(high_mask & high_keep) | (low_mask & low_keep)].copy()
else:
    df = df[df["ligand_concordant"]].copy()

print(f"Filter: concordance/abundance: {n_before} -> {len(df)} axes")

# === AXIS ID ===
key_cols = ["direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype"]
df = df.drop_duplicates(subset=key_cols).copy()

def axis_hash_id(row):
    s = "|".join(str(row[c]) for c in key_cols)
    h = hashlib.md5(s.encode()).hexdigest()[:6].upper()
    return ("H" if row["direction"] == "TIL_high" else "L") + h

df["axis_id"] = df.apply(axis_hash_id, axis=1)

# === RANK SCORE ===
df["de_strength"] = -np.log10(df["target_FDR"].clip(lower=1e-300))
df["rank_score"] = df["axis_score"] * df["de_strength"] * df["target_logFC"].abs()

# === VERIFY AXES PER TARGET ===
axes_per_target = df.groupby(["direction", "target"]).size()
print(f"\nAxes per target: min={axes_per_target.min()}, max={axes_per_target.max()}, mean={axes_per_target.mean():.1f}")

# === CATALOG COLUMNS ===
catalog_cols = ["axis_id", "direction", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
                "axis_score", "axis_rank", "rank_score", "target_logFC", "target_FDR",
                "ligand_activity", "ltm_weight", "ligand_concordant", "receptor_concordant"]

# Add optional columns if present
for c in ["receiver_rank_score", "global_best_celltype", "global_best_n_cells", "panel_coverage_ratio", "coverage_flag",
          "sender_mean_zone", "receiver_mean_zone", "sender_mean_zone_high_minus_low", "receiver_mean_zone_high_minus_low"]:
    if c in df.columns:
        catalog_cols.append(c)

axes_catalog = df[[c for c in catalog_cols if c in df.columns]].copy()
axes_catalog.to_csv(OUT_DIR / "axes_catalog.csv", index=False)

# === SHORTLIST (further filtered subset) ===
AXIS_SCORE_MIN = 0.5
MAX_AXIS_RANK_PER_TARGET = 3
TARGET_FDR_MAX = 0.05
TARGET_LOGFC_MIN = 0.25
AXES_PER_TARGET_DE_BAL = 2

short = df[(df["axis_score"] > AXIS_SCORE_MIN) & (df["axis_rank"] <= MAX_AXIS_RANK_PER_TARGET)].copy()
short = short.dropna(subset=["target_FDR", "target_logFC", "rank_score"])
short = short[short["target_FDR"] <= TARGET_FDR_MAX]

high = short[(short["direction"] == "TIL_high") & (short["target_logFC"] >= TARGET_LOGFC_MIN)]
low = short[(short["direction"] == "TIL_low") & (short["target_logFC"] <= -TARGET_LOGFC_MIN)]
short = pd.concat([high, low])
short = short.sort_values(["direction", "target", "rank_score"], ascending=[True, True, False])
axes_shortlist = short.groupby(["direction", "target"], as_index=False).head(AXES_PER_TARGET_DE_BAL)
axes_shortlist = axes_shortlist[[c for c in catalog_cols if c in axes_shortlist.columns]]
axes_shortlist.to_csv(OUT_DIR / "axes_shortlist.csv", index=False)

# === SUMMARY ===
print(f"\nCATALOG SUMMARY:")
for d in ["TIL_high", "TIL_low"]:
    sub = df[df["direction"] == d]
    print(f"  {d}: {len(sub)} axes, {sub['target'].nunique()} targets, {sub['ligand'].nunique()} ligands")

if "coverage_flag" in df.columns:
    print(f"\nPANEL COVERAGE:")
    for d in ["TIL_high", "TIL_low"]:
        sub = df[df["direction"] == d]
        if sub.empty:
            continue
        n_ok = (sub["coverage_flag"] == "OK").sum()
        n_low = (sub["coverage_flag"] == "LOW_COVERAGE").sum()
        print(f"  {d}: OK={n_ok}, LOW_COVERAGE={n_low}")

print(f"\nSHORTLIST: {len(axes_shortlist)} axes, {axes_shortlist['target'].nunique()} targets")

print(f"\nWROTE:")
print(f"  {OUT_DIR / 'axes_catalog.csv'}")
print(f"  {OUT_DIR / 'axes_shortlist.csv'}")

# === DISPLAY ===
display_cols = ["axis_id", "target", "ligand", "receptor", "sender_celltype", "receiver_celltype",
                "axis_score", "rank_score", "target_logFC"]
if "coverage_flag" in axes_catalog.columns:
    display_cols.append("coverage_flag")
display_cols = [c for c in display_cols if c in axes_catalog.columns]

print(f"\nTOP 15 TIL_high:")
display(axes_catalog[axes_catalog["direction"] == "TIL_high"].nlargest(15, "rank_score")[display_cols])

print(f"\nTOP 15 TIL_low:")
display(axes_catalog[axes_catalog["direction"] == "TIL_low"].nlargest(15, "rank_score")[display_cols])

# === WARNINGS ===
if "coverage_flag" in axes_shortlist.columns:
    low_cov = axes_shortlist[axes_shortlist["coverage_flag"] == "LOW_COVERAGE"]
    if len(low_cov) > 0:
        print(f"\n⚠️ WARNING: {len(low_cov)} shortlist axes have LOW_COVERAGE:")
        for _, r in low_cov.iterrows():
            gb_ct = r.get('global_best_celltype', 'N/A')
            gb_n = r.get('global_best_n_cells', 'N/A')
            print(f"  {r['axis_id']}: {r['target']} (global_best={gb_ct}, n={gb_n})")

print("\n✓ Done")

### Axes for TIL+ Circos plot

In [ ]:
# Cell 7: Enter selected axis IDs (uses current run's catalog on disk)

import pandas as pd
from pathlib import Path

if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR not found.")

OUT_DIR = Path(OUT_DIR)
catalog_path = OUT_DIR / "axes_catalog.csv"
if not catalog_path.exists():
    raise FileNotFoundError(f"Missing catalog: {catalog_path}. Run Cell 6 first.")

axes_catalog = pd.read_csv(catalog_path)

if "axis_id" not in axes_catalog.columns:
    raise KeyError(f"axes_catalog.csv missing axis_id. Columns: {list(axes_catalog.columns)}")

# Paste your chosen IDs here (from this run's axes_catalog.csv)
SELECT_TIL_HIGH = [
    "H44C8FC",
"H03A347",
"HA7EED4",
"HA3CF6D",
"HCA12C4",
"H266533",
"HEB1D49",
"HBF613E",
"H427632",
"H0FA4A5",
"HFC9AB0",
"H7A6231",
"HDEE2BD",
"H9D4FBE",
"HD81AE6",
"HCACCB6",
"HF40919"
]

SELECT_TIL_LOW = [
    "L0106B0",
"L41D6BB",
"LFA222F",
"LFEB8F6",
"L1B85ED",
"LF665F9",
"LAE4A05",
"LD3B868",
"LA05E2C",
"LEFDED1",
"L343594",
"LC9463B",
"LBA0BBA",
"LA0186A",
"LDCE9F1",
"L2CF857",
"LCE4187",
"LAE73CE",
"L629AD2",
"L1ACD23",
"L2563BF",
"LCD4CAD",
"L168E05",
"L5357AF",
"L2D556C",
"L40E87D",
"LF8D57B"
]

# Normalize and dedupe while preserving input order
raw_ids = [str(x).strip() for x in (SELECT_TIL_HIGH + SELECT_TIL_LOW) if str(x).strip()]
if len(raw_ids) == 0:
    raise ValueError("No axis IDs provided. Fill SELECT_TIL_HIGH and/or SELECT_TIL_LOW.")

seen = set()
selected_ids = []
for x in raw_ids:
    if x not in seen:
        selected_ids.append(x)
        seen.add(x)

catalog_ids = axes_catalog["axis_id"].astype(str)
missing = sorted(set(selected_ids) - set(catalog_ids))
if missing:
    raise ValueError(
        "Some axis IDs were not found in the current run's catalog.\n"
        "This usually means you copied IDs from a different run.\n\n"
        "Missing:\n" + "\n".join(missing)
    )

sel = axes_catalog[catalog_ids.isin(selected_ids)].copy()

# Preserve the order you typed them
order = {aid: i for i, aid in enumerate(selected_ids)}
sel["input_order"] = sel["axis_id"].astype(str).map(order)
sel = sel.sort_values(["input_order"]).drop(columns=["input_order"])

# Sanity: ensure selected HIGH/LOW match your lists
if "direction" in sel.columns:
    bad_high = sel[(sel["axis_id"].isin(SELECT_TIL_HIGH)) & (sel["direction"] != "TIL_high")]
    bad_low  = sel[(sel["axis_id"].isin(SELECT_TIL_LOW))  & (sel["direction"] != "TIL_low")]
    if len(bad_high) > 0 or len(bad_low) > 0:
        msg = []
        if len(bad_high) > 0:
            msg.append("Some IDs in SELECT_TIL_HIGH are not TIL_high:\n" + "\n".join(bad_high["axis_id"].astype(str).tolist()))
        if len(bad_low) > 0:
            msg.append("Some IDs in SELECT_TIL_LOW are not TIL_low:\n" + "\n".join(bad_low["axis_id"].astype(str).tolist()))
        raise ValueError("\n\n".join(msg))

out_path = OUT_DIR / "axes_selected.csv"
sel.to_csv(out_path, index=False)

print("Selected axes:", len(sel))
if "direction" in sel.columns:
    print("  HIGH:", int((sel["direction"] == "TIL_high").sum()))
    print("  LOW :", int((sel["direction"] == "TIL_low").sum()))
print("Wrote:", str(out_path))

show_cols = [c for c in [
    "axis_id","direction","target","ligand","receptor",
    "sender_celltype","receiver_celltype",
    "axis_score","rank_score","target_logFC","target_FDR",
    "ligand_activity","ltm_weight",
    "ligand_concordant","receptor_concordant",
] if c in sel.columns]

display(sel[show_cols])